
## 1. Project Setup

We begin by connecting Google Colab to Google Drive, where the project datasets, extracted features, trained models, and experimental results are stored.

We define the main project directories once so that subsequent cells can access files consistently without repeatedly specifying absolute paths.


In [ ]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Define project directories
PROJECT = Path("/content/drive/MyDrive/MSD-I_project")
EXP = PROJECT / "finetuning_experiments"

# Verify project directories
assert PROJECT.exists(), "Project folder not found."
assert EXP.exists(), "Experiments folder not found."

print("=" * 60)
print("MULTIMODAL MUSIC GENRE CLASSIFICATION")
print("PROJECT SETUP")
print("=" * 60)

print("\nProject directory:", PROJECT)
print("Experiments directory:", EXP)

print("\nGoogle Drive successfully connected.")
print("Project directories verified.")



### Audio Data Preparation

We begin by inspecting the available track metadata and identifying the audio data required for multimodal classification. This preliminary inspection helps verify the dataset structure before constructing the final aligned audio–visual representation.

In [ ]:
from pathlib import Path
import pandas as pd

# PROJECT is defined in the initial setup cell
assert PROJECT.is_dir(), "Project directory not found."

dataset_path = PROJECT / "final_modeling_dataset.csv"

assert dataset_path.is_file(), "Dataset not found."

metadata = pd.read_csv(dataset_path)

print("Dataset observations:", len(metadata))
print("Columns:", metadata.columns.tolist())

print("\nGenre distribution:")
print(metadata["genre"].value_counts())

print("\nSplit distribution:")
print(metadata["split"].value_counts())

print("\nTrack ID examples:")
print(metadata["msd_track_id"].head())

### Inspection of Available Audio Features

We inspect the available Essentia audio-feature dataset to understand its structure, identify potential track identifiers, and examine the numerical features. Only a small sample is loaded at this stage to avoid unnecessary memory usage.

This inspection is part of the preliminary audio-data investigation and does not yet construct the final audio embeddings used in our multimodal experiments.

In [ ]:
import pandas as pd
from pathlib import Path


audio_path = (
    PROJECT
    / "audio_features"
    / "msd_played_songs_essentia.csv.gz"
)

print("=" * 65)
print("PHASE 5 — ESSENTIA AUDIO DATA INSPECTION")
print("=" * 65)

print("\nFile exists:", audio_path.exists())

assert audio_path.exists(), "Audio features file not found."

print(
    "Compressed file size:",
    round(audio_path.stat().st_size / (1024**2), 2),
    "MB"
)

# Read only a small sample to avoid loading the entire file
audio_sample = pd.read_csv(
    audio_path,
    compression="gzip",
    nrows=5
)

print("\nNUMBER OF COLUMNS:")
print(len(audio_sample.columns))

print("\nALL COLUMN NAMES:")
for i, column in enumerate(audio_sample.columns, 1):
    print(f"{i:3d}. {column}")

print("\nFIRST FIVE OBSERVATIONS (FIRST 8 COLUMNS):")
display(audio_sample.iloc[:, :8])

print("\nPOSSIBLE IDENTIFIER COLUMNS:")

identifier_columns = [
    col for col in audio_sample.columns
    if any(
        word in col.lower()
        for word in [
            "id", "track", "song",
            "file", "name", "path"
        ]
    )
]

for col in identifier_columns:
    print(f"\n{col}:")
    print(audio_sample[col].head(5).tolist())

print("\nNUMERICAL FEATURE COUNT:")
print(
    len(
        audio_sample.select_dtypes(
            include="number"
        ).columns
    )
)

print("\n" + "=" * 65)
print("PHASE 5 — STEP 2 COMPLETE")
print("=" * 65)


### Audio and Album-Cover Identifier Compatibility

To investigate whether the available Essentia audio features could be matched with our album-cover dataset, we examined the identifier formats used by both sources.

The Essentia dataset contains Million Song Dataset (MSD) song identifiers, typically beginning with `SO`, whereas the album-cover metadata uses MSD track identifiers beginning with `TR`.

Because these identifiers represent different identifier systems, they cannot be matched directly. A reliable integration would require an explicit song-to-track mapping.

This audit forms part of our preliminary data investigation. The final multimodal classification experiments use aligned precomputed audio and visual embeddings rather than relying on a direct match between these two identifier columns.

In [ ]:
import pandas as pd

print("=" * 65)
print("AUDIO AND ALBUM-COVER IDENTIFIER AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Inspect Essentia audio identifiers
# ------------------------------------------------------------

audio_path = (
    PROJECT
    / "audio_features"
    / "msd_played_songs_essentia.csv.gz"
)

assert audio_path.is_file(), "Essentia audio file not found."

# Load only the identifier column
audio_ids = pd.read_csv(
    audio_path,
    compression="gzip",
    usecols=["metadata_tags_file_name"]
)

# Extract MSD song identifiers (SO...)
audio_ids["song_id"] = (
    audio_ids["metadata_tags_file_name"]
    .astype("string")
    .str.extract(
        r"^(SO[A-Z0-9]+)\.mp3$",
        expand=False
    )
)

valid_audio_ids = audio_ids["song_id"].dropna()

print("\nESSENTIA AUDIO DATASET")
print("Total records:", len(audio_ids))
print("Valid song IDs:", len(valid_audio_ids))
print("Unique song IDs:", valid_audio_ids.nunique())
print(
    "Duplicate valid song IDs:",
    valid_audio_ids.duplicated().sum()
)

print("\nExample song IDs:")
print(valid_audio_ids.head().tolist())

# ------------------------------------------------------------
# 2. Inspect album-cover track identifiers
# ------------------------------------------------------------

metadata_path = PROJECT / "final_modeling_dataset.csv"

assert metadata_path.is_file(), "Modeling metadata not found."

metadata = pd.read_csv(
    metadata_path,
    usecols=["msd_track_id", "genre", "split"]
)

track_ids = metadata["msd_track_id"].astype("string")

print("\nALBUM-COVER DATASET")
print("Total records:", len(metadata))
print("Valid track IDs:", track_ids.notna().sum())
print("Unique track IDs:", track_ids.nunique())

print("\nTrack identifier format:")
print(
    track_ids.str.startswith("TR").value_counts(
        dropna=False
    )
)

print("\nExample track IDs:")
print(track_ids.dropna().head().tolist())

# ------------------------------------------------------------
# 3. Summarize identifier compatibility
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("IDENTIFIER COMPATIBILITY")
print("=" * 65)

print("Essentia identifiers: MSD song IDs (SO...)")
print("Album-cover identifiers: MSD track IDs (TR...)")

print(
    "\nDirect matching is not appropriate without "
    "an explicit song-to-track mapping."
)

print("\nAUDIO IDENTIFIER AUDIT COMPLETE")


### Inspection of MSD-I Metadata

We inspect the original MSD-I metadata to identify the available track identifiers and determine whether the file can support alignment between album-cover observations and audio features.

The metadata structure is examined before attempting any joins, since the different audio and visual sources may use different identifier conventions.

In [ ]:
import pandas as pd

mapping_path = (
    PROJECT / "audio_features" / "MSD-I_dataset.tsv"
)

assert mapping_path.is_file(), (
    "MSD-I metadata file not found. "
    "Verify that the source file is available."
)

# Inspect metadata structure
original = pd.read_csv(
    mapping_path,
    sep="\t",
    nrows=5
)

print("MSD-I METADATA")
print("Columns:", original.columns.tolist())

display(original)

# Compare with project identifiers
metadata = pd.read_csv(
    PROJECT / "final_modeling_dataset.csv",
    usecols=["msd_track_id"]
)

print("\nProject track ID examples:")
print(metadata["msd_track_id"].head().tolist())


### Validation of Track Metadata

We compare our modeling dataset with the original MSD-I metadata to verify track identifier overlap and genre-label consistency.

We calculate the proportion of tracks that can be matched and check whether the corresponding genre labels agree. These checks help validate the metadata used in subsequent processing.

This comparison assesses metadata consistency rather than the availability of audio embeddings, which must be verified separately.

In [ ]:

# ============================================================
# PHASE 5 — VALIDATE MSD-I METADATA OVERLAP
# ============================================================

import pandas as pd

print("=" * 65)
print("MSD-I METADATA VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

original_path = (
    PROJECT / "audio_features" / "MSD-I_dataset.tsv"
)

modeling_path = PROJECT / "final_modeling_dataset.csv"

assert original_path.is_file(), "Original MSD-I metadata not found."
assert modeling_path.is_file(), "Modeling dataset not found."

original = pd.read_csv(original_path, sep="\t")
ours = pd.read_csv(modeling_path)

required_columns = {"msd_track_id", "genre"}

assert required_columns.issubset(original.columns)
assert required_columns.issubset(ours.columns)

# ------------------------------------------------------------
# 2. Validate track identifiers
# ------------------------------------------------------------

assert original["msd_track_id"].notna().all(), (
    "Missing track IDs in original metadata."
)

assert ours["msd_track_id"].notna().all(), (
    "Missing track IDs in modeling dataset."
)

assert original["msd_track_id"].is_unique, (
    "Duplicate track IDs in original metadata."
)

assert ours["msd_track_id"].is_unique, (
    "Duplicate track IDs in modeling dataset."
)

original_ids = set(original["msd_track_id"])
our_ids = set(ours["msd_track_id"])

matched_ids = our_ids.intersection(original_ids)
unmatched_ids = our_ids.difference(original_ids)

coverage = (
    100 * len(matched_ids) / len(our_ids)
    if len(our_ids) > 0 else 0
)

print("\nTRACK IDENTIFIER OVERLAP")
print("Original MSD-I tracks:", len(original_ids))
print("Modeling dataset tracks:", len(our_ids))
print("Matched tracks:", len(matched_ids))
print("Unmatched tracks:", len(unmatched_ids))
print(f"Coverage: {coverage:.2f}%")

# ------------------------------------------------------------
# 3. Compare genre labels
# ------------------------------------------------------------

comparison = ours[
    ["msd_track_id", "genre"]
].merge(
    original[["msd_track_id", "genre"]],
    on="msd_track_id",
    how="inner",
    suffixes=("_ours", "_original"),
    validate="one_to_one"
)

assert len(comparison) == len(matched_ids), (
    "Unexpected number of matched records."
)

assert comparison[
    ["genre_ours", "genre_original"]
].notna().all().all(), (
    "Missing genre labels among matched tracks."
)

comparison["genre_agreement"] = (
    comparison["genre_ours"]
    == comparison["genre_original"]
)

agreement_rate = (
    100 * comparison["genre_agreement"].mean()
    if len(comparison) > 0 else float("nan")
)

print("\nGENRE LABEL CONSISTENCY")
print("Matched observations:", len(comparison))
print(f"Genre agreement: {agreement_rate:.2f}%")

print(
    "Genre disagreements:",
    (~comparison["genre_agreement"]).sum()
)

# ------------------------------------------------------------
# 4. Inspect disagreements
# ------------------------------------------------------------

disagreements = comparison.loc[
    ~comparison["genre_agreement"],
    ["msd_track_id", "genre_ours", "genre_original"]
]

if len(disagreements) > 0:
    print("\nExample genre disagreements:")
    display(disagreements.head(10))
else:
    print("\nNo genre disagreements found.")

# ------------------------------------------------------------
# 5. Genre distributions
# ------------------------------------------------------------

print("\nORIGINAL MSD-I GENRE DISTRIBUTION")
display(original["genre"].value_counts())

print("\nMODELING DATASET GENRE DISTRIBUTION")
display(ours["genre"].value_counts())

print("\n" + "=" * 65)
print("METADATA VALIDATION COMPLETE")
print("=" * 65)



### Inspection of Audio–Visual Feature Matrices

We inspect the original MSD-I feature archive to identify the available audio and visual representations, their dimensionality, and the track-index files associated with each dataset split.

The index files are particularly important because audio and visual features must be matched to the correct observations before constructing the final multimodal dataset.

This inspection establishes the structure of the available data; the final feature alignment and selection are verified in subsequent steps.

In [ ]:
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.lib import format as npy_format

print("=" * 65)
print("AUDIO–VISUAL FEATURE MATRICES AND TRACK INDICES")
print("=" * 65)

# PROJECT is defined in Cell 1
archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

assert archive_path.is_file(), (
    "MSD-I feature archive not found."
)

# ------------------------------------------------------------
# 1. Helper: inspect NumPy header without loading the array
# ------------------------------------------------------------

def inspect_npy_header(file_obj):
    version = npy_format.read_magic(file_obj)

    if version == (1, 0):
        shape, fortran_order, dtype = (
            npy_format.read_array_header_1_0(file_obj)
        )
    elif version in [(2, 0), (3, 0)]:
        shape, fortran_order, dtype = (
            npy_format.read_array_header_2_0(file_obj)
        )
    else:
        raise ValueError(
            f"Unsupported NumPy format version: {version}"
        )

    return shape, dtype

# ------------------------------------------------------------
# 2. Inspect archive contents
# ------------------------------------------------------------

with tarfile.open(archive_path, "r:gz") as archive:

    members = {
        member.name: member
        for member in archive.getmembers()
        if member.isfile()
    }

    print("\nFEATURE MATRICES")
    print("-" * 65)

    feature_names = [
        f"X_{split}_{modality}_MSD-I.npy"
        for split in ["train", "val", "test"]
        for modality in ["audio", "visual", "mm-audio"]
    ]

    feature_results = []

    for filename in feature_names:

        matches = [
            name for name in members
            if Path(name).name == filename
        ]

        if len(matches) == 0:
            print(f"{filename}: NOT FOUND")
            continue

        assert len(matches) == 1, (
            f"Multiple archive entries found for {filename}"
        )

        member = members[matches[0]]

        with archive.extractfile(member) as file:
            shape, dtype = inspect_npy_header(file)

        feature_results.append({
            "filename": filename,
            "shape": str(shape),
            "dtype": str(dtype),
            "archive_path": matches[0]
        })

        print(
            f"{filename:35s} "
            f"shape={shape}, dtype={dtype}"
        )

    # --------------------------------------------------------
    # 3. Inspect available track-index files
    # --------------------------------------------------------

    print("\nTRACK INDEX FILES")
    print("-" * 65)

    index_files = sorted([
        name for name in members
        if Path(name).name.startswith("items_index_")
        and name.endswith(".tsv")
    ])

    assert index_files, (
        "No track-index files found in archive."
    )

    index_results = []

    for member_name in index_files:

        filename = Path(member_name).name

        with archive.extractfile(
            members[member_name]
        ) as file:

            index = pd.read_csv(
                file,
                sep="\t",
                nrows=3
            )

        index_results.append({
            "filename": filename,
            "columns": ", ".join(index.columns),
            "archive_path": member_name
        })

        print(f"\n{filename}")
        print("Archive path:", member_name)
        print("Columns:", index.columns.tolist())

        display(index)

# ------------------------------------------------------------
# 4. Summarize inspection
# ------------------------------------------------------------

feature_summary = pd.DataFrame(feature_results)
index_summary = pd.DataFrame(index_results)

print("\n" + "=" * 65)
print("FEATURE SUMMARY")
print("=" * 65)

display(feature_summary)

print("\nINDEX FILE SUMMARY")
display(index_summary)

print("\nIMPORTANT:")
print(
    "Matching matrix dimensions alone does not establish "
    "track-level alignment."
)
print(
    "Track identifiers must be checked before combining "
    "audio and visual features."
)

print("\nFEATURE INSPECTION COMPLETE")



### Audio Feature Coverage and Track Alignment

We examine the track indices provided with the original MSD-I feature archive and compare them with the identifiers in our modeling dataset.

This allows us to assess feature-index coverage, identify unmatched tracks, and inspect differences between the original dataset partitions and our artist-disjoint train, validation, and test splits.

The original split assignments are used only for verification. Our final modeling procedure retains the project-specific artist-disjoint splits to prevent artist overlap between training and evaluation.

Matching identifiers is a preliminary alignment check; the correspondence between feature rows and the final audio embeddings must also be verified.

In [ ]:
import tarfile
import pandas as pd
from pathlib import Path

print("=" * 65)
print("AUDIO FEATURE COVERAGE AND TRACK ALIGNMENT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1
archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

metadata_path = PROJECT / "final_modeling_dataset.csv"

assert archive_path.is_file(), "MSD-I archive not found."
assert metadata_path.is_file(), "Modeling dataset not found."

# ------------------------------------------------------------
# 2. Load our modeling metadata
# ------------------------------------------------------------

ours = pd.read_csv(metadata_path)

required_columns = {"msd_track_id", "genre", "split"}

assert required_columns.issubset(ours.columns), (
    "Required metadata columns are missing."
)

assert ours["msd_track_id"].notna().all(), (
    "Missing track identifiers in modeling dataset."
)

assert ours["msd_track_id"].is_unique, (
    "Duplicate track identifiers in modeling dataset."
)

assert ours["split"].notna().all(), (
    "Missing project split assignments."
)

assert set(ours["split"]) == {"train", "val", "test"}, (
    "Unexpected project split labels."
)

print("\nOUR MODELING DATASET")
print("Total tracks:", len(ours))
print("Unique track IDs:", ours["msd_track_id"].nunique())

print("\nProject split distribution:")
display(ours["split"].value_counts())

# ------------------------------------------------------------
# 3. Read original MSD-I track indices
# ------------------------------------------------------------

all_indices = []

with tarfile.open(archive_path, "r:gz") as archive:

    for split in ["train", "val", "test"]:

        filename = (
            f"msdi-data/splits/"
            f"items_index_{split}_MSD-I.tsv"
        )

        member = archive.getmember(filename)

        with archive.extractfile(member) as file:

            # Original index files contain track IDs
            # without a column header.
            index = pd.read_csv(
                file,
                sep="\t",
                header=None,
                names=["msd_track_id"],
                dtype={"msd_track_id": "string"}
            )

        assert index["msd_track_id"].notna().all(), (
            f"Missing IDs in original {split} index."
        )

        assert index["msd_track_id"].is_unique, (
            f"Duplicate IDs in original {split} index."
        )

        index["original_split"] = split

        # Row position in the general split index.
        # This is NOT necessarily the row position
        # in a modality-specific feature matrix.
        index["general_index_row"] = range(len(index))

        print(f"\nORIGINAL {split.upper()} SPLIT")
        print("Indexed tracks:", len(index))
        print(
            "Example track IDs:",
            index["msd_track_id"].head(3).tolist()
        )

        all_indices.append(index)

# ------------------------------------------------------------
# 4. Combine original indices
# ------------------------------------------------------------

combined = pd.concat(
    all_indices,
    ignore_index=True
)

assert combined["msd_track_id"].is_unique, (
    "Track IDs appear multiple times across original splits."
)

print("\n" + "=" * 65)
print("ORIGINAL MSD-I INDEX SUMMARY")
print("=" * 65)

print("Total indexed tracks:", len(combined))
print(
    "Unique track IDs:",
    combined["msd_track_id"].nunique()
)

# ------------------------------------------------------------
# 5. Match our dataset to original indices
# ------------------------------------------------------------

matched = ours.merge(
    combined,
    on="msd_track_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(matched) == len(ours), (
    "Track count changed during metadata matching."
)

n_matched = (matched["_merge"] == "both").sum()
n_unmatched = (matched["_merge"] == "left_only").sum()

coverage = 100 * n_matched / len(ours)

print("\n" + "=" * 65)
print("FEATURE INDEX COVERAGE")
print("=" * 65)

print("Our dataset tracks:", len(ours))
print("Matched tracks:", n_matched)
print("Unmatched tracks:", n_unmatched)
print(f"Coverage: {coverage:.2f}%")

if n_unmatched > 0:
    print("\nExample unmatched track IDs:")
    display(
        matched.loc[
            matched["_merge"] == "left_only",
            ["msd_track_id", "genre", "split"]
        ].head(10)
    )

# ------------------------------------------------------------
# 6. Compare original and project-specific splits
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ORIGINAL VS PROJECT SPLIT ASSIGNMENTS")
print("=" * 65)

split_comparison = pd.crosstab(
    matched["split"],
    matched["original_split"],
    dropna=False
)

display(split_comparison)

# ------------------------------------------------------------
# 7. Coverage by project split
# ------------------------------------------------------------

coverage_by_split = (
    matched.groupby("split")
    .agg(
        total_tracks=("msd_track_id", "size"),
        matched_tracks=(
            "_merge",
            lambda x: (x == "both").sum()
        )
    )
)

coverage_by_split["coverage_pct"] = (
    100
    * coverage_by_split["matched_tracks"]
    / coverage_by_split["total_tracks"]
)

print("\nCOVERAGE BY PROJECT SPLIT")
display(coverage_by_split.round(2))

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ALIGNMENT AUDIT COMPLETE")
print("=" * 65)

print(
    "Original MSD-I split assignments were used "
    "only for source-index verification."
)

print(
    "The project's artist-disjoint split assignments "
    "remain unchanged."
)

print(
    "Modality-specific audio and visual index alignment "
    "must be verified separately before extracting features."
)



In [ ]:
!pip -q install pymupdf

### Verification of Audio Feature Alignment

Before constructing the multimodal dataset, we examine the track indices associated with the original MSD-I audio features.

We compare the general dataset indices with the audio-specific indices to determine whether they contain the same tracks in the same order. This check is important because precomputed feature matrices must be associated with the correct track identifiers.

If the indices differ, features must be aligned explicitly using track identifiers rather than assuming that row positions correspond.

The final aligned dataset is subsequently validated to ensure that each observation contains the correct audio and visual representations.

In [ ]:

# ============================================================
# PHASE 5 — VERIFY MSD-I AUDIO FEATURE INDEX ALIGNMENT
# ============================================================

import tarfile
import pandas as pd

print("=" * 65)
print("MSD-I AUDIO FEATURE INDEX ALIGNMENT")
print("=" * 65)

# PROJECT is defined in Cell 1
archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

assert archive_path.is_file(), (
    "Original MSD-I archive not found."
)

# ------------------------------------------------------------
# 1. Load and compare track indices
# ------------------------------------------------------------

alignment_results = []

with tarfile.open(archive_path, "r:gz") as archive:

    for split in ["train", "val"]:

        general_name = (
            f"msdi-data/splits/"
            f"items_index_{split}_MSD-I.tsv"
        )

        audio_name = (
            f"msdi-data/train_data/"
            f"items_index_{split}_audio_MSD-I.tsv"
        )

        with archive.extractfile(general_name) as file:
            general = pd.read_csv(
                file,
                sep="\t",
                header=None,
                dtype="string"
            ).iloc[:, 0].reset_index(drop=True)

        with archive.extractfile(audio_name) as file:
            audio = pd.read_csv(
                file,
                sep="\t",
                header=None,
                dtype="string"
            ).iloc[:, 0].reset_index(drop=True)

        # ----------------------------------------------------
        # 2. Validate identifiers
        # ----------------------------------------------------

        assert general.notna().all(), (
            f"Missing general track IDs in {split}."
        )

        assert audio.notna().all(), (
            f"Missing audio track IDs in {split}."
        )

        assert general.is_unique, (
            f"Duplicate general track IDs in {split}."
        )

        assert audio.is_unique, (
            f"Duplicate audio track IDs in {split}."
        )

        # ----------------------------------------------------
        # 3. Compare track coverage and ordering
        # ----------------------------------------------------

        general_ids = set(general)
        audio_ids = set(audio)

        same_length = len(general) == len(audio)
        same_tracks = general_ids == audio_ids
        same_order = general.equals(audio)

        missing_from_audio = general_ids - audio_ids
        extra_in_audio = audio_ids - general_ids

        print(f"\n{split.upper()} SPLIT")
        print("-" * 45)

        print("General index rows:", len(general))
        print("Audio index rows:", len(audio))

        print("Same number of rows:", same_length)
        print("Same track ID set:", same_tracks)
        print("Identical track order:", same_order)

        print(
            "Tracks missing from audio index:",
            len(missing_from_audio)
        )

        print(
            "Extra tracks in audio index:",
            len(extra_in_audio)
        )

        # ----------------------------------------------------
        # 4. Interpret results
        # ----------------------------------------------------

        if same_order:
            print(
                "\nResult: General and audio indices "
                "have identical track ordering."
            )

        elif same_tracks:
            print(
                "\nResult: The same tracks are present, "
                "but their ordering differs."
            )
            print(
                "Explicit track-ID alignment is required."
            )

        else:
            print(
                "\nResult: The indices contain "
                "different track sets."
            )
            print(
                "Match available tracks explicitly "
                "and investigate missing observations."
            )

        alignment_results.append({
            "split": split,
            "general_rows": len(general),
            "audio_rows": len(audio),
            "same_length": same_length,
            "same_track_set": same_tracks,
            "same_order": same_order,
            "missing_from_audio": len(missing_from_audio),
            "extra_in_audio": len(extra_in_audio)
        })

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

alignment_summary = pd.DataFrame(alignment_results)

print("\n" + "=" * 65)
print("AUDIO INDEX ALIGNMENT SUMMARY")
print("=" * 65)

display(alignment_summary)

print("\nIMPORTANT:")
print(
    "This check compares original MSD-I track indices."
)
print(
    "Feature-matrix row counts and ordering must also "
    "be verified before extracting embeddings."
)
print(
    "Our final artist-disjoint split assignments "
    "are not changed by this inspection."
)

print("\nAUDIO INDEX VERIFICATION COMPLETE")

### Feature Quality Validation

We inspect the available MSD-I feature matrices to verify their dimensions and numerical integrity. In particular, we check for missing (`NaN`) and infinite values, which could interfere with subsequent model training.

Feature quality and feature alignment are treated as separate validation tasks. Numerical validity alone does not establish that feature rows correspond to the correct track identifiers.

The final aligned audio–visual representations are validated separately before model fitting.

In [ ]:
import tarfile
import io
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("MSD-I FEATURE QUALITY VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Locate feature archive
# ------------------------------------------------------------

archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

assert archive_path.is_file(), (
    "Original MSD-I feature archive not found."
)

modalities = [
    "audio",
    "visual",
    "mm-audio",
    "mm-visual"
]

splits = ["train", "val", "test"]

quality_results = []
missing_files = []

# ------------------------------------------------------------
# 2. Inspect feature matrices
# ------------------------------------------------------------

with tarfile.open(archive_path, "r:gz") as archive:

    members = {
        member.name: member
        for member in archive.getmembers()
        if member.isfile()
    }

    for split in splits:

        print(f"\n{split.upper()} SPLIT")
        print("-" * 50)

        for modality in modalities:

            filename = (
                f"X_{split}_{modality}_MSD-I.npy"
            )

            matches = [
                name for name in members
                if Path(name).name == filename
            ]

            if len(matches) == 0:
                missing_files.append(filename)
                print(f"{modality}: NOT AVAILABLE")
                continue

            assert len(matches) == 1, (
                f"Multiple entries found for {filename}"
            )

            member = members[matches[0]]

            with archive.extractfile(member) as file:
                X = np.load(
                    io.BytesIO(file.read()),
                    allow_pickle=False
                )

            assert np.issubdtype(
                X.dtype, np.number
            ), f"Non-numeric features in {filename}"

            nan_count = int(np.isnan(X).sum())
            inf_count = int(np.isinf(X).sum())
            all_finite = bool(np.isfinite(X).all())

            quality_results.append({
                "split": split,
                "modality": modality,
                "shape": str(X.shape),
                "dtype": str(X.dtype),
                "nan_values": nan_count,
                "infinite_values": inf_count,
                "all_finite": all_finite
            })

            print(f"\n{modality.upper()}")
            print("Shape:", X.shape)
            print("Data type:", X.dtype)
            print("NaN values:", nan_count)
            print("Infinite values:", inf_count)
            print("All values finite:", all_finite)

            del X

# ------------------------------------------------------------
# 3. Display summary
# ------------------------------------------------------------

quality_summary = pd.DataFrame(quality_results)

assert not quality_summary.empty, (
    "No feature matrices were found."
)

print("\n" + "=" * 65)
print("FEATURE QUALITY SUMMARY")
print("=" * 65)

display(quality_summary)

# ------------------------------------------------------------
# 4. Check numerical integrity
# ------------------------------------------------------------

invalid_matrices = quality_summary.loc[
    ~quality_summary["all_finite"]
]

if invalid_matrices.empty:
    print("\nAll inspected feature matrices contain finite values.")
else:
    print("\nWARNING: Invalid feature values detected.")
    display(invalid_matrices)

print("\nUnavailable feature files:", len(missing_files))

if missing_files:
    print("Missing matrices:")
    for filename in missing_files:
        print(" -", filename)

assert invalid_matrices.empty, (
    "Some feature matrices contain NaN or infinite values."
)

print("\nIMPORTANT:")
print(
    "Numerical validity does not establish track-level "
    "feature alignment."
)
print(
    "The final artist-disjoint audio–visual matrices "
    "must be validated separately."
)

print("\nFEATURE QUALITY VALIDATION COMPLETE")


### Construction of the Track-Level Feature Index

To prepare the multimodal dataset, we match our modeling metadata with the track indices supplied in the original MSD-I archive.

For each matched track, we record its original dataset partition and corresponding feature-row index. These references allow us to locate precomputed feature representations in the source archive.

We verify that track identifiers are unique and that all observations are successfully matched. Because the source archive may contain modality-specific indices, the correspondence between these row references and the audio and visual feature matrices must be checked before extracting embeddings.

The original partition labels are retained for feature lookup only. Model training and evaluation use our separately constructed artist-disjoint splits.

In [ ]:
import pandas as pd
import tarfile
from pathlib import Path

print("=" * 65)
print("TRACK-LEVEL FEATURE INDEX CONSTRUCTION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

metadata_path = (
    PROJECT / "final_modeling_dataset.csv"
)

output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_original_split"
)

assert archive_path.is_file(), (
    "Original MSD-I feature archive not found."
)

assert metadata_path.is_file(), (
    "Final modeling metadata not found."
)

output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Load project metadata
# ------------------------------------------------------------

ours = pd.read_csv(metadata_path)

required_columns = {
    "msd_track_id",
    "genre",
    "split"
}

assert required_columns.issubset(ours.columns), (
    "Required columns missing from modeling dataset."
)

assert ours["msd_track_id"].notna().all(), (
    "Missing track IDs in modeling dataset."
)

assert ours["msd_track_id"].is_unique, (
    "Duplicate track IDs in modeling dataset."
)

assert ours["split"].notna().all(), (
    "Missing project split assignments."
)

assert set(ours["split"]) == {"train", "val", "test"}, (
    "Unexpected project split labels."
)

print("\nPROJECT MODELING DATASET")
print("Tracks:", len(ours))

print("\nArtist-disjoint project split labels:")
display(ours["split"].value_counts())

# ------------------------------------------------------------
# 3. Helper to read track-index files
# ------------------------------------------------------------

def read_track_index(archive, member_name):

    with archive.extractfile(member_name) as file:
        index = pd.read_csv(
            file,
            sep="\t",
            header=None,
            dtype="string"
        )

    assert index.shape[1] >= 1, (
        f"No identifier column in {member_name}"
    )

    track_ids = index.iloc[:, 0].reset_index(drop=True)

    assert track_ids.notna().all(), (
        f"Missing track IDs in {member_name}"
    )

    assert track_ids.is_unique, (
        f"Duplicate track IDs in {member_name}"
    )

    return track_ids

# ------------------------------------------------------------
# 4. Load original general and modality indices
# ------------------------------------------------------------

general_frames = []
modality_frames = []

with tarfile.open(archive_path, "r:gz") as archive:

    available_members = {
        member.name
        for member in archive.getmembers()
        if member.isfile()
    }

    for original_split in ["train", "val", "test"]:

        general_name = (
            f"msdi-data/splits/"
            f"items_index_{original_split}_MSD-I.tsv"
        )

        assert general_name in available_members, (
            f"Missing general index: {general_name}"
        )

        general_ids = read_track_index(
            archive,
            general_name
        )

        general_frame = pd.DataFrame({
            "msd_track_id": general_ids,
            "original_split": original_split,
            "general_index_row": range(len(general_ids))
        })

        general_frames.append(general_frame)

        print(
            f"\nOriginal {original_split}: "
            f"{len(general_frame)} general index rows"
        )

        # Inspect modality-specific indices if available.
        for modality in [
            "audio",
            "visual",
            "mm-audio",
            "mm-visual"
        ]:

            modality_name = (
                f"msdi-data/train_data/"
                f"items_index_{original_split}_"
                f"{modality}_MSD-I.tsv"
            )

            if modality_name not in available_members:
                print(
                    f"  {modality}: "
                    "no modality-specific index found"
                )
                continue

            modality_ids = read_track_index(
                archive,
                modality_name
            )

            modality_frame = pd.DataFrame({
                "msd_track_id": modality_ids,
                "original_split": original_split,
                "modality": modality,
                "modality_index_row": range(
                    len(modality_ids)
                )
            })

            modality_frames.append(modality_frame)

            print(
                f"  {modality}: "
                f"{len(modality_frame)} indexed rows"
            )

# ------------------------------------------------------------
# 5. Validate original general indices
# ------------------------------------------------------------

original_index = pd.concat(
    general_frames,
    ignore_index=True
)

assert original_index["msd_track_id"].is_unique, (
    "Duplicate track IDs across original splits."
)

assert original_index["general_index_row"].ge(0).all()

# ------------------------------------------------------------
# 6. Match project tracks to original indices
# ------------------------------------------------------------

aligned = ours.merge(
    original_index,
    on="msd_track_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(aligned) == len(ours), (
    "Merge changed the number of observations."
)

unmatched = aligned["_merge"] != "both"

print("\nGENERAL INDEX COVERAGE")
print("Matched tracks:", int((~unmatched).sum()))
print("Unmatched tracks:", int(unmatched.sum()))

if unmatched.any():
    display(
        aligned.loc[
            unmatched,
            ["msd_track_id", "genre", "split"]
        ].head(10)
    )

assert not unmatched.any(), (
    "Some project tracks are absent from original indices."
)

aligned = aligned.drop(columns="_merge")

aligned["general_index_row"] = (
    aligned["general_index_row"].astype(int)
)

# ------------------------------------------------------------
# 7. Add modality-specific row references
# ------------------------------------------------------------

if modality_frames:

    modality_index = pd.concat(
        modality_frames,
        ignore_index=True
    )

    for modality in sorted(
        modality_index["modality"].unique()
    ):

        subset = modality_index.loc[
            modality_index["modality"] == modality,
            [
                "msd_track_id",
                "original_split",
                "modality_index_row"
            ]
        ].copy()

        column_name = (
            modality.replace("-", "_") + "_index_row"
        )

        subset = subset.rename(
            columns={
                "modality_index_row": column_name
            }
        )

        assert not subset.duplicated(
            ["msd_track_id", "original_split"]
        ).any(), (
            f"Duplicate {modality} index references."
        )

        aligned = aligned.merge(
            subset,
            on=["msd_track_id", "original_split"],
            how="left",
            validate="one_to_one"
        )

        matched_count = int(
            aligned[column_name].notna().sum()
        )

        print(
            f"{modality} index coverage: "
            f"{matched_count}/{len(aligned)}"
        )

# ------------------------------------------------------------
# 8. Validate final lookup metadata
# ------------------------------------------------------------

assert len(aligned) == len(ours)
assert aligned["msd_track_id"].is_unique

assert aligned["original_split"].notna().all()
assert aligned["general_index_row"].notna().all()

# Confirm project split assignments were preserved.
original_project_splits = ours.set_index(
    "msd_track_id"
)["split"]

aligned_project_splits = aligned.set_index(
    "msd_track_id"
)["split"]

assert original_project_splits.sort_index().equals(
    aligned_project_splits.sort_index()
), "Project split assignments changed unexpectedly."

# ------------------------------------------------------------
# 9. Save lookup metadata
# ------------------------------------------------------------

output_path = (
    output_dir / "multimodal_aligned_dataset.csv"
)

aligned.to_csv(output_path, index=False)

print("\n" + "=" * 65)
print("FINAL LOOKUP SUMMARY")
print("=" * 65)

print("\nTotal matched tracks:", len(aligned))

print("\nOriginal MSD-I split distribution:")
display(aligned["original_split"].value_counts())

print("\nProject vs original split:")
display(
    pd.crosstab(
        aligned["split"],
        aligned["original_split"]
    )
)

print("\nSample track-level lookup records:")

display(
    aligned.head(10)
)

print("\nSaved metadata:")
print(output_path)

print("\nIMPORTANT:")
print(
    "This CSV contains metadata and source row references, "
    "not extracted feature embeddings."
)

print(
    "Modality-specific row references must be verified "
    "against the corresponding feature matrices."
)

print(
    "Project-specific artist-disjoint split labels "
    "have been preserved."
)

print("\nTRACK-LEVEL INDEX CONSTRUCTION COMPLETE")


### Extraction and Alignment of Audio and Visual Embeddings

We extract precomputed 2,048-dimensional audio and visual representations from the original MSD-I feature archive.

Each observation is matched to its corresponding feature vector using track identifiers and the appropriate modality-specific index. We verify identifier uniqueness, matrix dimensions, numerical integrity, and consistency between feature matrices and their associated indices.

The original MSD-I partitions are used only to locate the source representations. After extraction, observations are reorganized according to our project-specific artist-disjoint train, validation, and test assignments.

We verify that artists do not appear across multiple project partitions and save the audio and visual feature matrices together with their corresponding track metadata in consistent row order.

These aligned matrices provide the inputs for the subsequent audio-only, visual-only, and multimodal classification experiments.

In [ ]:

# CELL 15 — REUSE VERIFIED ORIGINAL MULTIMODAL DATA

import numpy as np
import pandas as pd
import shutil
from pathlib import Path

source = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_original_split"
)

output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

output_dir.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(
    PROJECT / "final_modeling_dataset.csv"
)

print("=" * 65)
print("PREPARING VERIFIED MULTIMODAL DATA")
print("=" * 65)

artist_sets = {}
all_track_ids = []

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        source / f"labels_{split}.csv"
    )

    audio = np.load(
        source / f"X_{split}_audio.npy",
        mmap_mode="r"
    )

    visual = np.load(
        source / f"X_{split}_visual.npy",
        mmap_mode="r"
    )

    assert len(labels) == len(audio) == len(visual)
    assert audio.shape[1] == visual.shape[1] == 2048
    assert labels["msd_track_id"].is_unique

    labels = labels.merge(
        metadata[
            ["msd_track_id", "genre", "msd_artist_id"]
        ],
        on="msd_track_id",
        suffixes=("", "_check"),
        validate="one_to_one"
    )

    assert (
        labels["genre"] == labels["genre_check"]
    ).all()

    assert (
        labels["msd_artist_id"]
        == labels["msd_artist_id_check"]
    ).all()

    labels = labels.drop(
        columns=["genre_check", "msd_artist_id_check"]
    )

    labels["split"] = split
    labels["original_split"] = split

    artist_sets[split] = set(
        labels["msd_artist_id"].astype(str)
    )

    all_track_ids.extend(
        labels["msd_track_id"].tolist()
    )

    for modality in ["audio", "visual"]:

        filename = f"X_{split}_{modality}.npy"

        shutil.copy2(
            source / filename,
            output_dir / filename
        )

    labels.to_csv(
        output_dir / f"labels_{split}.csv",
        index=False
    )

    print(
        f"{split.upper():5s} | "
        f"{len(labels):5d} tracks | "
        f"{len(artist_sets[split]):5d} artists"
    )

assert not (
    artist_sets["train"] & artist_sets["val"]
)

assert not (
    artist_sets["train"] & artist_sets["test"]
)

assert not (
    artist_sets["val"] & artist_sets["test"]
)

assert len(all_track_ids) == 11870
assert len(set(all_track_ids)) == 11870
assert set(all_track_ids) == set(metadata["msd_track_id"])

print("\nArtist-disjoint validation: PASSED")
print("Track and label validation: PASSED")
print("Saved to:", output_dir)
print("\nCELL 15 COMPLETE")


### Audio-Only Baseline: Logistic Regression

We establish an audio-only classification baseline using the precomputed 2,048-dimensional audio embeddings.

A multinomial logistic regression classifier is trained using standardized features. Regularization strength is selected from \(C \in \{0.001, 0.01, 0.1, 1, 10\}\) according to validation macro F1.

Feature standardization is fitted exclusively on the training data to avoid data leakage. The audio-only model provides a reference against which the contribution of visual information can be evaluated.

The final baseline is trained and evaluated using the artist-disjoint dataset partitions.

In [ ]:

# ============================================================
# PHASE 5 — AUDIO-ONLY LOGISTIC REGRESSION BASELINE
# ARTIST-DISJOINT TRAINING AND VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("AUDIO-ONLY LOGISTIC REGRESSION")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is already defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint feature directory not found. "
    "Run Cell 15 first."
)

# ------------------------------------------------------------
# 2. Load audio features and metadata
# ------------------------------------------------------------

X_train = np.load(
    output_dir / "X_train_audio.npy",
    allow_pickle=False
)

X_val = np.load(
    output_dir / "X_val_audio.npy",
    allow_pickle=False
)

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for name, labels in [
    ("train", train_labels),
    ("val", val_labels)
]:
    assert required_columns.issubset(labels.columns), (
        f"Missing metadata columns in {name} labels."
    )

    assert labels[list(required_columns)].notna().all().all(), (
        f"Missing metadata values in {name} labels."
    )

    assert labels["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {name} labels."
    )

    assert (labels["split"] == name).all(), (
        f"Incorrect split labels in {name} metadata."
    )

y_train = train_labels["genre"].to_numpy()
y_val = val_labels["genre"].to_numpy()

# ------------------------------------------------------------
# 3. Validate feature matrices
# ------------------------------------------------------------

assert X_train.ndim == 2
assert X_val.ndim == 2

assert X_train.shape == (len(y_train), 2048), (
    f"Unexpected training shape: {X_train.shape}"
)

assert X_val.shape == (len(y_val), 2048), (
    f"Unexpected validation shape: {X_val.shape}"
)

assert np.isfinite(X_train).all(), (
    "Training features contain invalid values."
)

assert np.isfinite(X_val).all(), (
    "Validation features contain invalid values."
)

assert set(y_val).issubset(set(y_train)), (
    "Validation contains genres absent from training."
)

train_artists = set(train_labels["msd_artist_id"])
val_artists = set(val_labels["msd_artist_id"])

assert train_artists.isdisjoint(val_artists), (
    "Artist leakage detected between training "
    "and validation."
)

assert set(train_labels["msd_track_id"]).isdisjoint(
    set(val_labels["msd_track_id"])
), "Track overlap detected between train and validation."

print("\nDATASET VALIDATION")
print("-" * 45)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

print("Training artists:", len(train_artists))
print("Validation artists:", len(val_artists))

print("Artist overlap:", len(train_artists & val_artists))

print("\nTraining genre distribution:")
display(train_labels["genre"].value_counts())

print("\nValidation genre distribution:")
display(val_labels["genre"].value_counts())

# ------------------------------------------------------------
# 4. Hyperparameter selection
# ------------------------------------------------------------

C_values = [0.001, 0.01, 0.1, 1, 10]

results = []

best_model = None
best_f1 = -np.inf
best_C = None
best_predictions = None

print("\n" + "=" * 65)
print("VALIDATION PERFORMANCE BY REGULARIZATION")
print("=" * 65)

for C in C_values:

    # The scaler is fitted only on training data.
    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced = balanced_accuracy_score(
        y_val,
        predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "C": C,
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "macro_f1": macro_f1
    })

    print(
        f"C={C:<6} | "
        f"Accuracy={accuracy:.4f} | "
        f"Balanced Accuracy={balanced:.4f} | "
        f"Macro F1={macro_f1:.4f}"
    )

    # Strict comparison preserves the first C in a tie.
    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_C = C
        best_model = model
        best_predictions = predictions.copy()

# ------------------------------------------------------------
# 5. Save validation results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_path = (
    output_dir / "audio_logistic_validation_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# 6. Save selected model
# ------------------------------------------------------------

model_path = (
    output_dir / "audio_logistic_best.joblib"
)

joblib.dump(
    best_model,
    model_path
)

# ------------------------------------------------------------
# 7. Save validation predictions
# ------------------------------------------------------------

predictions_df = val_labels.copy()

predictions_df["audio_prediction"] = (
    best_predictions
)

predictions_df["audio_correct"] = (
    best_predictions == y_val
)

predictions_path = (
    output_dir
    / "audio_logistic_validation_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST AUDIO-ONLY MODEL")
print("=" * 65)

print("Selected C:", best_C)
print("Validation macro F1:", round(best_f1, 4))

print("\nValidation results:")
display(
    results_df.sort_values(
        "macro_f1",
        ascending=False
    ).reset_index(drop=True)
)

print("\nSaved files:")
print("-", results_path.name)
print("-", model_path.name)
print("-", predictions_path.name)

print("\nIMPORTANT:")
print(
    "Regularization was selected using "
    "artist-disjoint validation data."
)

print(
    "The held-out test set was not used "
    "for hyperparameter selection."
)

print(
    "Feature standardization was fitted "
    "on training observations only."
)

print("\nAUDIO-ONLY BASELINE COMPLETE")

### Visual-Only Baseline: Logistic Regression

We train a visual-only classifier using the precomputed 2,048-dimensional album-cover embeddings.

The model uses multinomial logistic regression with standardized features. We evaluate regularization strengths \(C \in \{0.001, 0.01, 0.1, 1, 10\}\) and select the best configuration according to validation macro F1.

This baseline measures the predictive information contained in album-cover images independently of audio features. It provides a reference for evaluating whether multimodal fusion improves classification performance.

The final visual-only baseline is evaluated using the same artist-disjoint partitions as the other models.

In [ ]:

# ============================================================
# PHASE 5 — VISUAL-ONLY LOGISTIC REGRESSION BASELINE
# ARTIST-DISJOINT TRAINING AND VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("VISUAL-ONLY LOGISTIC REGRESSION")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint feature directory not found. "
    "Run Cell 15 first."
)

# ------------------------------------------------------------
# 2. Load visual features and labels
# ------------------------------------------------------------

X_train = np.load(
    output_dir / "X_train_visual.npy",
    allow_pickle=False
)

X_val = np.load(
    output_dir / "X_val_visual.npy",
    allow_pickle=False
)

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for name, labels in [
    ("train", train_labels),
    ("val", val_labels)
]:

    assert required_columns.issubset(labels.columns), (
        f"Missing metadata columns in {name} labels."
    )

    assert labels[list(required_columns)].notna().all().all(), (
        f"Missing metadata values in {name} labels."
    )

    assert labels["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {name} labels."
    )

    assert (labels["split"] == name).all(), (
        f"Incorrect split assignments in {name} metadata."
    )

y_train = train_labels["genre"].to_numpy()
y_val = val_labels["genre"].to_numpy()

# ------------------------------------------------------------
# 3. Validate feature matrices and artist separation
# ------------------------------------------------------------

assert X_train.shape == (len(y_train), 2048), (
    f"Unexpected training shape: {X_train.shape}"
)

assert X_val.shape == (len(y_val), 2048), (
    f"Unexpected validation shape: {X_val.shape}"
)

assert np.isfinite(X_train).all(), (
    "Training features contain invalid values."
)

assert np.isfinite(X_val).all(), (
    "Validation features contain invalid values."
)

assert set(y_val).issubset(set(y_train)), (
    "Validation contains genres absent from training."
)

train_artists = set(train_labels["msd_artist_id"])
val_artists = set(val_labels["msd_artist_id"])

assert train_artists.isdisjoint(val_artists), (
    "Artist leakage detected between training "
    "and validation."
)

assert set(train_labels["msd_track_id"]).isdisjoint(
    set(val_labels["msd_track_id"])
), "Track overlap detected between train and validation."

print("\nDATASET VALIDATION")
print("-" * 45)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

print("Training artists:", len(train_artists))
print("Validation artists:", len(val_artists))
print("Artist overlap:", len(train_artists & val_artists))

print("\nTraining genre distribution:")
display(train_labels["genre"].value_counts())

print("\nValidation genre distribution:")
display(val_labels["genre"].value_counts())

# ------------------------------------------------------------
# 4. Hyperparameter selection
# ------------------------------------------------------------

C_values = [0.001, 0.01, 0.1, 1, 10]

results = []

best_model = None
best_f1 = -np.inf
best_C = None
best_predictions = None

print("\n" + "=" * 65)
print("VALIDATION PERFORMANCE BY REGULARIZATION")
print("=" * 65)

for C in C_values:

    # StandardScaler is fitted on training data only.
    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced = balanced_accuracy_score(
        y_val,
        predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "C": C,
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "macro_f1": macro_f1
    })

    print(
        f"C={C:<6} | "
        f"Accuracy={accuracy:.4f} | "
        f"Balanced Accuracy={balanced:.4f} | "
        f"Macro F1={macro_f1:.4f}"
    )

    # Preserve the first C if validation scores tie.
    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_C = C
        best_model = model
        best_predictions = predictions.copy()

# ------------------------------------------------------------
# 5. Save validation results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_path = (
    output_dir / "visual_logistic_validation_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# 6. Save selected model
# ------------------------------------------------------------

model_path = (
    output_dir / "visual_logistic_best.joblib"
)

joblib.dump(
    best_model,
    model_path
)

# ------------------------------------------------------------
# 7. Save validation predictions
# ------------------------------------------------------------

predictions_df = val_labels.copy()

predictions_df["visual_prediction"] = (
    best_predictions
)

predictions_df["visual_correct"] = (
    best_predictions == y_val
)

predictions_path = (
    output_dir
    / "visual_logistic_validation_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST VISUAL-ONLY MODEL")
print("=" * 65)

print("Selected C:", best_C)
print("Validation macro F1:", round(best_f1, 4))

print("\nValidation results:")
display(
    results_df.sort_values(
        "macro_f1",
        ascending=False
    ).reset_index(drop=True)
)

print("\nSaved files:")
print("-", results_path.name)
print("-", model_path.name)
print("-", predictions_path.name)

print("\nIMPORTANT:")
print(
    "Regularization was selected using "
    "artist-disjoint validation data."
)

print(
    "The held-out test set was not used "
    "for hyperparameter selection."
)

print(
    "Feature standardization was fitted "
    "on training observations only."
)

print("\nVISUAL-ONLY BASELINE COMPLETE")

### Early Fusion: Combining Audio and Visual Features

Early fusion combines the audio and visual representations before classification.

Each track is represented by a 2,048-dimensional audio embedding and a 2,048-dimensional visual embedding. Concatenating these representations produces a 4,096-dimensional multimodal feature vector.

We train logistic regression on the combined representation, applying standardization fitted exclusively on the training data. Regularization strength is selected using validation macro F1.

This approach allows the classifier to learn how audio and visual features jointly contribute to genre prediction. Its performance is compared against the individual audio-only and visual-only baselines to assess whether combining modalities improves classification.

The final comparison uses the same artist-disjoint data partitions across all models.

In [ ]:

# ============================================================
# PHASE 5 — EARLY-FUSION LOGISTIC REGRESSION
# ARTIST-DISJOINT TRAINING AND VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("EARLY-FUSION MULTIMODAL LOGISTIC REGRESSION")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint feature directory not found. "
    "Run Cell 15 first."
)

# ------------------------------------------------------------
# 2. Load audio and visual embeddings
# ------------------------------------------------------------

audio_train = np.load(
    output_dir / "X_train_audio.npy",
    allow_pickle=False
)

visual_train = np.load(
    output_dir / "X_train_visual.npy",
    allow_pickle=False
)

audio_val = np.load(
    output_dir / "X_val_audio.npy",
    allow_pickle=False
)

visual_val = np.load(
    output_dir / "X_val_visual.npy",
    allow_pickle=False
)

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for split, labels in [
    ("train", train_labels),
    ("val", val_labels)
]:
    assert required_columns.issubset(labels.columns), (
        f"Missing metadata columns in {split} labels."
    )

    assert labels[list(required_columns)].notna().all().all(), (
        f"Missing metadata values in {split} labels."
    )

    assert labels["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {split} labels."
    )

    assert (labels["split"] == split).all(), (
        f"Incorrect split assignments in {split} labels."
    )

y_train = train_labels["genre"].to_numpy()
y_val = val_labels["genre"].to_numpy()

# ------------------------------------------------------------
# 3. Validate feature dimensions and alignment
# ------------------------------------------------------------

assert audio_train.shape == (len(y_train), 2048), (
    f"Unexpected training audio shape: {audio_train.shape}"
)

assert visual_train.shape == (len(y_train), 2048), (
    f"Unexpected training visual shape: {visual_train.shape}"
)

assert audio_val.shape == (len(y_val), 2048), (
    f"Unexpected validation audio shape: {audio_val.shape}"
)

assert visual_val.shape == (len(y_val), 2048), (
    f"Unexpected validation visual shape: {visual_val.shape}"
)

for name, X in [
    ("training audio", audio_train),
    ("training visual", visual_train),
    ("validation audio", audio_val),
    ("validation visual", visual_val)
]:
    assert np.isfinite(X).all(), (
        f"Invalid values in {name} embeddings."
    )

assert set(y_val).issubset(set(y_train)), (
    "Validation contains genres absent from training."
)

# Check artist-disjoint validation.
train_artists = set(train_labels["msd_artist_id"])
val_artists = set(val_labels["msd_artist_id"])

assert train_artists.isdisjoint(val_artists), (
    "Artist leakage between training and validation."
)

assert set(train_labels["msd_track_id"]).isdisjoint(
    set(val_labels["msd_track_id"])
), "Track overlap between training and validation."

print("\nDATASET VALIDATION")
print("-" * 45)

print("Training audio:", audio_train.shape)
print("Training visual:", visual_train.shape)

print("Validation audio:", audio_val.shape)
print("Validation visual:", visual_val.shape)

print("Artist overlap:", len(train_artists & val_artists))

# Audio and visual matrices were extracted in the same
# metadata order in Cell 15. Shape checks here verify
# consistency with those saved metadata rows.

# ------------------------------------------------------------
# 4. Concatenate modalities
# ------------------------------------------------------------

X_train = np.concatenate(
    [audio_train, visual_train],
    axis=1
)

X_val = np.concatenate(
    [audio_val, visual_val],
    axis=1
)

assert X_train.shape == (len(y_train), 4096)
assert X_val.shape == (len(y_val), 4096)

print("\nEARLY-FUSION FEATURE MATRICES")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)

# ------------------------------------------------------------
# 5. Select regularization using validation macro F1
# ------------------------------------------------------------

C_values = [0.001, 0.01, 0.1, 1, 10]

results = []

best_model = None
best_f1 = -np.inf
best_C = None
best_predictions = None

print("\n" + "=" * 65)
print("VALIDATION PERFORMANCE BY REGULARIZATION")
print("=" * 65)

for C in C_values:

    # The scaler is fitted on training data only.
    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced = balanced_accuracy_score(
        y_val,
        predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "C": C,
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "macro_f1": macro_f1
    })

    print(
        f"C={C:<6} | "
        f"Accuracy={accuracy:.4f} | "
        f"Balanced Accuracy={balanced:.4f} | "
        f"Macro F1={macro_f1:.4f}"
    )

    # Preserve the first C in case of a tie.
    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_C = C
        best_model = model
        best_predictions = predictions.copy()

# ------------------------------------------------------------
# 6. Save validation results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_path = (
    output_dir / "fusion_logistic_validation_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

# ------------------------------------------------------------
# 7. Save selected early-fusion model
# ------------------------------------------------------------

model_path = (
    output_dir / "fusion_logistic_best.joblib"
)

joblib.dump(
    best_model,
    model_path
)

# ------------------------------------------------------------
# 8. Save validation predictions
# ------------------------------------------------------------

predictions_df = val_labels.copy()

predictions_df["fusion_prediction"] = (
    best_predictions
)

predictions_df["fusion_correct"] = (
    best_predictions == y_val
)

predictions_path = (
    output_dir
    / "fusion_logistic_validation_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST EARLY-FUSION MODEL")
print("=" * 65)

print("Selected C:", best_C)
print("Validation macro F1:", round(best_f1, 4))

print("\nValidation results:")
display(
    results_df.sort_values(
        "macro_f1",
        ascending=False
    ).reset_index(drop=True)
)

print("\nSaved files:")
print("-", results_path.name)
print("-", model_path.name)
print("-", predictions_path.name)

print("\nIMPORTANT:")
print(
    "Early fusion concatenates 2048 audio and "
    "2048 visual features into 4096 dimensions."
)

print(
    "Standardization was fitted on training data only."
)

print(
    "Regularization was selected using "
    "artist-disjoint validation macro F1."
)

print(
    "The held-out test set was not used "
    "for model selection."
)

print("\nEARLY-FUSION BASELINE COMPLETE")

### Per-Genre Performance and Error Analysis

We compare audio-only, visual-only, and early-fusion classification performance across the six music genres.

For each model and genre, we calculate precision, recall, and F1-score. We then measure the difference in per-genre F1 between early fusion and each unimodal baseline.

To examine how visual information changes individual predictions, we also identify two types of outcomes:

- **Corrected errors:** tracks misclassified by the audio-only model but correctly classified by early fusion.
- **New errors:** tracks correctly classified by the audio-only model but misclassified by early fusion.

This analysis helps identify whether improvements from multimodal fusion are consistent across genres or concentrated in particular categories.

The definitive per-genre comparison is conducted using predictions from our artist-disjoint evaluation dataset.

In [ ]:

# ============================================================
# PHASE 5 — PER-GENRE PERFORMANCE AND ERROR ANALYSIS
# ARTIST-DISJOINT VALIDATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import classification_report

print("=" * 65)
print("PER-GENRE MULTIMODAL PERFORMANCE")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load validation predictions
# ------------------------------------------------------------

audio = pd.read_csv(
    output_dir / "audio_logistic_validation_predictions.csv"
)

visual = pd.read_csv(
    output_dir / "visual_logistic_validation_predictions.csv"
)

fusion = pd.read_csv(
    output_dir / "fusion_logistic_validation_predictions.csv"
)

models = {
    "Audio": (audio, "audio_prediction"),
    "Visual": (visual, "visual_prediction"),
    "Fusion": (fusion, "fusion_prediction")
}

# ------------------------------------------------------------
# 3. Validate prediction alignment
# ------------------------------------------------------------

required_metadata = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for model_name, (frame, prediction_col) in models.items():

    required = required_metadata | {prediction_col}

    assert required.issubset(frame.columns), (
        f"Missing columns in {model_name} predictions."
    )

    assert frame[list(required)].notna().all().all(), (
        f"Missing values in {model_name} predictions."
    )

    assert frame["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {model_name} predictions."
    )

    assert (frame["split"] == "val").all(), (
        f"{model_name} contains non-validation observations."
    )

    assert len(frame) == len(audio), (
        f"Different prediction counts for {model_name}."
    )

    for column in [
        "msd_track_id",
        "msd_artist_id",
        "genre",
        "split"
    ]:
        assert frame[column].equals(audio[column]), (
            f"{model_name} is not aligned on {column}."
        )

genres = sorted(audio["genre"].unique())

for model_name, (frame, prediction_col) in models.items():
    assert set(frame[prediction_col]).issubset(set(genres)), (
        f"Unexpected predicted genre in {model_name}."
    )

y_true = audio["genre"].to_numpy()

print("\nValidated observations:", len(audio))
print("Validation artists:", audio["msd_artist_id"].nunique())
print("Genres:", genres)

# ------------------------------------------------------------
# 4. Calculate per-genre classification metrics
# ------------------------------------------------------------

records = []

for model_name, (frame, prediction_col) in models.items():

    report = classification_report(
        y_true,
        frame[prediction_col].to_numpy(),
        labels=genres,
        output_dict=True,
        zero_division=0
    )

    for genre in genres:

        records.append({
            "model": model_name,
            "genre": genre,
            "precision": report[genre]["precision"],
            "recall": report[genre]["recall"],
            "f1": report[genre]["f1-score"],
            "support": report[genre]["support"]
        })

results = pd.DataFrame(records)

print("\n" + "=" * 65)
print("PER-GENRE PRECISION, RECALL AND F1")
print("=" * 65)

display(
    results.round(4)
)

# ------------------------------------------------------------
# 5. Compare per-genre F1 scores
# ------------------------------------------------------------

comparison = results.pivot(
    index="genre",
    columns="model",
    values="f1"
)

comparison["Fusion - Audio"] = (
    comparison["Fusion"] - comparison["Audio"]
)

comparison["Fusion - Visual"] = (
    comparison["Fusion"] - comparison["Visual"]
)

comparison = comparison.sort_values(
    "Fusion - Audio",
    ascending=False
)

print("\n" + "=" * 65)
print("PER-GENRE F1 COMPARISON")
print("=" * 65)

display(comparison.round(4))

# ------------------------------------------------------------
# 6. Analyze corrected and newly introduced errors
# ------------------------------------------------------------

audio_correct = (
    audio["audio_prediction"].to_numpy() == y_true
)

visual_correct = (
    visual["visual_prediction"].to_numpy() == y_true
)

fusion_correct = (
    fusion["fusion_prediction"].to_numpy() == y_true
)

# Compare early fusion against audio-only.
corrected_audio = (
    ~audio_correct & fusion_correct
)

new_errors_audio = (
    audio_correct & ~fusion_correct
)

# Compare early fusion against visual-only.
corrected_visual = (
    ~visual_correct & fusion_correct
)

new_errors_visual = (
    visual_correct & ~fusion_correct
)

print("\n" + "=" * 65)
print("OVERALL ERROR ANALYSIS")
print("=" * 65)

print("\nFusion vs Audio:")
print("Audio errors corrected:", int(corrected_audio.sum()))
print("New errors introduced:", int(new_errors_audio.sum()))
print(
    "Net additional correct predictions:",
    int(corrected_audio.sum() - new_errors_audio.sum())
)

print("\nFusion vs Visual:")
print("Visual errors corrected:", int(corrected_visual.sum()))
print("New errors introduced:", int(new_errors_visual.sum()))
print(
    "Net additional correct predictions:",
    int(corrected_visual.sum() - new_errors_visual.sum())
)

# ------------------------------------------------------------
# 7. Per-genre error analysis
# ------------------------------------------------------------

error_records = []

for genre in genres:

    mask = y_true == genre

    error_records.append({
        "genre": genre,
        "support": int(mask.sum()),
        "audio_errors_corrected": int(
            (corrected_audio & mask).sum()
        ),
        "new_errors_vs_audio": int(
            (new_errors_audio & mask).sum()
        ),
        "net_correct_vs_audio": int(
            (corrected_audio & mask).sum()
            - (new_errors_audio & mask).sum()
        ),
        "visual_errors_corrected": int(
            (corrected_visual & mask).sum()
        ),
        "new_errors_vs_visual": int(
            (new_errors_visual & mask).sum()
        ),
        "net_correct_vs_visual": int(
            (corrected_visual & mask).sum()
            - (new_errors_visual & mask).sum()
        )
    })

error_analysis = pd.DataFrame(error_records)

error_analysis = error_analysis.sort_values(
    "net_correct_vs_audio",
    ascending=False
)

print("\n" + "=" * 65)
print("PER-GENRE ERROR ANALYSIS")
print("=" * 65)

display(error_analysis)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

metrics_path = (
    output_dir / "per_genre_validation_metrics.csv"
)

comparison_path = (
    output_dir / "per_genre_fusion_comparison.csv"
)

errors_path = (
    output_dir / "per_genre_error_analysis.csv"
)

results.to_csv(
    metrics_path,
    index=False
)

comparison.to_csv(
    comparison_path
)

error_analysis.to_csv(
    errors_path,
    index=False
)

print("\nSaved files:")
print("-", metrics_path.name)
print("-", comparison_path.name)
print("-", errors_path.name)

print("\nIMPORTANT:")
print(
    "All comparisons use the same artist-disjoint "
    "validation observations."
)

print(
    "Per-genre F1 differences and changes in the "
    "number of correct predictions measure "
    "different aspects of performance."
)

print("\nPER-GENRE ANALYSIS COMPLETE")

### Multimodal Performance by Visual Typicality

We investigate whether the contribution of album-cover information depends on how visually representative an image is of its music genre.

First, we standardize the visual embeddings using statistics estimated exclusively from the training data. We then calculate a centroid for each genre by averaging the standardized visual embeddings of its training observations.

For each evaluation observation, visual typicality is measured as the cosine distance between its visual embedding and the centroid of its true genre. Smaller distances indicate more visually typical album covers, while larger distances indicate more atypical covers.

To ensure that typicality is assessed relative to each genre, observations are divided into five genre-conditional quintiles, ranging from Q1 (most typical) to Q5 (most atypical).

We compare audio-only, visual-only, and early-fusion accuracy across these groups, focusing on whether the improvement from multimodal fusion changes as visual typicality decreases.

Because the true genre is used to calculate typicality, this measure serves as a retrospective analysis tool rather than a prediction-time input.

In [ ]:


import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import paired_cosine_distances

print("=" * 65)
print("MULTIMODAL PERFORMANCE BY VISUAL TYPICALITY")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint feature directory not found."
)

# ------------------------------------------------------------
# 2. Load visual embeddings and metadata
# ------------------------------------------------------------

X_train = np.load(
    output_dir / "X_train_visual.npy",
    allow_pickle=False
)

X_val = np.load(
    output_dir / "X_val_visual.npy",
    allow_pickle=False
)

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

required_metadata = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for split, labels in [
    ("train", train_labels),
    ("val", val_labels)
]:

    assert required_metadata.issubset(labels.columns), (
        f"Missing metadata columns in {split}."
    )

    assert labels[list(required_metadata)].notna().all().all(), (
        f"Missing metadata values in {split}."
    )

    assert labels["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {split}."
    )

    assert (labels["split"] == split).all(), (
        f"Incorrect split assignments in {split}."
    )

assert X_train.shape == (len(train_labels), 2048)
assert X_val.shape == (len(val_labels), 2048)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

train_artists = set(train_labels["msd_artist_id"])
val_artists = set(val_labels["msd_artist_id"])

assert train_artists.isdisjoint(val_artists), (
    "Artist leakage between training and validation."
)

print("\nTraining visual embeddings:", X_train.shape)
print("Validation visual embeddings:", X_val.shape)
print("Artist overlap:", len(train_artists & val_artists))

# ------------------------------------------------------------
# 3. Standardize visual embeddings
# ------------------------------------------------------------

# Fit exclusively on training observations.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()

# ------------------------------------------------------------
# 4. Compute training-set genre centroids
# ------------------------------------------------------------

genres = sorted(train_labels["genre"].unique())

train_genres = train_labels["genre"].to_numpy()
val_genres = val_labels["genre"].to_numpy()

assert set(val_genres).issubset(set(genres)), (
    "Validation contains a genre absent from training."
)

centroids = {}

for genre in genres:

    mask = train_genres == genre

    assert mask.sum() > 0, (
        f"No training observations for genre {genre}."
    )

    centroids[genre] = (
        X_train_scaled[mask].mean(axis=0)
    )

print("\nTraining genre centroids:")
for genre in genres:
    print(
        f"  {genre}: "
        f"{int((train_genres == genre).sum())} tracks"
    )

# ------------------------------------------------------------
# 5. Calculate distance to TRUE-GENRE centroid
# ------------------------------------------------------------

# This is a retrospective diagnostic:
# true validation labels are used only to define typicality.

matched_centroids = np.stack([
    centroids[genre]
    for genre in val_genres
])

distances = paired_cosine_distances(
    X_val_scaled,
    matched_centroids
)

assert np.isfinite(distances).all()

details = val_labels.copy()
details["visual_distance"] = distances

# ------------------------------------------------------------
# 6. Assign genre-conditional typicality quintiles
# ------------------------------------------------------------

# Rank within each genre before applying qcut.
# This gives approximately equal-sized groups, even if
# several observations have identical cosine distances.

details["typicality_quintile"] = (
    details.groupby("genre")["visual_distance"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            q=5,
            labels=False
        ) + 1
    )
    .astype(int)
)

# Check sufficient observations for quintile assignment.
genre_counts = details["genre"].value_counts()

assert (genre_counts >= 5).all(), (
    "At least one genre has fewer than five "
    "validation observations."
)

assert set(details["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

print("\nTypicality quintile counts:")
display(
    pd.crosstab(
        details["genre"],
        details["typicality_quintile"]
    )
)

# ------------------------------------------------------------
# 7. Load model validation predictions
# ------------------------------------------------------------

audio = pd.read_csv(
    output_dir / "audio_logistic_validation_predictions.csv"
)

visual = pd.read_csv(
    output_dir / "visual_logistic_validation_predictions.csv"
)

fusion = pd.read_csv(
    output_dir / "fusion_logistic_validation_predictions.csv"
)

prediction_sources = {
    "audio": (audio, "audio_prediction"),
    "visual": (visual, "visual_prediction"),
    "fusion": (fusion, "fusion_prediction")
}

# ------------------------------------------------------------
# 8. Verify exact track and label alignment
# ------------------------------------------------------------

for model_name, (frame, prediction_col) in (
    prediction_sources.items()
):

    required = required_metadata | {prediction_col}

    assert required.issubset(frame.columns), (
        f"Missing columns in {model_name} predictions."
    )

    assert frame[list(required)].notna().all().all(), (
        f"Missing values in {model_name} predictions."
    )

    assert frame["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {model_name} predictions."
    )

    assert len(frame) == len(details), (
        f"Prediction count mismatch for {model_name}."
    )

    for column in [
        "msd_track_id",
        "msd_artist_id",
        "genre",
        "split"
    ]:
        assert frame[column].equals(details[column]), (
            f"Misaligned {model_name} predictions: {column}."
        )

    assert set(frame[prediction_col]).issubset(
        set(genres)
    ), f"Unexpected predicted genre in {model_name}."

# ------------------------------------------------------------
# 9. Attach predictions and correctness indicators
# ------------------------------------------------------------

for model_name, (frame, prediction_col) in (
    prediction_sources.items()
):

    details[f"{model_name}_prediction"] = (
        frame[prediction_col].to_numpy()
    )

    details[f"{model_name}_correct"] = (
        details[f"{model_name}_prediction"].to_numpy()
        == val_genres
    )

# ------------------------------------------------------------
# 10. Summarize accuracy by visual typicality
# ------------------------------------------------------------

summary = (
    details.groupby("typicality_quintile")
    .agg(
        n=("genre", "size"),
        mean_visual_distance=(
            "visual_distance",
            "mean"
        ),
        visual_accuracy=("visual_correct", "mean"),
        audio_accuracy=("audio_correct", "mean"),
        fusion_accuracy=("fusion_correct", "mean")
    )
    .sort_index()
)

summary["fusion_minus_audio"] = (
    summary["fusion_accuracy"]
    - summary["audio_accuracy"]
)

summary["fusion_minus_visual"] = (
    summary["fusion_accuracy"]
    - summary["visual_accuracy"]
)

summary["audio_minus_visual"] = (
    summary["audio_accuracy"]
    - summary["visual_accuracy"]
)

print("\n" + "=" * 65)
print("ACCURACY BY VISUAL TYPICALITY")
print("=" * 65)

display(summary.round(4))

# ------------------------------------------------------------
# 11. Calculate Q1–Q5 accuracy gaps
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("Q1 - Q5 ACCURACY GAPS")
print("=" * 65)

for model in ["audio", "visual", "fusion"]:

    gap = (
        summary.loc[1, f"{model}_accuracy"]
        - summary.loc[5, f"{model}_accuracy"]
    )

    print(
        f"{model.capitalize():7s}: {gap:+.4f}"
    )

# Compare fusion benefit in Q1 and Q5.
q1_gain = summary.loc[1, "fusion_minus_audio"]
q5_gain = summary.loc[5, "fusion_minus_audio"]

print("\nFUSION BENEFIT RELATIVE TO AUDIO")

print(f"Q1 (most typical):  {q1_gain:+.4f}")
print(f"Q5 (most atypical): {q5_gain:+.4f}")

print(
    "Q1 - Q5 difference in fusion gain:",
    f"{q1_gain - q5_gain:+.4f}"
)

# ------------------------------------------------------------
# 12. Save results
# ------------------------------------------------------------

details_path = (
    output_dir / "multimodal_typicality_details.csv"
)

summary_path = (
    output_dir / "multimodal_typicality_summary.csv"
)

details.to_csv(
    details_path,
    index=False
)

summary.to_csv(
    summary_path
)

print("\nSaved files:")
print("-", details_path.name)
print("-", summary_path.name)

print("\nIMPORTANT:")
print(
    "Centroids and standardization were fitted "
    "using training data only."
)

print(
    "Typicality quintiles were assigned "
    "within each true genre."
)

print(
    "This is a retrospective validation analysis, "
    "not a prediction-time feature."
)

print(
    "The held-out artist-disjoint test set "
    "remains untouched."
)

print("\nVISUAL TYPICALITY ANALYSIS COMPLETE")


### Fixed Late Fusion: Weighted Probability Averaging

Late fusion combines the predictions of separately trained audio-only and visual-only classifiers rather than concatenating their feature representations.

For each observation, the two models produce class-probability vectors. We calculate a weighted average:

\[
P_{\mathrm{fusion}}(y \mid x)
=
\alpha P_{\mathrm{audio}}(y \mid x)
+
(1-\alpha)P_{\mathrm{visual}}(y \mid x),
\]

where \(\alpha\) controls the contribution of audio information.

We evaluate candidate audio weights between 0 and 1 using validation macro F1. The selected weight is then held fixed during final test evaluation.

This approach allows us to compare feature-level integration (early fusion) with prediction-level integration (late fusion), helping determine which strategy makes more effective use of the two modalities.

In [ ]:

# ============================================================
# PHASE 5 — FIXED LATE FUSION
# WEIGHTED PROBABILITY AVERAGING
# ARTIST-DISJOINT VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("FIXED LATE FUSION — WEIGHTED PROBABILITIES")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load selected unimodal models
# ------------------------------------------------------------

audio_model = joblib.load(
    output_dir / "audio_logistic_best.joblib"
)

visual_model = joblib.load(
    output_dir / "visual_logistic_best.joblib"
)

# ------------------------------------------------------------
# 3. Load artist-disjoint validation data
# ------------------------------------------------------------

X_audio = np.load(
    output_dir / "X_val_audio.npy",
    allow_pickle=False
)

X_visual = np.load(
    output_dir / "X_val_visual.npy",
    allow_pickle=False
)

labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

assert required_columns.issubset(labels.columns)
assert labels[list(required_columns)].notna().all().all()
assert labels["msd_track_id"].is_unique
assert (labels["split"] == "val").all()

y_val = labels["genre"].to_numpy()

assert X_audio.shape == (len(labels), 2048), (
    f"Unexpected audio shape: {X_audio.shape}"
)

assert X_visual.shape == (len(labels), 2048), (
    f"Unexpected visual shape: {X_visual.shape}"
)

assert np.isfinite(X_audio).all()
assert np.isfinite(X_visual).all()

print("\nVALIDATION DATA")
print("Audio features:", X_audio.shape)
print("Visual features:", X_visual.shape)
print("Validation artists:", labels["msd_artist_id"].nunique())

# ------------------------------------------------------------
# 4. Obtain class probabilities
# ------------------------------------------------------------

audio_classes = audio_model.classes_
visual_classes = visual_model.classes_

assert np.array_equal(
    audio_classes,
    visual_classes
), (
    "Audio and visual models have different "
    "class ordering."
)

classes = audio_classes

assert set(y_val).issubset(set(classes)), (
    "Validation contains genres absent from the models."
)

p_audio = audio_model.predict_proba(X_audio)
p_visual = visual_model.predict_proba(X_visual)

assert p_audio.shape == (
    len(labels), len(classes)
)

assert p_visual.shape == (
    len(labels), len(classes)
)

assert np.isfinite(p_audio).all()
assert np.isfinite(p_visual).all()

assert np.allclose(
    p_audio.sum(axis=1), 1.0
)

assert np.allclose(
    p_visual.sum(axis=1), 1.0
)

print("\nProbability matrices:")
print("Audio:", p_audio.shape)
print("Visual:", p_visual.shape)
print("Class order:", list(classes))

# ------------------------------------------------------------
# 5. Search over fixed audio weights
# ------------------------------------------------------------

# Alpha = 0: visual-only predictions
# Alpha = 1: audio-only predictions

weights = np.linspace(0.0, 1.0, 21)

records = []

best_f1 = -np.inf
best_weight = None
best_predictions = None

print("\n" + "=" * 65)
print("VALIDATION PERFORMANCE BY AUDIO WEIGHT")
print("=" * 65)

for alpha in weights:

    probabilities = (
        alpha * p_audio
        + (1.0 - alpha) * p_visual
    )

    predictions = classes[
        np.argmax(probabilities, axis=1)
    ]

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced = balanced_accuracy_score(
        y_val,
        predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    records.append({
        "audio_weight": float(alpha),
        "visual_weight": float(1.0 - alpha),
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "macro_f1": macro_f1
    })

    print(
        f"Audio weight={alpha:.2f} | "
        f"Visual weight={1.0-alpha:.2f} | "
        f"Accuracy={accuracy:.4f} | "
        f"Macro F1={macro_f1:.4f}"
    )

    # Select the highest validation macro F1.
    # In a tie, retain the first (smallest) audio weight.
    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_weight = float(alpha)
        best_predictions = predictions.copy()

# ------------------------------------------------------------
# 6. Save complete weight-search results
# ------------------------------------------------------------

results = pd.DataFrame(records)

search_path = (
    output_dir / "late_fusion_weight_search.csv"
)

results.to_csv(
    search_path,
    index=False
)

# ------------------------------------------------------------
# 7. Save selected validation predictions
# ------------------------------------------------------------

prediction_df = labels.copy()

prediction_df["late_fusion_prediction"] = (
    best_predictions
)

prediction_df["late_fusion_correct"] = (
    best_predictions == y_val
)

predictions_path = (
    output_dir
    / "late_fusion_validation_predictions.csv"
)

prediction_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 8. Save selected fusion configuration
# ------------------------------------------------------------

config = pd.DataFrame([{
    "best_audio_weight": best_weight,
    "best_visual_weight": 1.0 - best_weight,
    "validation_macro_f1": best_f1
}])

config_path = (
    output_dir / "late_fusion_best_config.csv"
)

config.to_csv(
    config_path,
    index=False
)

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST FIXED LATE-FUSION CONFIGURATION")
print("=" * 65)

print("Audio weight:", round(best_weight, 2))
print("Visual weight:", round(1.0 - best_weight, 2))
print("Validation macro F1:", round(best_f1, 4))

print("\nTop validation configurations:")
display(
    results.sort_values(
        "macro_f1",
        ascending=False,
        kind="stable"
    ).head(10).reset_index(drop=True).round(4)
)

print("\nSaved files:")
print("-", search_path.name)
print("-", predictions_path.name)
print("-", config_path.name)

print("\nIMPORTANT:")
print(
    "Both unimodal classifiers were selected "
    "using artist-disjoint validation data."
)

print(
    "The fusion weight was also selected "
    "using validation macro F1."
)

print(
    "The selected weight must remain fixed "
    "during held-out test evaluation."
)

print(
    "The test set was not used "
    "to select the fusion weight."
)

print("\nFIXED LATE-FUSION VALIDATION COMPLETE")


### Early vs Late Fusion Across Visual Typicality

We compare early fusion and fixed late fusion to investigate whether different multimodal integration strategies behave differently as album covers become less visually representative of their genres.

Using the previously calculated genre-conditional visual-typicality quintiles, we evaluate the accuracy of audio-only, visual-only, early-fusion, and late-fusion classifiers.

For each quintile, we calculate the accuracy difference between late fusion and the audio-only baseline, as well as the difference between late and early fusion.

We pay particular attention to Q5, which contains the most visually atypical album covers. Within this group, we count how many audio-only classification errors are corrected by late fusion and how many new errors are introduced.

This analysis helps determine whether combining predictions rather than features can reduce the negative effects of visually atypical album covers.

The definitive comparison uses artist-disjoint evaluation data.

In [ ]:

# ============================================================
# PHASE 5 — EARLY VS FIXED LATE FUSION
# PERFORMANCE BY VISUAL TYPICALITY
# ARTIST-DISJOINT VALIDATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("EARLY VS LATE FUSION BY VISUAL TYPICALITY")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT is defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load visual typicality analysis
# ------------------------------------------------------------

details = pd.read_csv(
    output_dir / "multimodal_typicality_details.csv"
)

required_details = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split",
    "visual_distance",
    "typicality_quintile",
    "audio_correct",
    "visual_correct",
    "fusion_correct"
}

assert required_details.issubset(details.columns), (
    "Missing columns in visual typicality results."
)

assert details[list(required_details)].notna().all().all()
assert details["msd_track_id"].is_unique
assert (details["split"] == "val").all()

assert set(details["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

# ------------------------------------------------------------
# 3. Load fixed late-fusion predictions
# ------------------------------------------------------------

late = pd.read_csv(
    output_dir / "late_fusion_validation_predictions.csv"
)

required_late = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split",
    "late_fusion_prediction"
}

assert required_late.issubset(late.columns)
assert late[list(required_late)].notna().all().all()

assert late["msd_track_id"].is_unique
assert (late["split"] == "val").all()

# ------------------------------------------------------------
# 4. Verify exact observation alignment
# ------------------------------------------------------------

assert len(details) == len(late), (
    "Different numbers of validation observations."
)

for column in [
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
]:
    assert details[column].equals(late[column]), (
        f"Misalignment between datasets: {column}"
    )

print("\nValidated observations:", len(details))
print(
    "Validation artists:",
    details["msd_artist_id"].nunique()
)

# ------------------------------------------------------------
# 5. Add late-fusion predictions and correctness
# ------------------------------------------------------------

details["late_fusion_prediction"] = (
    late["late_fusion_prediction"].to_numpy()
)

details["late_fusion_correct"] = (
    details["late_fusion_prediction"].to_numpy()
    == details["genre"].to_numpy()
)

# Explicitly convert saved correctness columns to Boolean.
# These columns were originally created as Boolean values.

for column in [
    "audio_correct",
    "visual_correct",
    "fusion_correct"
]:
    assert details[column].isin([True, False]).all(), (
        f"Invalid correctness values in {column}."
    )

    details[column] = details[column].astype(bool)

# ------------------------------------------------------------
# 6. Summarize accuracy by visual typicality
# ------------------------------------------------------------

summary = (
    details.groupby("typicality_quintile")
    .agg(
        n=("genre", "size"),
        mean_visual_distance=(
            "visual_distance",
            "mean"
        ),
        audio=("audio_correct", "mean"),
        visual=("visual_correct", "mean"),
        early_fusion=("fusion_correct", "mean"),
        late_fusion=("late_fusion_correct", "mean")
    )
    .sort_index()
)

# Improvement relative to audio-only.
summary["early_minus_audio"] = (
    summary["early_fusion"] - summary["audio"]
)

summary["late_minus_audio"] = (
    summary["late_fusion"] - summary["audio"]
)

# Direct comparison between fusion strategies.
summary["late_minus_early"] = (
    summary["late_fusion"] - summary["early_fusion"]
)

print("\n" + "=" * 65)
print("ACCURACY BY VISUAL TYPICALITY")
print("=" * 65)

display(summary.round(4))

# ------------------------------------------------------------
# 7. Calculate Q1–Q5 accuracy gaps
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("Q1 - Q5 ACCURACY GAPS")
print("=" * 65)

for model in [
    "audio",
    "visual",
    "early_fusion",
    "late_fusion"
]:

    gap = (
        summary.loc[1, model]
        - summary.loc[5, model]
    )

    print(f"{model:15s}: {gap:+.4f}")

# ------------------------------------------------------------
# 8. Compare fusion gains in Q1 and Q5
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FUSION BENEFIT RELATIVE TO AUDIO")
print("=" * 65)

for quintile in [1, 5]:

    early_gain = summary.loc[
        quintile, "early_minus_audio"
    ]

    late_gain = summary.loc[
        quintile, "late_minus_audio"
    ]

    print(f"\nQ{quintile}:")

    print(
        f"  Early fusion - Audio: {early_gain:+.4f}"
    )

    print(
        f"  Late fusion  - Audio: {late_gain:+.4f}"
    )

    print(
        "  Late fusion - Early fusion:",
        f"{summary.loc[quintile, 'late_minus_early']:+.4f}"
    )

# ------------------------------------------------------------
# 9. Analyze Q5 classification errors
# ------------------------------------------------------------

q5 = details.loc[
    details["typicality_quintile"] == 5
].copy()

audio_correct = q5["audio_correct"].to_numpy()
early_correct = q5["fusion_correct"].to_numpy()
late_correct = q5["late_fusion_correct"].to_numpy()

print("\n" + "=" * 65)
print("Q5 — MOST VISUALLY ATYPICAL COVERS")
print("=" * 65)

print("Number of tracks:", len(q5))

print("\nCorrect predictions:")
print("Audio-only:", int(audio_correct.sum()))
print("Early fusion:", int(early_correct.sum()))
print("Late fusion:", int(late_correct.sum()))

# Early fusion compared with audio-only.
early_corrected = (
    ~audio_correct & early_correct
)

early_new_errors = (
    audio_correct & ~early_correct
)

# Late fusion compared with audio-only.
late_corrected = (
    ~audio_correct & late_correct
)

late_new_errors = (
    audio_correct & ~late_correct
)

print("\nEARLY FUSION VS AUDIO")

print(
    "Audio errors corrected:",
    int(early_corrected.sum())
)

print(
    "New errors introduced:",
    int(early_new_errors.sum())
)

print(
    "Net additional correct predictions:",
    int(early_corrected.sum() - early_new_errors.sum())
)

print("\nLATE FUSION VS AUDIO")

print(
    "Audio errors corrected:",
    int(late_corrected.sum())
)

print(
    "New errors introduced:",
    int(late_new_errors.sum())
)

print(
    "Net additional correct predictions:",
    int(late_corrected.sum() - late_new_errors.sum())
)

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

summary_path = (
    output_dir / "early_vs_late_typicality_summary.csv"
)

details_path = (
    output_dir / "early_vs_late_typicality_details.csv"
)

summary.to_csv(
    summary_path
)

details.to_csv(
    details_path,
    index=False
)

print("\nSaved files:")
print("-", summary_path.name)
print("-", details_path.name)

print("\nIMPORTANT:")
print(
    "All results use artist-disjoint validation data."
)

print(
    "Q1 is the most visually typical quintile; "
    "Q5 is the most visually atypical."
)

print(
    "Visual typicality is retrospective and uses "
    "the true genre."
)

print(
    "The held-out test set remains reserved "
    "for final evaluation."
)

print("\nEARLY VS LATE FUSION ANALYSIS COMPLETE")


### Confidence-Adaptive Late Fusion

We investigate whether dynamically adjusting the contribution of visual information can improve multimodal genre classification.

Unlike fixed late fusion, which assigns the same weights to every observation, confidence-adaptive late fusion determines the visual weight separately for each track.

We define visual confidence as the maximum class probability predicted by the visual-only classifier. The visual weight is then calculated as

\[
w_i=\beta c_i^\gamma,
\]

where \(c_i\) represents visual confidence and \(\beta\) and \(\gamma\) control the weighting function.

The final prediction probabilities are obtained through

\[
P_i^{\mathrm{adaptive}}
=
(1-w_i)P_i^{\mathrm{audio}}
+
w_iP_i^{\mathrm{visual}}.
\]

We evaluate candidate parameter combinations using validation macro F1 and select the best-performing configuration.

We also examine accuracy among the most visually atypical album covers (Q5) to investigate whether adaptive weighting can reduce classification errors associated with visual information.

The selected parameters are held fixed for final evaluation on the artist-disjoint test set.

In [ ]:

# ============================================================
# PHASE 5 — CONFIDENCE-ADAPTIVE LATE FUSION
# ARTIST-DISJOINT VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("CONFIDENCE-ADAPTIVE LATE FUSION")
print("ARTIST-DISJOINT VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load selected unimodal models
# ------------------------------------------------------------

audio_model = joblib.load(
    output_dir / "audio_logistic_best.joblib"
)

visual_model = joblib.load(
    output_dir / "visual_logistic_best.joblib"
)

# ------------------------------------------------------------
# 3. Load validation features and metadata
# ------------------------------------------------------------

X_audio = np.load(
    output_dir / "X_val_audio.npy",
    allow_pickle=False
)

X_visual = np.load(
    output_dir / "X_val_visual.npy",
    allow_pickle=False
)

labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

# Typicality results from Cell 22.
details = pd.read_csv(
    output_dir / "early_vs_late_typicality_details.csv"
)

required_metadata = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

assert required_metadata.issubset(labels.columns)
assert required_metadata.issubset(details.columns)

assert labels[list(required_metadata)].notna().all().all()
assert details[list(required_metadata)].notna().all().all()

assert labels["msd_track_id"].is_unique
assert details["msd_track_id"].is_unique

assert (labels["split"] == "val").all()
assert (details["split"] == "val").all()

assert len(labels) == len(details)

for column in [
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
]:
    assert labels[column].equals(details[column]), (
        f"Validation alignment mismatch: {column}"
    )

assert "typicality_quintile" in details.columns

assert details["typicality_quintile"].notna().all()

assert set(details["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

y_val = labels["genre"].to_numpy()

assert X_audio.shape == (len(labels), 2048)
assert X_visual.shape == (len(labels), 2048)

assert np.isfinite(X_audio).all()
assert np.isfinite(X_visual).all()

print("\nVALIDATION DATA")
print("Audio features:", X_audio.shape)
print("Visual features:", X_visual.shape)
print("Validation tracks:", len(labels))
print("Validation artists:", labels["msd_artist_id"].nunique())

# ------------------------------------------------------------
# 4. Obtain unimodal predicted probabilities
# ------------------------------------------------------------

assert np.array_equal(
    audio_model.classes_,
    visual_model.classes_
), (
    "Audio and visual models use different class orders."
)

classes = audio_model.classes_

assert set(y_val).issubset(set(classes))

p_audio = audio_model.predict_proba(X_audio)
p_visual = visual_model.predict_proba(X_visual)

assert p_audio.shape == (len(labels), len(classes))
assert p_visual.shape == (len(labels), len(classes))

assert np.isfinite(p_audio).all()
assert np.isfinite(p_visual).all()

assert np.allclose(p_audio.sum(axis=1), 1.0)
assert np.allclose(p_visual.sum(axis=1), 1.0)

# ------------------------------------------------------------
# 5. Calculate visual confidence
# ------------------------------------------------------------

# Confidence is the highest predicted class probability
# from the visual-only classifier.
#
# Importantly, this does NOT use the true genre.

visual_confidence = p_visual.max(axis=1)

assert np.all(
    (visual_confidence >= 0.0)
    & (visual_confidence <= 1.0)
)

print("\nVISUAL CONFIDENCE")
print("Mean:", round(float(visual_confidence.mean()), 4))
print("Minimum:", round(float(visual_confidence.min()), 4))
print("Maximum:", round(float(visual_confidence.max()), 4))

# ------------------------------------------------------------
# 6. Search adaptive weighting configurations
# ------------------------------------------------------------

beta_values = [0.25, 0.50, 0.75, 1.00]
gamma_values = [1, 2, 3]

q5_mask = (
    details["typicality_quintile"].to_numpy() == 5
)

assert q5_mask.any(), "No Q5 observations found."

results = []

best_f1 = -np.inf
best_config = None
best_predictions = None
best_weights = None

print("\n" + "=" * 65)
print("ADAPTIVE FUSION PARAMETER SEARCH")
print("=" * 65)

for beta in beta_values:

    for gamma in gamma_values:

        # Track-specific visual weight.
        visual_weight = (
            beta * visual_confidence ** gamma
        )

        assert np.all(
            (visual_weight >= 0.0)
            & (visual_weight <= 1.0)
        )

        # Combine audio and visual probabilities.
        probabilities = (
            (1.0 - visual_weight[:, None]) * p_audio
            + visual_weight[:, None] * p_visual
        )

        predictions = classes[
            np.argmax(probabilities, axis=1)
        ]

        accuracy = accuracy_score(
            y_val,
            predictions
        )

        balanced = balanced_accuracy_score(
            y_val,
            predictions
        )

        macro_f1 = f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        )

        # Q5 is diagnostic only, not the selection metric.
        q5_accuracy = accuracy_score(
            y_val[q5_mask],
            predictions[q5_mask]
        )

        results.append({
            "beta": beta,
            "gamma": gamma,
            "accuracy": accuracy,
            "balanced_accuracy": balanced,
            "macro_f1": macro_f1,
            "q5_accuracy": q5_accuracy,
            "mean_visual_weight": float(
                visual_weight.mean()
            )
        })

        print(
            f"beta={beta:.2f}, "
            f"gamma={gamma} | "
            f"Macro F1={macro_f1:.4f} | "
            f"Q5 accuracy={q5_accuracy:.4f}"
        )

        # Select exclusively by overall validation macro F1.
        # Ties retain the first configuration in grid order.
        if macro_f1 > best_f1:

            best_f1 = macro_f1
            best_config = (beta, gamma)

            best_predictions = predictions.copy()
            best_weights = visual_weight.copy()

# ------------------------------------------------------------
# 7. Save parameter-search results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

grid_path = (
    output_dir / "adaptive_fusion_grid_results.csv"
)

results_df.to_csv(
    grid_path,
    index=False
)

# ------------------------------------------------------------
# 8. Save validation predictions
# ------------------------------------------------------------

prediction_df = details.copy()

prediction_df["visual_confidence"] = (
    visual_confidence
)

prediction_df["adaptive_prediction"] = (
    best_predictions
)

prediction_df["adaptive_correct"] = (
    best_predictions == y_val
)

prediction_df["adaptive_visual_weight"] = (
    best_weights
)

predictions_path = (
    output_dir
    / "adaptive_fusion_validation_predictions.csv"
)

prediction_df.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 9. Save selected adaptive configuration
# ------------------------------------------------------------

config_df = pd.DataFrame([{
    "beta": best_config[0],
    "gamma": best_config[1],
    "validation_macro_f1": best_f1
}])

config_path = (
    output_dir / "adaptive_fusion_best_config.csv"
)

config_df.to_csv(
    config_path,
    index=False
)

# ------------------------------------------------------------
# 10. Compare against existing validation models
# ------------------------------------------------------------

baseline_sources = {
    "Audio-only": (
        "audio_logistic_validation_predictions.csv",
        "audio_prediction"
    ),
    "Visual-only": (
        "visual_logistic_validation_predictions.csv",
        "visual_prediction"
    ),
    "Early fusion": (
        "fusion_logistic_validation_predictions.csv",
        "fusion_prediction"
    ),
    "Fixed late fusion": (
        "late_fusion_validation_predictions.csv",
        "late_fusion_prediction"
    )
}

comparison_records = []

for model_name, (filename, prediction_col) in (
    baseline_sources.items()
):

    frame = pd.read_csv(output_dir / filename)

    assert prediction_col in frame.columns
    assert len(frame) == len(labels)

    for column in [
        "msd_track_id",
        "msd_artist_id",
        "genre",
        "split"
    ]:
        assert frame[column].equals(labels[column]), (
            f"Misaligned baseline: {model_name}, {column}"
        )

    predictions = frame[prediction_col].to_numpy()

    comparison_records.append({
        "model": model_name,
        "validation_accuracy": accuracy_score(
            y_val,
            predictions
        ),
        "validation_macro_f1": f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "q5_accuracy": accuracy_score(
            y_val[q5_mask],
            predictions[q5_mask]
        )
    })

# Add adaptive late fusion.
comparison_records.append({
    "model": "Adaptive late fusion",
    "validation_accuracy": accuracy_score(
        y_val,
        best_predictions
    ),
    "validation_macro_f1": best_f1,
    "q5_accuracy": accuracy_score(
        y_val[q5_mask],
        best_predictions[q5_mask]
    )
})

comparison_df = pd.DataFrame(comparison_records)

comparison_path = (
    output_dir / "adaptive_fusion_validation_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEST CONFIDENCE-ADAPTIVE CONFIGURATION")
print("=" * 65)

print("Best beta:", best_config[0])
print("Best gamma:", best_config[1])
print("Validation macro F1:", round(best_f1, 4))

print("\nTop adaptive configurations:")
display(
    results_df.sort_values(
        "macro_f1",
        ascending=False,
        kind="stable"
    ).head(10).reset_index(drop=True).round(4)
)

print("\nVALIDATION MODEL COMPARISON")
display(
    comparison_df.sort_values(
        "validation_macro_f1",
        ascending=False
    ).reset_index(drop=True).round(4)
)

print("\nSaved files:")
print("-", grid_path.name)
print("-", predictions_path.name)
print("-", config_path.name)
print("-", comparison_path.name)

print("\nIMPORTANT:")
print(
    "Adaptive weights use predicted visual "
    "confidence, not true genre labels."
)

print(
    "Visual typicality and Q5 accuracy are used "
    "only for retrospective diagnostics."
)

print(
    "Beta and gamma were selected exclusively "
    "using validation macro F1."
)

print(
    "The selected configuration must remain "
    "fixed during final test evaluation."
)

print("\nCONFIDENCE-ADAPTIVE FUSION COMPLETE")


### Final Evaluation on the Held-Out Test Set

After selecting model hyperparameters using the validation data, we evaluate the five classification approaches on an independent held-out test set.

The models considered are audio-only logistic regression, visual-only logistic regression, early fusion, fixed late fusion, and confidence-adaptive late fusion.

All preprocessing parameters, classifier coefficients, and fusion weights are estimated or selected without using test observations.

We evaluate each model using three complementary metrics:

- **Accuracy:** the proportion of correctly classified tracks.
- **Balanced accuracy:** the average recall across genres.
- **Macro F1:** the unweighted mean of the per-genre F1-scores, ensuring that minority genres contribute equally to the overall evaluation.

The definitive evaluation uses artist-disjoint training, validation, and test partitions to assess generalization to previously unseen artists.

We save both aggregate performance metrics and track-level predictions for subsequent visual-typicality analysis and artist-cluster bootstrap inference.

In [ ]:

# ============================================================
# PHASE 5 — FINAL HELD-OUT TEST EVALUATION
# ARTIST-DISJOINT TEST SET
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 65)
print("FINAL HELD-OUT TEST EVALUATION")
print("ARTIST-DISJOINT PARTITIONS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint modeling directory not found."
)

# ------------------------------------------------------------
# 2. Load train, validation and test metadata
# ------------------------------------------------------------

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    output_dir / "labels_val.csv"
)

test_labels = pd.read_csv(
    output_dir / "labels_test.csv"
)

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for split_name, frame in [
    ("train", train_labels),
    ("val", val_labels),
    ("test", test_labels)
]:

    assert required_columns.issubset(frame.columns), (
        f"Missing columns in {split_name} metadata."
    )

    assert frame[list(required_columns)].notna().all().all(), (
        f"Missing values in {split_name} metadata."
    )

    assert frame["msd_track_id"].is_unique, (
        f"Duplicate tracks in {split_name}."
    )

    assert (frame["split"] == split_name).all(), (
        f"Incorrect split assignments in {split_name}."
    )

# ------------------------------------------------------------
# 3. Verify artist-disjoint partitions
# ------------------------------------------------------------

train_artists = set(train_labels["msd_artist_id"])
val_artists = set(val_labels["msd_artist_id"])
test_artists = set(test_labels["msd_artist_id"])

assert train_artists.isdisjoint(val_artists), (
    "Artist overlap between train and validation."
)

assert train_artists.isdisjoint(test_artists), (
    "Artist overlap between train and test."
)

assert val_artists.isdisjoint(test_artists), (
    "Artist overlap between validation and test."
)

train_tracks = set(train_labels["msd_track_id"])
val_tracks = set(val_labels["msd_track_id"])
test_tracks = set(test_labels["msd_track_id"])

assert train_tracks.isdisjoint(val_tracks)
assert train_tracks.isdisjoint(test_tracks)
assert val_tracks.isdisjoint(test_tracks)

print("\nDATASET PARTITIONS")
print("-" * 45)

for split_name, frame in [
    ("Train", train_labels),
    ("Validation", val_labels),
    ("Test", test_labels)
]:
    print(
        f"{split_name:12s}: "
        f"{len(frame):5d} tracks | "
        f"{frame['msd_artist_id'].nunique():4d} artists"
    )

print("\nArtist overlap: NONE")
print("Track overlap: NONE")

# ------------------------------------------------------------
# 4. Load untouched test embeddings
# ------------------------------------------------------------

X_audio = np.load(
    output_dir / "X_test_audio.npy",
    allow_pickle=False
)

X_visual = np.load(
    output_dir / "X_test_visual.npy",
    allow_pickle=False
)

y_test = test_labels["genre"].to_numpy()

assert X_audio.shape == (len(test_labels), 2048), (
    f"Unexpected audio shape: {X_audio.shape}"
)

assert X_visual.shape == (len(test_labels), 2048), (
    f"Unexpected visual shape: {X_visual.shape}"
)

assert np.isfinite(X_audio).all()
assert np.isfinite(X_visual).all()

# Feature ordering:
# 2048 audio dimensions followed by 2048 visual dimensions.
# This matches the early-fusion training procedure.

X_fusion = np.concatenate(
    [X_audio, X_visual],
    axis=1
)

assert X_fusion.shape == (len(test_labels), 4096)

print("\nTEST FEATURE MATRICES")
print("Audio:", X_audio.shape)
print("Visual:", X_visual.shape)
print("Early fusion:", X_fusion.shape)

# ------------------------------------------------------------
# 5. Load previously selected models
# ------------------------------------------------------------

audio_model = joblib.load(
    output_dir / "audio_logistic_best.joblib"
)

visual_model = joblib.load(
    output_dir / "visual_logistic_best.joblib"
)

fusion_model = joblib.load(
    output_dir / "fusion_logistic_best.joblib"
)

# ------------------------------------------------------------
# 6. Load validation-selected fusion configurations
# ------------------------------------------------------------

late_config_df = pd.read_csv(
    output_dir / "late_fusion_best_config.csv"
)

adaptive_config_df = pd.read_csv(
    output_dir / "adaptive_fusion_best_config.csv"
)

assert len(late_config_df) == 1
assert len(adaptive_config_df) == 1

late_config = late_config_df.iloc[0]
adaptive_config = adaptive_config_df.iloc[0]

alpha = float(late_config["best_audio_weight"])

beta = float(adaptive_config["beta"])
gamma = float(adaptive_config["gamma"])

assert np.isfinite([alpha, beta, gamma]).all()
assert 0.0 <= alpha <= 1.0
assert 0.0 <= beta <= 1.0
assert gamma > 0.0

print("\nVALIDATION-SELECTED FUSION PARAMETERS")
print("Fixed late fusion audio weight:", alpha)
print("Adaptive beta:", beta)
print("Adaptive gamma:", gamma)

# ------------------------------------------------------------
# 7. Verify model class compatibility
# ------------------------------------------------------------

assert np.array_equal(
    audio_model.classes_,
    visual_model.classes_
), "Audio and visual class orders differ."

assert np.array_equal(
    audio_model.classes_,
    fusion_model.classes_
), "Early-fusion class order differs."

classes = audio_model.classes_

assert set(y_test).issubset(set(classes)), (
    "Test set contains genres absent from training."
)

print("\nModel classes:", list(classes))

# ------------------------------------------------------------
# 8. Obtain test probabilities
# ------------------------------------------------------------

p_audio = audio_model.predict_proba(X_audio)
p_visual = visual_model.predict_proba(X_visual)

assert p_audio.shape == (len(y_test), len(classes))
assert p_visual.shape == (len(y_test), len(classes))

assert np.isfinite(p_audio).all()
assert np.isfinite(p_visual).all()

assert np.allclose(p_audio.sum(axis=1), 1.0)
assert np.allclose(p_visual.sum(axis=1), 1.0)

# ------------------------------------------------------------
# 9. Audio-only and visual-only predictions
# ------------------------------------------------------------

pred_audio = audio_model.predict(X_audio)
pred_visual = visual_model.predict(X_visual)

# ------------------------------------------------------------
# 10. Early-fusion predictions
# ------------------------------------------------------------

pred_early = fusion_model.predict(X_fusion)

# ------------------------------------------------------------
# 11. Fixed late-fusion predictions
# ------------------------------------------------------------

p_late = (
    alpha * p_audio
    + (1.0 - alpha) * p_visual
)

pred_late = classes[
    np.argmax(p_late, axis=1)
]

# ------------------------------------------------------------
# 12. Confidence-adaptive late fusion
# ------------------------------------------------------------

# Prediction-time visual confidence:
# maximum visual-only predicted class probability.
visual_confidence = p_visual.max(axis=1)

# Apply validation-selected parameters without refitting.
visual_weight = (
    beta * visual_confidence ** gamma
)

assert np.all(
    (visual_weight >= 0.0)
    & (visual_weight <= 1.0)
)

p_adaptive = (
    (1.0 - visual_weight[:, None]) * p_audio
    + visual_weight[:, None] * p_visual
)

pred_adaptive = classes[
    np.argmax(p_adaptive, axis=1)
]

# ------------------------------------------------------------
# 13. Evaluate all five models
# ------------------------------------------------------------

predictions = {
    "Visual-only": pred_visual,
    "Audio-only": pred_audio,
    "Early fusion": pred_early,
    "Fixed late fusion": pred_late,
    "Adaptive late fusion": pred_adaptive
}

records = []

for model_name, pred in predictions.items():

    records.append({
        "model": model_name,
        "accuracy": accuracy_score(
            y_test, pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, pred
        ),
        "macro_f1": f1_score(
            y_test,
            pred,
            average="macro",
            zero_division=0
        )
    })

results = pd.DataFrame(records)

results = results.sort_values(
    "macro_f1",
    ascending=False,
    kind="stable"
).reset_index(drop=True)

print("\n" + "=" * 65)
print("FINAL HELD-OUT TEST PERFORMANCE")
print("=" * 65)

display(results.round(4))

# ------------------------------------------------------------
# 14. Save track-level predictions
# ------------------------------------------------------------

prediction_df = test_labels.copy()

# Explicit names ensure consistency in later analysis.
prediction_columns = {
    "visual": pred_visual,
    "audio": pred_audio,
    "early_fusion": pred_early,
    "late_fusion": pred_late,
    "adaptive_fusion": pred_adaptive
}

for model_name, pred in prediction_columns.items():

    prediction_df[f"{model_name}_prediction"] = pred

    prediction_df[f"{model_name}_correct"] = (
        pred == y_test
    )

# Store adaptive confidence and weights for diagnostics.
prediction_df["visual_confidence"] = (
    visual_confidence
)

prediction_df["adaptive_visual_weight"] = (
    visual_weight
)

# ------------------------------------------------------------
# 15. Save final test results
# ------------------------------------------------------------

metrics_path = (
    output_dir / "final_test_metrics.csv"
)

predictions_path = (
    output_dir / "final_test_predictions.csv"
)

results.to_csv(
    metrics_path,
    index=False
)

prediction_df.to_csv(
    predictions_path,
    index=False
)

print("\nSaved files:")
print("-", metrics_path.name)
print("-", predictions_path.name)

# ------------------------------------------------------------
# 16. Final verification
# ------------------------------------------------------------

assert len(prediction_df) == len(test_labels)
assert prediction_df["msd_track_id"].is_unique

print("\nFINAL VERIFICATION")

print(
    "Test tracks evaluated:",
    len(prediction_df)
)

print(
    "Previously unseen test artists:",
    prediction_df["msd_artist_id"].nunique()
)

print(
    "Fixed late-fusion weight:",
    alpha
)

print(
    "Adaptive parameters:",
    (beta, gamma)
)

print("\nIMPORTANT:")
print(
    "All model and fusion parameters were "
    "selected before test evaluation."
)

print(
    "No models were fitted or tuned "
    "using test observations in this cell."
)

print(
    "Test predictions are saved for visual "
    "typicality and bootstrap analyses."
)

print("\nFINAL HELD-OUT TEST EVALUATION COMPLETE")


### Test Performance Across Visual Typicality Quintiles

We investigate how the usefulness of album-cover information changes with visual typicality.

Using visual embeddings standardized with training-set statistics, we calculate the centroid of each genre from its training observations. For each test track, we measure the cosine distance between its visual embedding and the centroid corresponding to its true genre.

We divide test observations into five genre-conditional quintiles, from Q1 (most visually typical) to Q5 (most visually atypical).

For each quintile, we calculate the accuracy of all five classification approaches: visual-only, audio-only, early fusion, fixed late fusion, and confidence-adaptive late fusion.

We focus on the accuracy difference between each fusion model and the audio-only baseline to determine whether adding visual information helps or hurts classification at different levels of visual typicality.

Because the true genre is used to determine visual typicality, this analysis is retrospective and does not use typicality as a prediction-time feature.

The definitive analysis uses the artist-disjoint held-out test set.

In [ ]:

# ============================================================
# PHASE 5 — TEST PERFORMANCE BY VISUAL TYPICALITY
# ARTIST-DISJOINT HELD-OUT TEST SET
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import paired_cosine_distances

print("=" * 65)
print("TEST PERFORMANCE BY VISUAL TYPICALITY")
print("ARTIST-DISJOINT HELD-OUT TEST SET")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load visual embeddings and metadata
# ------------------------------------------------------------

X_train = np.load(
    output_dir / "X_train_visual.npy",
    allow_pickle=False
)

X_test = np.load(
    output_dir / "X_test_visual.npy",
    allow_pickle=False
)

train_labels = pd.read_csv(
    output_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    output_dir / "labels_test.csv"
)

predictions = pd.read_csv(
    output_dir / "final_test_predictions.csv"
)

required_metadata = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

for split_name, frame in [
    ("train", train_labels),
    ("test", test_labels)
]:

    assert required_metadata.issubset(frame.columns), (
        f"Missing metadata columns in {split_name}."
    )

    assert frame[list(required_metadata)].notna().all().all(), (
        f"Missing metadata values in {split_name}."
    )

    assert frame["msd_track_id"].is_unique, (
        f"Duplicate track IDs in {split_name}."
    )

    assert (frame["split"] == split_name).all(), (
        f"Incorrect split assignments in {split_name}."
    )

assert X_train.shape == (len(train_labels), 2048)
assert X_test.shape == (len(test_labels), 2048)

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

# ------------------------------------------------------------
# 3. Verify artist-disjoint evaluation
# ------------------------------------------------------------

train_artists = set(train_labels["msd_artist_id"])
test_artists = set(test_labels["msd_artist_id"])

assert train_artists.isdisjoint(test_artists), (
    "Artist overlap between training and test."
)

assert set(train_labels["msd_track_id"]).isdisjoint(
    set(test_labels["msd_track_id"])
), "Track overlap between training and test."

print("\nDATASET INFORMATION")
print("Training tracks:", len(train_labels))
print("Test tracks:", len(test_labels))
print("Test artists:", test_labels["msd_artist_id"].nunique())
print("Artist overlap:", len(train_artists & test_artists))

# ------------------------------------------------------------
# 4. Verify test prediction alignment
# ------------------------------------------------------------

required_predictions = required_metadata | {
    "visual_prediction",
    "audio_prediction",
    "early_fusion_prediction",
    "late_fusion_prediction",
    "adaptive_fusion_prediction"
}

assert required_predictions.issubset(predictions.columns), (
    "Missing columns in final test predictions."
)

assert predictions[list(required_predictions)].notna().all().all()
assert predictions["msd_track_id"].is_unique

assert len(predictions) == len(test_labels)

for column in [
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
]:
    assert predictions[column].equals(test_labels[column]), (
        f"Test prediction alignment mismatch: {column}"
    )

# ------------------------------------------------------------
# 5. Standardize visual embeddings
# ------------------------------------------------------------

# Fit only on training observations.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_test_scaled).all()

# ------------------------------------------------------------
# 6. Compute training-set genre centroids
# ------------------------------------------------------------

genres = sorted(train_labels["genre"].unique())

train_genres = train_labels["genre"].to_numpy()
test_genres = test_labels["genre"].to_numpy()

assert set(test_genres).issubset(set(genres)), (
    "Test contains a genre absent from training."
)

centroids = {}

for genre in genres:

    mask = train_genres == genre

    assert mask.any(), (
        f"No training observations for {genre}."
    )

    centroids[genre] = (
        X_train_scaled[mask].mean(axis=0)
    )

print("\nTRAINING GENRE CENTROIDS")

for genre in genres:
    print(
        f"{genre:15s}: "
        f"{int((train_genres == genre).sum())} tracks"
    )

# ------------------------------------------------------------
# 7. Calculate true-genre visual distances
# ------------------------------------------------------------

# IMPORTANT:
# The true genre is used only for retrospective
# typicality analysis, never for model prediction.

matched_centroids = np.stack([
    centroids[genre]
    for genre in test_genres
])

distances = paired_cosine_distances(
    X_test_scaled,
    matched_centroids
)

assert np.isfinite(distances).all()

details = test_labels.copy()

details["visual_distance"] = distances

# ------------------------------------------------------------
# 8. Assign genre-conditional typicality quintiles
# ------------------------------------------------------------

genre_counts = details["genre"].value_counts()

assert (genre_counts >= 5).all(), (
    "Every genre needs at least five test tracks."
)

# Rank distances within each genre before assigning
# quintiles, ensuring approximately equal group sizes.
#
# Q1 = most typical
# Q5 = most atypical

details["typicality_quintile"] = (
    details.groupby("genre")["visual_distance"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            q=5,
            labels=False
        ) + 1
    )
    .astype(int)
)

assert set(details["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

print("\nGENRE-CONDITIONAL QUINTILE COUNTS")

display(
    pd.crosstab(
        details["genre"],
        details["typicality_quintile"]
    )
)

# ------------------------------------------------------------
# 9. Add predictions and correctness indicators
# ------------------------------------------------------------

model_columns = {
    "visual": "visual_prediction",
    "audio": "audio_prediction",
    "early_fusion": "early_fusion_prediction",
    "late_fusion": "late_fusion_prediction",
    "adaptive_fusion": "adaptive_fusion_prediction"
}

for model_name, prediction_column in model_columns.items():

    model_predictions = (
        predictions[prediction_column].to_numpy()
    )

    assert set(model_predictions).issubset(set(genres)), (
        f"Unexpected predicted genre in {model_name}."
    )

    details[prediction_column] = model_predictions

    details[f"{model_name}_correct"] = (
        model_predictions == test_genres
    )

# Preserve adaptive model diagnostics when available.
for column in [
    "visual_confidence",
    "adaptive_visual_weight"
]:
    if column in predictions.columns:
        details[column] = predictions[column].to_numpy()

# ------------------------------------------------------------
# 10. Summarize accuracy by visual typicality
# ------------------------------------------------------------

summary = (
    details.groupby("typicality_quintile")
    .agg(
        n=("genre", "size"),
        mean_visual_distance=(
            "visual_distance",
            "mean"
        ),
        visual=("visual_correct", "mean"),
        audio=("audio_correct", "mean"),
        early_fusion=("early_fusion_correct", "mean"),
        late_fusion=("late_fusion_correct", "mean"),
        adaptive_fusion=("adaptive_fusion_correct", "mean")
    )
    .sort_index()
)

# Compare each fusion method against audio-only.
summary["early_minus_audio"] = (
    summary["early_fusion"] - summary["audio"]
)

summary["late_minus_audio"] = (
    summary["late_fusion"] - summary["audio"]
)

summary["adaptive_minus_audio"] = (
    summary["adaptive_fusion"] - summary["audio"]
)

print("\n" + "=" * 65)
print("TEST ACCURACY BY VISUAL TYPICALITY")
print("=" * 65)

display(summary.round(4))

# ------------------------------------------------------------
# 11. Calculate Q1–Q5 accuracy gaps
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("Q1 - Q5 ACCURACY GAPS")
print("=" * 65)

for model in [
    "visual",
    "audio",
    "early_fusion",
    "late_fusion",
    "adaptive_fusion"
]:

    gap = (
        summary.loc[1, model]
        - summary.loc[5, model]
    )

    print(f"{model:17s}: {gap:+.4f}")

# ------------------------------------------------------------
# 12. Compare fusion gains in Q1 and Q5
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FUSION GAINS RELATIVE TO AUDIO")
print("=" * 65)

for model in [
    "early",
    "late",
    "adaptive"
]:

    column = f"{model}_minus_audio"

    q1_gain = summary.loc[1, column]
    q5_gain = summary.loc[5, column]

    contrast = q1_gain - q5_gain

    print(f"\n{model.capitalize()} fusion:")

    print(f"  Q1 gain: {q1_gain:+.4f}")
    print(f"  Q5 gain: {q5_gain:+.4f}")
    print(f"  Q1 - Q5 contrast: {contrast:+.4f}")

# ------------------------------------------------------------
# 13. Focus on Q5: most visually atypical covers
# ------------------------------------------------------------

q5 = details.loc[
    details["typicality_quintile"] == 5
].copy()

print("\n" + "=" * 65)
print("Q5 — MOST VISUALLY ATYPICAL TEST COVERS")
print("=" * 65)

print("Number of tracks:", len(q5))
print("Number of artists:", q5["msd_artist_id"].nunique())

print("\nQ5 ACCURACY")

for model in [
    "visual",
    "audio",
    "early_fusion",
    "late_fusion",
    "adaptive_fusion"
]:

    accuracy = q5[f"{model}_correct"].mean()

    print(f"{model:17s}: {accuracy:.4f}")

# ------------------------------------------------------------
# 14. Q5 corrected errors and new errors
# ------------------------------------------------------------

audio_correct = q5["audio_correct"].to_numpy()

print("\nQ5 ERROR ANALYSIS RELATIVE TO AUDIO")

for model in [
    "early_fusion",
    "late_fusion",
    "adaptive_fusion"
]:

    fusion_correct = (
        q5[f"{model}_correct"].to_numpy()
    )

    corrected = (
        ~audio_correct & fusion_correct
    )

    new_errors = (
        audio_correct & ~fusion_correct
    )

    net_change = (
        int(corrected.sum())
        - int(new_errors.sum())
    )

    print(f"\n{model}:")

    print(
        "  Audio errors corrected:",
        int(corrected.sum())
    )

    print(
        "  New errors introduced:",
        int(new_errors.sum())
    )

    print(
        "  Net additional correct predictions:",
        net_change
    )

# ------------------------------------------------------------
# 15. Save final typicality results
# ------------------------------------------------------------

details_path = (
    output_dir / "final_test_typicality_details.csv"
)

summary_path = (
    output_dir / "final_test_typicality_summary.csv"
)

details.to_csv(
    details_path,
    index=False
)

summary.to_csv(
    summary_path
)

print("\nSaved files:")
print("-", details_path.name)
print("-", summary_path.name)

print("\nIMPORTANT:")
print(
    "Visual standardization and genre centroids "
    "were fitted using training data only."
)

print(
    "Typicality quintiles were assigned "
    "within each true genre."
)

print(
    "Visual typicality is retrospective and "
    "was not used for test predictions."
)

print(
    "The saved track-level results can be used "
    "for artist-cluster bootstrap inference."
)

print("\nFINAL TEST TYPICALITY ANALYSIS COMPLETE")


### Statistical Uncertainty: Artist-Cluster Bootstrap

To quantify uncertainty in the differences between classification models, we use an artist-cluster bootstrap with 2,000 replicates.

Rather than resampling individual tracks independently, we resample artists with replacement and include all test tracks associated with each selected artist. This preserves within-artist dependence and accounts for the possibility that tracks by the same artist share musical or visual characteristics.

For each bootstrap replicate, we recalculate the differences in macro F1 between the fusion models and the audio-only baseline. We also evaluate accuracy differences for visually atypical album covers (Q5) and the contrast between fusion gains in Q1 and Q5.

We construct percentile-based 95% confidence intervals using the 2.5th and 97.5th percentiles of the bootstrap distribution.

These intervals help assess whether the observed performance differences are consistent across artist-level resamples.

The analysis uses predictions from the held-out artist-disjoint test set. All model parameters and fusion weights are fixed before the bootstrap procedure.

In [ ]:

# ============================================================
# PHASE 5 — ARTIST-CLUSTER BOOTSTRAP
# HELD-OUT ARTIST-DISJOINT TEST SET
# 2000 REPLICATES — 95% PERCENTILE CONFIDENCE INTERVALS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score

print("=" * 65)
print("ARTIST-CLUSTER BOOTSTRAP CONFIDENCE INTERVALS")
print("ARTIST-DISJOINT HELD-OUT TEST SET")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load final test predictions and typicality
# ------------------------------------------------------------

pred = pd.read_csv(
    output_dir / "final_test_predictions.csv"
)

typ = pd.read_csv(
    output_dir / "final_test_typicality_details.csv"
)

required_metadata = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
}

model_columns = {
    "visual": "visual_prediction",
    "audio": "audio_prediction",
    "early_fusion": "early_fusion_prediction",
    "late_fusion": "late_fusion_prediction",
    "adaptive_fusion": "adaptive_fusion_prediction"
}

required_predictions = (
    required_metadata | set(model_columns.values())
)

required_typicality = (
    required_metadata | {"typicality_quintile"}
)

assert required_predictions.issubset(pred.columns), (
    "Missing columns in final test predictions."
)

assert required_typicality.issubset(typ.columns), (
    "Missing columns in test typicality results."
)

assert pred[list(required_predictions)].notna().all().all()
assert typ[list(required_typicality)].notna().all().all()

assert pred["msd_track_id"].is_unique
assert typ["msd_track_id"].is_unique

assert len(pred) == len(typ)

assert (pred["split"] == "test").all()
assert (typ["split"] == "test").all()

# ------------------------------------------------------------
# 3. Verify exact observation alignment
# ------------------------------------------------------------

for column in [
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split"
]:
    assert pred[column].equals(typ[column]), (
        f"Alignment mismatch: {column}"
    )

assert set(typ["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

# ------------------------------------------------------------
# 4. Extract arrays
# ------------------------------------------------------------

y = pred["genre"].to_numpy()
artists = pred["msd_artist_id"].to_numpy()

quintiles = (
    typ["typicality_quintile"]
    .to_numpy(dtype=int)
)

predictions = {
    model: pred[column].to_numpy()
    for model, column in model_columns.items()
}

genres = sorted(np.unique(y))

for model_name, values in predictions.items():

    assert set(values).issubset(set(genres)), (
        f"Unexpected predicted genre in {model_name}."
    )

correct = {
    model: values == y
    for model, values in predictions.items()
}

print("\nTEST DATA")
print("Tracks:", len(pred))
print("Unique artists:", pred["msd_artist_id"].nunique())
print("Genres:", genres)

# ------------------------------------------------------------
# 5. Group track indices by artist
# ------------------------------------------------------------

artist_groups = [
    np.asarray(indices, dtype=int)
    for indices in (
        pred.groupby(
            "msd_artist_id",
            sort=False
        ).indices.values()
    )
]

n_artists = len(artist_groups)

assert n_artists > 1

assert sum(
    len(indices) for indices in artist_groups
) == len(pred)

print("\nBOOTSTRAP DESIGN")
print("Resampling unit: artist")
print("Number of artists:", n_artists)
print("Replicates: 2000")
print("Random seed: 42")

# ------------------------------------------------------------
# 6. Define statistics
# ------------------------------------------------------------

statistic_names = [
    "early_minus_audio_macro_f1",
    "late_minus_audio_macro_f1",
    "adaptive_minus_audio_macro_f1",
    "early_minus_late_macro_f1",
    "early_minus_audio_q5_accuracy",
    "late_minus_audio_q5_accuracy",
    "adaptive_minus_audio_q5_accuracy",
    "early_gain_q1_minus_q5"
]

def compute_statistics(indices):

    y_sample = y[indices]
    q_sample = quintiles[indices]

    # Use the same six genre labels for every replicate.
    # This keeps macro F1 comparable across samples.

    f1 = {
        model: f1_score(
            y_sample,
            predictions[model][indices],
            labels=genres,
            average="macro",
            zero_division=0
        )
        for model in model_columns
    }

    stats = {
        "early_minus_audio_macro_f1":
            f1["early_fusion"] - f1["audio"],

        "late_minus_audio_macro_f1":
            f1["late_fusion"] - f1["audio"],

        "adaptive_minus_audio_macro_f1":
            f1["adaptive_fusion"] - f1["audio"],

        "early_minus_late_macro_f1":
            f1["early_fusion"] - f1["late_fusion"]
    }

    q1 = q_sample == 1
    q5 = q_sample == 5

    # These statistics require at least one observation
    # in the relevant typicality quintile.

    if q5.any():

        audio_q5 = correct["audio"][indices][q5].mean()

        for model, prefix in [
            ("early_fusion", "early"),
            ("late_fusion", "late"),
            ("adaptive_fusion", "adaptive")
        ]:

            fusion_q5 = (
                correct[model][indices][q5].mean()
            )

            stats[
                f"{prefix}_minus_audio_q5_accuracy"
            ] = fusion_q5 - audio_q5

    else:

        for prefix in ["early", "late", "adaptive"]:

            stats[
                f"{prefix}_minus_audio_q5_accuracy"
            ] = np.nan

    # Difference between early-fusion gains in Q1 and Q5.

    if q1.any() and q5.any():

        early_correct = correct["early_fusion"][indices]
        audio_correct = correct["audio"][indices]

        q1_gain = (
            early_correct[q1].mean()
            - audio_correct[q1].mean()
        )

        q5_gain = (
            early_correct[q5].mean()
            - audio_correct[q5].mean()
        )

        stats["early_gain_q1_minus_q5"] = (
            q1_gain - q5_gain
        )

    else:

        stats["early_gain_q1_minus_q5"] = np.nan

    return stats

# ------------------------------------------------------------
# 7. Calculate observed test differences
# ------------------------------------------------------------

observed = compute_statistics(
    np.arange(len(pred))
)

assert all(
    np.isfinite(value)
    for value in observed.values()
), "Observed statistics contain missing values."

print("\n" + "=" * 65)
print("OBSERVED TEST PERFORMANCE DIFFERENCES")
print("=" * 65)

for metric in statistic_names:

    print(
        f"{metric}: {observed[metric]:+.4f}"
    )

# ------------------------------------------------------------
# 8. Bootstrap artists with replacement
# ------------------------------------------------------------

rng = np.random.default_rng(42)

B = 2000

bootstrap_results = []

for b in range(B):

    # Resample artists, not tracks.
    selected_artists = rng.integers(
        low=0,
        high=n_artists,
        size=n_artists
    )

    # Include every track belonging to each sampled artist.
    # Artists selected multiple times contribute their
    # entire group multiple times.

    sampled_indices = np.concatenate([
        artist_groups[i]
        for i in selected_artists
    ])

    bootstrap_results.append(
        compute_statistics(sampled_indices)
    )

    if (b + 1) % 500 == 0:

        print(
            f"Completed {b + 1}/{B} bootstrap replicates."
        )

bootstrap_df = pd.DataFrame(bootstrap_results)

assert len(bootstrap_df) == B

# ------------------------------------------------------------
# 9. Calculate percentile confidence intervals
# ------------------------------------------------------------

records = []

for metric in statistic_names:

    values = (
        bootstrap_df[metric]
        .dropna()
        .to_numpy()
    )

    n_valid = len(values)

    assert n_valid > 0, (
        f"No valid bootstrap replicates for {metric}."
    )

    # Do not silently accept missing bootstrap statistics.
    if n_valid < B:

        print(
            f"WARNING: {metric} has "
            f"{B - n_valid} invalid replicates."
        )

    lower, upper = np.percentile(
        values,
        [2.5, 97.5]
    )

    records.append({
        "comparison": metric,
        "observed_difference": observed[metric],
        "ci_lower": lower,
        "ci_upper": upper,
        "n_valid_bootstrap": n_valid,
        "n_bootstrap": B
    })

results = pd.DataFrame(records)

# ------------------------------------------------------------
# 10. Display confidence intervals
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("95% ARTIST-CLUSTER BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 65)

display(results.round(4))

# ------------------------------------------------------------
# 11. Save bootstrap results
# ------------------------------------------------------------

ci_path = (
    output_dir / "artist_bootstrap_confidence_intervals.csv"
)

replicates_path = (
    output_dir / "artist_bootstrap_replicates.csv"
)

results.to_csv(
    ci_path,
    index=False
)

bootstrap_df.to_csv(
    replicates_path,
    index=False
)

print("\nSaved files:")
print("-", ci_path.name)
print("-", replicates_path.name)

# ------------------------------------------------------------
# 12. Final methodological checks
# ------------------------------------------------------------

print("\nIMPORTANT:")
print(
    "Artists, rather than individual tracks, "
    "were resampled with replacement."
)

print(
    "All tracks from each sampled artist "
    "were included together."
)

print(
    "Model predictions and fusion parameters "
    "remained fixed throughout the bootstrap."
)

print(
    "The intervals are percentile-based "
    "95% bootstrap confidence intervals."
)

print(
    "Typicality quintiles were held fixed "
    "from the original test analysis."
)

print("\nARTIST-CLUSTER BOOTSTRAP COMPLETE")


### Genre-Specific Effects of Visual Atypicality

We extend the visual-typicality analysis by examining whether the effects of multimodal fusion differ across music genres.

First, we calculate the accuracy of all five classifiers separately for each genre within Q5, the group containing the most visually atypical album covers.

We then compare early-fusion and audio-only accuracy across all five typicality quintiles within each genre.

This analysis helps investigate whether the disadvantages associated with visually atypical album covers occur across multiple genres or are concentrated in particular categories.

Because genre-specific quintile groups contain relatively small numbers of observations, these results are interpreted as exploratory rather than definitive evidence of genre-level differences.

In [ ]:

# ============================================================
# PHASE 5 — GENRE-SPECIFIC EFFECTS
# OF VISUAL ATYPICALITY
# ARTIST-DISJOINT HELD-OUT TEST SET
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("GENRE-SPECIFIC EFFECTS OF VISUAL ATYPICALITY")
print("ARTIST-DISJOINT HELD-OUT TEST SET")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load final test typicality details
# ------------------------------------------------------------

df = pd.read_csv(
    output_dir / "final_test_typicality_details.csv"
)

models = {
    "Visual": "visual_correct",
    "Audio": "audio_correct",
    "Early fusion": "early_fusion_correct",
    "Late fusion": "late_fusion_correct",
    "Adaptive fusion": "adaptive_fusion_correct"
}

required_columns = {
    "msd_track_id",
    "msd_artist_id",
    "genre",
    "split",
    "visual_distance",
    "typicality_quintile"
} | set(models.values())

assert required_columns.issubset(df.columns), (
    "Missing columns in final test typicality details."
)

assert df[list(required_columns)].notna().all().all(), (
    "Missing values in test typicality details."
)

assert df["msd_track_id"].is_unique
assert (df["split"] == "test").all()

assert set(df["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

# Check that saved correctness indicators are Boolean.
for column in models.values():

    assert df[column].isin([True, False]).all(), (
        f"Invalid correctness values in {column}."
    )

    df[column] = df[column].astype(bool)

print("\nDATASET INFORMATION")
print("Test tracks:", len(df))
print("Test artists:", df["msd_artist_id"].nunique())
print("Genres:", sorted(df["genre"].unique()))

# ------------------------------------------------------------
# 3. Inspect genre-conditional quintile sizes
# ------------------------------------------------------------

group_sizes = pd.crosstab(
    df["genre"],
    df["typicality_quintile"]
)

print("\n" + "=" * 65)
print("NUMBER OF TEST TRACKS BY GENRE AND QUINTILE")
print("=" * 65)

display(group_sizes)

# ------------------------------------------------------------
# 4. Calculate Q5 accuracy by genre
# ------------------------------------------------------------

q5 = df.loc[
    df["typicality_quintile"] == 5
].copy()

assert not q5.empty

records = []

for genre, group in q5.groupby("genre"):

    row = {
        "genre": genre,
        "n_q5": len(group),
        "n_artists_q5": group["msd_artist_id"].nunique()
    }

    for model_name, column in models.items():

        row[model_name] = group[column].mean()

    # Fusion gains relative to audio-only.
    row["early_minus_audio"] = (
        row["Early fusion"] - row["Audio"]
    )

    row["late_minus_audio"] = (
        row["Late fusion"] - row["Audio"]
    )

    row["adaptive_minus_audio"] = (
        row["Adaptive fusion"] - row["Audio"]
    )

    records.append(row)

q5_results = pd.DataFrame(records)

q5_results = q5_results.sort_values(
    "early_minus_audio",
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 65)
print("Q5 ACCURACY BY GENRE")
print("=" * 65)

display(q5_results.round(4))

# ------------------------------------------------------------
# 5. Compare models across genre-specific quintiles
# ------------------------------------------------------------

comparison = (
    df.groupby(
        ["genre", "typicality_quintile"]
    )
    .agg(
        n=("genre", "size"),
        n_artists=("msd_artist_id", "nunique"),
        visual=("visual_correct", "mean"),
        audio=("audio_correct", "mean"),
        early=("early_fusion_correct", "mean"),
        late=("late_fusion_correct", "mean"),
        adaptive=("adaptive_fusion_correct", "mean")
    )
    .reset_index()
)

# Calculate fusion gains relative to audio.
comparison["early_minus_audio"] = (
    comparison["early"] - comparison["audio"]
)

comparison["late_minus_audio"] = (
    comparison["late"] - comparison["audio"]
)

comparison["adaptive_minus_audio"] = (
    comparison["adaptive"] - comparison["audio"]
)

# ------------------------------------------------------------
# 6. Display early-fusion gains by genre and quintile
# ------------------------------------------------------------

early_pivot = comparison.pivot(
    index="genre",
    columns="typicality_quintile",
    values="early_minus_audio"
)

print("\n" + "=" * 65)
print("EARLY FUSION MINUS AUDIO BY GENRE")
print("=" * 65)

display(early_pivot.round(4))

# ------------------------------------------------------------
# 7. Display fixed late-fusion gains
# ------------------------------------------------------------

late_pivot = comparison.pivot(
    index="genre",
    columns="typicality_quintile",
    values="late_minus_audio"
)

print("\n" + "=" * 65)
print("FIXED LATE FUSION MINUS AUDIO BY GENRE")
print("=" * 65)

display(late_pivot.round(4))

# ------------------------------------------------------------
# 8. Display adaptive late-fusion gains
# ------------------------------------------------------------

adaptive_pivot = comparison.pivot(
    index="genre",
    columns="typicality_quintile",
    values="adaptive_minus_audio"
)

print("\n" + "=" * 65)
print("ADAPTIVE FUSION MINUS AUDIO BY GENRE")
print("=" * 65)

display(adaptive_pivot.round(4))

# ------------------------------------------------------------
# 9. Compare Q1 and Q5 early-fusion gains by genre
# ------------------------------------------------------------

genre_contrasts = []

for genre, group in comparison.groupby("genre"):

    by_quintile = group.set_index(
        "typicality_quintile"
    )

    assert 1 in by_quintile.index
    assert 5 in by_quintile.index

    q1_gain = by_quintile.loc[
        1, "early_minus_audio"
    ]

    q5_gain = by_quintile.loc[
        5, "early_minus_audio"
    ]

    genre_contrasts.append({
        "genre": genre,
        "n_q1": int(by_quintile.loc[1, "n"]),
        "n_q5": int(by_quintile.loc[5, "n"]),
        "early_gain_q1": q1_gain,
        "early_gain_q5": q5_gain,
        "q1_minus_q5_contrast": q1_gain - q5_gain
    })

contrast_df = pd.DataFrame(genre_contrasts)

contrast_df = contrast_df.sort_values(
    "q1_minus_q5_contrast",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("GENRE-SPECIFIC Q1–Q5 CONTRASTS")
print("=" * 65)

display(contrast_df.round(4))

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

q5_path = (
    output_dir / "q5_accuracy_by_genre.csv"
)

comparison_path = (
    output_dir / "genre_typicality_test_comparison.csv"
)

contrast_path = (
    output_dir / "genre_typicality_q1_q5_contrasts.csv"
)

q5_results.to_csv(
    q5_path,
    index=False
)

comparison.to_csv(
    comparison_path,
    index=False
)

contrast_df.to_csv(
    contrast_path,
    index=False
)

print("\nSaved files:")
print("-", q5_path.name)
print("-", comparison_path.name)
print("-", contrast_path.name)

# ------------------------------------------------------------
# 11. Final methodological notes
# ------------------------------------------------------------

print("\nIMPORTANT:")
print(
    "All analyses use the held-out "
    "artist-disjoint test set."
)

print(
    "Typicality quintiles were defined "
    "separately within each true genre."
)

print(
    "Genre-level subgroup sizes are relatively small."
)

print(
    "These results are exploratory and should not "
    "be interpreted as definitive genre-level effects."
)

print("\nGENRE-SPECIFIC TYPICALITY ANALYSIS COMPLETE")


### Visualization of Final Multimodal Classification Results

We summarize the main findings of our multimodal classification study using a three-panel figure.

**Panel A — Overall classification performance:** Compares the test macro F1 scores of the visual-only, audio-only, early-fusion, fixed late-fusion, and adaptive late-fusion models.

**Panel B — Performance across visual typicality:** Shows how the accuracy of each model changes from the most visually typical album covers (Q1) to the most visually atypical (Q5).

**Panel C — Benefits and disadvantages of fusion:** Displays the accuracy difference between each multimodal approach and the audio-only baseline across the five typicality quintiles.

Together, these visualizations distinguish improvements in overall classification performance from the potentially negative effects of incorporating visual information for atypical album covers.

All figures are generated from the held-out artist-disjoint test results.

In [ ]:

# ============================================================
# PHASE 5 — FINAL RESEARCH VISUALIZATIONS
# ARTIST-DISJOINT HELD-OUT TEST RESULTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 65)
print("FINAL MULTIMODAL CLASSIFICATION FIGURES")
print("ARTIST-DISJOINT HELD-OUT TEST RESULTS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

# PROJECT was defined in Cell 1.
output_dir = (
    PROJECT
    / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

assert output_dir.is_dir(), (
    "Artist-disjoint results directory not found."
)

# ------------------------------------------------------------
# 2. Load final test results
# ------------------------------------------------------------

metrics = pd.read_csv(
    output_dir / "final_test_metrics.csv"
)

# Cell 25 saves the quintile as the CSV index.
typicality = pd.read_csv(
    output_dir / "final_test_typicality_summary.csv",
    index_col=0
)

typicality.index = typicality.index.astype(int)
typicality.index.name = "typicality_quintile"

# ------------------------------------------------------------
# 3. Validate saved results
# ------------------------------------------------------------

model_order = [
    "Visual-only",
    "Audio-only",
    "Early fusion",
    "Fixed late fusion",
    "Adaptive late fusion"
]

required_metrics = {
    "model",
    "accuracy",
    "balanced_accuracy",
    "macro_f1"
}

required_typicality = {
    "n",
    "visual",
    "audio",
    "early_fusion",
    "late_fusion",
    "adaptive_fusion",
    "early_minus_audio",
    "late_minus_audio",
    "adaptive_minus_audio"
}

assert required_metrics.issubset(metrics.columns)
assert required_typicality.issubset(typicality.columns)

assert metrics["model"].is_unique

assert set(metrics["model"]) == set(model_order), (
    "Unexpected model names in final test metrics."
)

assert set(typicality.index) == {1, 2, 3, 4, 5}, (
    "Expected typicality quintiles Q1–Q5."
)

assert typicality.index.is_unique

assert metrics[
    ["accuracy", "balanced_accuracy", "macro_f1"]
].notna().all().all()

assert typicality[
    list(required_typicality)
].notna().all().all()

assert (typicality["n"] > 0).all()

for column in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1"
]:
    assert metrics[column].between(0, 1).all()

for column in [
    "visual",
    "audio",
    "early_fusion",
    "late_fusion",
    "adaptive_fusion"
]:
    assert typicality[column].between(0, 1).all()

typicality = typicality.sort_index()

ordered = metrics.set_index("model").loc[model_order]

print("\nFINAL TEST MACRO F1")
display(
    ordered[["macro_f1"]].round(4)
)

print("\nTEST ACCURACY BY TYPICALITY")
display(
    typicality.round(4)
)

# ------------------------------------------------------------
# 4. Set up dark-background figure
# ------------------------------------------------------------

with plt.style.context("dark_background"):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(19, 5.8),
        facecolor="#111318"
    )

    for ax in axes:
        ax.set_facecolor("#111318")

    colors = {
        "Visual-only": "#9B8AD5",
        "Audio-only": "#50B4D8",
        "Early fusion": "#58C79A",
        "Fixed late fusion": "#F1B65B",
        "Adaptive late fusion": "#E58CA9"
    }

    # --------------------------------------------------------
    # PANEL A — OVERALL MACRO F1
    # --------------------------------------------------------

    x_positions = np.arange(len(model_order))

    macro_f1_values = (
        ordered["macro_f1"].to_numpy()
    )

    bars = axes[0].bar(
        x_positions,
        macro_f1_values,
        width=0.68,
        color=[
            colors[model]
            for model in model_order
        ]
    )

    axes[0].set_xticks(x_positions)

    axes[0].set_xticklabels(
        [
            "Visual",
            "Audio",
            "Early",
            "Fixed\nlate",
            "Adaptive\nlate"
        ],
        fontsize=10
    )

    axes[0].set_ylim(
        0,
        max(0.8, float(macro_f1_values.max()) + 0.10)
    )

    axes[0].set_ylabel(
        "Test macro F1",
        fontsize=11
    )

    axes[0].set_title(
        "A. Overall test performance",
        fontsize=13,
        fontweight="bold",
        pad=15
    )

    for bar, value in zip(bars, macro_f1_values):

        axes[0].annotate(
            f"{value:.3f}",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height()
            ),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10
        )

    # --------------------------------------------------------
    # PANEL B — ACCURACY BY VISUAL TYPICALITY
    # --------------------------------------------------------

    quintiles = typicality.index.to_numpy()

    accuracy_columns = {
        "Visual-only": "visual",
        "Audio-only": "audio",
        "Early fusion": "early_fusion",
        "Fixed late fusion": "late_fusion",
        "Adaptive late fusion": "adaptive_fusion"
    }

    for model_name, column in accuracy_columns.items():

        axes[1].plot(
            quintiles,
            typicality[column].to_numpy(),
            marker="o",
            linewidth=2.2,
            markersize=6,
            label=model_name,
            color=colors[model_name]
        )

    axes[1].set_xticks([1, 2, 3, 4, 5])

    axes[1].set_xticklabels(
        ["Q1", "Q2", "Q3", "Q4", "Q5"]
    )

    axes[1].set_ylim(0, 1)

    axes[1].set_ylabel(
        "Test accuracy",
        fontsize=11
    )

    axes[1].set_xlabel(
        "Visual typicality (typical → atypical)",
        fontsize=10
    )

    axes[1].set_title(
        "B. Accuracy by visual typicality",
        fontsize=13,
        fontweight="bold",
        pad=15
    )

    axes[1].legend(
        fontsize=8,
        loc="lower left",
        framealpha=0.85
    )

    # --------------------------------------------------------
    # PANEL C — FUSION GAIN RELATIVE TO AUDIO
    # --------------------------------------------------------

    gain_columns = {
        "Early fusion": "early_minus_audio",
        "Fixed late fusion": "late_minus_audio",
        "Adaptive late fusion": "adaptive_minus_audio"
    }

    for model_name, column in gain_columns.items():

        axes[2].plot(
            quintiles,
            100 * typicality[column].to_numpy(),
            marker="o",
            linewidth=2.2,
            markersize=6,
            label=model_name,
            color=colors[model_name]
        )

    axes[2].axhline(
        0,
        color="white",
        linestyle="--",
        linewidth=1.2,
        alpha=0.75
    )

    axes[2].set_xticks([1, 2, 3, 4, 5])

    axes[2].set_xticklabels(
        ["Q1", "Q2", "Q3", "Q4", "Q5"]
    )

    axes[2].set_ylabel(
        "Accuracy difference vs audio (pp)",
        fontsize=11
    )

    axes[2].set_xlabel(
        "Visual typicality (typical → atypical)",
        fontsize=10
    )

    axes[2].set_title(
        "C. When does fusion help?",
        fontsize=13,
        fontweight="bold",
        pad=15
    )

    axes[2].legend(
        fontsize=8,
        loc="best",
        framealpha=0.85
    )

    # --------------------------------------------------------
    # 5. Final formatting
    # --------------------------------------------------------

    for ax in axes:

        ax.grid(
            axis="y",
            alpha=0.18,
            linewidth=0.8
        )

        ax.set_axisbelow(True)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.tick_params(
            axis="both",
            labelsize=10
        )

    fig.suptitle(
        "Multimodal Music Genre Classification",
        fontsize=18,
        fontweight="bold",
        y=1.02
    )

    fig.text(
        0.5,
        0.965,
        "Held-out artist-disjoint test set",
        ha="center",
        fontsize=11,
        color="#BFC5CE"
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.93],
        w_pad=2.5
    )

    # --------------------------------------------------------
    # 6. Save figure
    # --------------------------------------------------------

    png_path = (
        figure_dir / "final_multimodal_results.png"
    )

    pdf_path = (
        figure_dir / "final_multimodal_results.pdf"
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor=fig.get_facecolor()
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor=fig.get_facecolor()
    )

    plt.show()
    plt.close(fig)

print("\n" + "=" * 65)
print("FINAL FIGURES SAVED")
print("=" * 65)

print("PNG:", png_path)
print("PDF:", pdf_path)

print("\nIMPORTANT:")
print(
    "All figure values were loaded from "
    "saved artist-disjoint test results."
)

print(
    "Q1 represents the most visually typical "
    "covers and Q5 the most atypical."
)

print(
    "Panel C reports accuracy differences "
    "in percentage points."
)

print("\nFINAL RESEARCH VISUALIZATIONS COMPLETE")


In [ ]:

# EXPERIMENT 2 — STEP 1
# Inspect original multimodal embeddings and index alignment
# This cell does NOT train models or modify existing files.

import tarfile
import io
import numpy as np
import pandas as pd
from pathlib import Path

archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

assert archive_path.exists(), "MSD-I archive not found"
assert saved_dir.exists(), "Saved experiment not found"

print("=" * 65)
print("EXPERIMENT 2: MULTIMODAL REPRESENTATIONS")
print("=" * 65)

with tarfile.open(archive_path, "r:gz") as tar:

    members = {
        m.name: m
        for m in tar.getmembers()
        if m.isfile()
    }

    def read_array(name):
        assert name in members, f"Missing: {name}"
        raw = tar.extractfile(members[name]).read()
        return np.load(io.BytesIO(raw), allow_pickle=False)

    def read_index(name):
        assert name in members, f"Missing: {name}"
        raw = tar.extractfile(members[name]).read()
        df = pd.read_csv(
            io.BytesIO(raw),
            sep="\t",
            header=None
        )
        return df.iloc[:, 0].astype(str).str.strip().tolist()

    for split in ["train", "val", "test"]:

        print(f"\n{'-' * 55}")
        print("ORIGINAL SOURCE SPLIT:", split.upper())

        prefix = "msdi-data/train_data/"

        index_path = (
            f"msdi-data/splits/"
            f"items_index_{split}_MSD-I.tsv"
        )

        track_ids = read_index(index_path)

        arrays = {}

        for modality in [
            "audio",
            "visual",
            "mm-audio",
            "mm-visual"
        ]:

            name = (
                prefix
                + f"X_{split}_{modality}_MSD-I.npy"
            )

            X = read_array(name)
            arrays[modality] = X

            print(
                f"{modality:10s}: "
                f"{X.shape} | "
                f"index rows: {len(track_ids)}"
            )

            assert X.shape[0] == len(track_ids)
            assert np.isfinite(X).all()

        assert arrays["audio"].shape[1] == 2048
        assert arrays["visual"].shape[1] == 2048
        assert arrays["mm-audio"].shape[1] == 200
        assert arrays["mm-visual"].shape[1] == 200

        assert len(set(track_ids)) == len(track_ids)

        print("All dimensions and row counts: PASSED")

print("\n" + "=" * 65)
print("SAVED EXPERIMENT CHECK")
print("=" * 65)

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    print(
        split.upper(),
        "| tracks:", len(labels),
        "| unique IDs:",
        labels["msd_track_id"].nunique()
    )

print("\nINSPECTION COMPLETE")
print("No models trained.")
print("No files overwritten.")


In [ ]:

# EXPERIMENT 2 — STEP 2
# Align and extract multimodal representations.
# Original experiment files are never overwritten.

import tarfile
import io
import numpy as np
import pandas as pd
from pathlib import Path

archive_path = PROJECT / "audio_features" / "msdi-data.tar.gz"

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

new_dir.mkdir(parents=True, exist_ok=True)

splits = ["train", "val", "test"]
modalities = ["mm-audio", "mm-visual"]

# Load the previously verified label files.
labels_by_split = {
    s: pd.read_csv(saved_dir / f"labels_{s}.csv")
    for s in splits
}

all_labels = pd.concat(
    labels_by_split.values(),
    ignore_index=True
)

assert len(all_labels) == 11870
assert all_labels["msd_track_id"].is_unique

# Build a lookup from track ID to its original archive row.
source_lookup = {}

with tarfile.open(archive_path, "r:gz") as tar:

    members = {
        m.name: m
        for m in tar.getmembers()
        if m.isfile()
    }

    def load_member_array(name):
        raw = tar.extractfile(members[name]).read()
        return np.load(io.BytesIO(raw), allow_pickle=False)

    for original_split in splits:

        index_name = (
            f"msdi-data/splits/"
            f"items_index_{original_split}_MSD-I.tsv"
        )

        index_raw = tar.extractfile(
            members[index_name]
        ).read()

        index_df = pd.read_csv(
            io.BytesIO(index_raw),
            sep="\t",
            header=None
        )

        track_ids = (
            index_df.iloc[:, 0]
            .astype(str)
            .str.strip()
            .tolist()
        )

        assert len(set(track_ids)) == len(track_ids)

        for row, track_id in enumerate(track_ids):
            assert track_id not in source_lookup
            source_lookup[track_id] = (
                original_split, row
            )

    print("Original archive tracks:", len(source_lookup))

    assert set(all_labels["msd_track_id"]).issubset(
        source_lookup.keys()
    )

    # Prepare target arrays in the exact saved label order.
    extracted = {
        s: {
            m: np.empty(
                (len(labels_by_split[s]), 200),
                dtype=np.float32
            )
            for m in modalities
        }
        for s in splits
    }

    filled = {
        s: np.zeros(
            len(labels_by_split[s]),
            dtype=bool
        )
        for s in splits
    }

    # Verify old CNN embeddings before relying on archive order.
    old_audio = {
        s: np.load(
            saved_dir / f"X_{s}_audio.npy",
            mmap_mode="r"
        )
        for s in splits
    }

    old_visual = {
        s: np.load(
            saved_dir / f"X_{s}_visual.npy",
            mmap_mode="r"
        )
        for s in splits
    }

    print("\nChecking source alignment...")

    for original_split in splits:

        source_audio = load_member_array(
            f"msdi-data/train_data/"
            f"X_{original_split}_audio_MSD-I.npy"
        )

        source_visual = load_member_array(
            f"msdi-data/train_data/"
            f"X_{original_split}_visual_MSD-I.npy"
        )

        source_mm = {
            m: load_member_array(
                f"msdi-data/train_data/"
                f"X_{original_split}_{m}_MSD-I.npy"
            )
            for m in modalities
        }

        for target_split in splits:

            labels = labels_by_split[target_split]

            selected = [
                (i, source_lookup[track_id][1])
                for i, track_id in enumerate(
                    labels["msd_track_id"]
                )
                if source_lookup[track_id][0]
                == original_split
            ]

            if not selected:
                continue

            target_rows, source_rows = map(
                np.asarray,
                zip(*selected)
            )

            target_rows = target_rows.astype(int)
            source_rows = source_rows.astype(int)

            audio_matches = np.allclose(
                old_audio[target_split][target_rows],
                source_audio[source_rows],
                rtol=1e-5,
                atol=1e-6
            )

            visual_matches = np.allclose(
                old_visual[target_split][target_rows],
                source_visual[source_rows],
                rtol=1e-5,
                atol=1e-6
            )

            print(
                f"{original_split} -> {target_split}: "
                f"{len(target_rows)} tracks | "
                f"audio={audio_matches} | "
                f"visual={visual_matches}"
            )

            assert audio_matches, (
                "Audio row alignment failed"
            )

            assert visual_matches, (
                "Visual row alignment failed"
            )

            for m in modalities:
                extracted[target_split][m][target_rows] = (
                    source_mm[m][source_rows]
                )

            filled[target_split][target_rows] = True

    print("\nChecking complete extraction...")

    for split in splits:

        assert filled[split].all(), (
            f"Missing rows in {split}"
        )

        for m in modalities:
            X = extracted[split][m]

            assert X.shape == (
                len(labels_by_split[split]), 200
            )

            assert np.isfinite(X).all()

            path = new_dir / f"X_{split}_{m}.npy"

            np.save(path, X)

            print(
                f"Saved {path.name}: {X.shape}"
            )

        labels_by_split[split].to_csv(
            new_dir / f"labels_{split}.csv",
            index=False
        )

print("\nEXPERIMENT 2 — STEP 2 COMPLETE")
print("Original files preserved.")


In [ ]:

# EXPERIMENT 2 — STEP 3
# Train CNN Audio + MM Audio logistic regression.
# Select C on validation only.
# Save the trained model and validation results.
# Do NOT evaluate on the test set yet.

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from pathlib import Path

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

new_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# 1. LOAD FEATURES
# -------------------------------------------------

X_train_audio = np.load(
    saved_dir / "X_train_audio.npy"
)

X_val_audio = np.load(
    saved_dir / "X_val_audio.npy"
)

X_train_mm_audio = np.load(
    new_dir / "X_train_mm-audio.npy"
)

X_val_mm_audio = np.load(
    new_dir / "X_val_mm-audio.npy"
)

train_labels = pd.read_csv(
    new_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    new_dir / "labels_val.csv"
)

# -------------------------------------------------
# 2. COMBINE REPRESENTATIONS
# -------------------------------------------------

X_train = np.concatenate(
    [X_train_audio, X_train_mm_audio],
    axis=1
)

X_val = np.concatenate(
    [X_val_audio, X_val_mm_audio],
    axis=1
)

y_train = train_labels["genre"].to_numpy()
y_val = val_labels["genre"].to_numpy()

assert X_train.shape == (8242, 2248)
assert X_val.shape == (1836, 2248)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# -------------------------------------------------
# 3. VALIDATION HYPERPARAMETER SEARCH
# -------------------------------------------------

C_values = [0.001, 0.01, 0.1, 1, 10]

results = []
models = {}

for C in C_values:

    print(f"\nTraining C = {C}")

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=C,
            solver="lbfgs",
            max_iter=2000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    metrics = {
        "C": C,
        "accuracy": accuracy_score(y_val, pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, pred
        ),
        "macro_f1": f1_score(
            y_val, pred,
            average="macro"
        )
    }

    results.append(metrics)
    models[C] = model

    print(
        f"Validation macro F1: "
        f"{metrics['macro_f1']:.4f}"
    )

results_df = pd.DataFrame(results)

# -------------------------------------------------
# 4. SELECT BEST MODEL
# -------------------------------------------------

best_row = results_df.sort_values(
    ["macro_f1", "C"],
    ascending=[False, True]
).iloc[0]

best_C = float(best_row["C"])
best_model = models[best_C]

print("\n" + "=" * 60)
print("CNN AUDIO + MM AUDIO — VALIDATION RESULTS")
print("=" * 60)

print(results_df.to_string(index=False))

print("\nBest C:", best_C)
print(
    "Best validation macro F1:",
    round(float(best_row["macro_f1"]), 4)
)

# -------------------------------------------------
# 5. SAVE MODEL AND RESULTS
# -------------------------------------------------

results_df.to_csv(
    new_dir / "cnn_audio_mm_audio_validation.csv",
    index=False
)

joblib.dump(
    best_model,
    new_dir / "cnn_audio_mm_audio_model.joblib"
)

best_pred = best_model.predict(X_val)

predictions_df = val_labels.copy()
predictions_df["prediction"] = best_pred
predictions_df["correct"] = (
    best_pred == y_val
)

predictions_df.to_csv(
    new_dir / "cnn_audio_mm_audio_val_predictions.csv",
    index=False
)

print("\nModel and validation results saved.")
print("Test set has NOT been evaluated.")
print("\nSTEP 3 COMPLETE")


In [ ]:

# EXPERIMENT 2 — STEP 4
# Compare original audio-only and CNN Audio + MM Audio
# on the SAME validation set.

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

# Load original audio-only model.
audio_model = joblib.load(
    saved_dir / "audio_logistic_best.joblib"
)

# Load new model.
mm_model = joblib.load(
    new_dir / "cnn_audio_mm_audio_model.joblib"
)

# Load validation features.
X_audio = np.load(
    saved_dir / "X_val_audio.npy"
)

X_mm_audio = np.load(
    new_dir / "X_val_mm-audio.npy"
)

X_combined = np.concatenate(
    [X_audio, X_mm_audio],
    axis=1
)

labels = pd.read_csv(
    new_dir / "labels_val.csv"
)

y_true = labels["genre"].to_numpy()

# Generate predictions.
pred_audio = audio_model.predict(X_audio)
pred_mm = mm_model.predict(X_combined)

def evaluate(name, predictions):
    return {
        "model": name,
        "accuracy": accuracy_score(
            y_true, predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "macro_f1": f1_score(
            y_true, predictions,
            average="macro"
        )
    }

comparison = pd.DataFrame([
    evaluate("CNN Audio", pred_audio),
    evaluate("CNN Audio + MM Audio", pred_mm)
])

print("=" * 65)
print("VALIDATION COMPARISON")
print("=" * 65)

print(comparison.to_string(index=False))

audio_f1 = comparison.iloc[0]["macro_f1"]
mm_f1 = comparison.iloc[1]["macro_f1"]

difference = mm_f1 - audio_f1

print("\nMacro F1 difference:", round(difference, 4))

if difference > 0:
    print("Adding MM Audio IMPROVED validation macro F1.")
elif difference < 0:
    print("Adding MM Audio REDUCED validation macro F1.")
else:
    print("Both models achieved the same macro F1.")

comparison.to_csv(
    new_dir / "audio_mm_audio_validation_comparison.csv",
    index=False
)

print("\nSTEP 4 COMPLETE")


In [ ]:

# EXPERIMENT 2 — STEP 5
# Train four additional representation combinations.
# Validation only — do not evaluate test set.

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from pathlib import Path

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

new_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------
# 1. LOAD FEATURE MATRICES
# ---------------------------------------------

features = {}

for split in ["train", "val"]:

    features[split] = {
        "cnn_audio": np.load(
            saved_dir / f"X_{split}_audio.npy"
        ),
        "cnn_visual": np.load(
            saved_dir / f"X_{split}_visual.npy"
        ),
        "mm_audio": np.load(
            new_dir / f"X_{split}_mm-audio.npy"
        ),
        "mm_visual": np.load(
            new_dir / f"X_{split}_mm-visual.npy"
        )
    }

labels_train = pd.read_csv(
    new_dir / "labels_train.csv"
)

labels_val = pd.read_csv(
    new_dir / "labels_val.csv"
)

y_train = labels_train["genre"].to_numpy()
y_val = labels_val["genre"].to_numpy()

# ---------------------------------------------
# 2. DEFINE EXPERIMENTS
# ---------------------------------------------

experiments = {
    "mm_audio": ["mm_audio"],
    "mm_visual": ["mm_visual"],
    "mm_fusion": ["mm_audio", "mm_visual"],
    "all_four": [
        "cnn_audio",
        "cnn_visual",
        "mm_audio",
        "mm_visual"
    ]
}

C_values = [0.001, 0.01, 0.1, 1, 10]

all_results = []
best_models = {}

# ---------------------------------------------
# 3. TRAIN EACH MODEL
# ---------------------------------------------

for model_name, feature_names in experiments.items():

    print("\n" + "=" * 65)
    print("MODEL:", model_name.upper())
    print("=" * 65)

    X_train = np.concatenate(
        [features["train"][f] for f in feature_names],
        axis=1
    )

    X_val = np.concatenate(
        [features["val"][f] for f in feature_names],
        axis=1
    )

    print("Train shape:", X_train.shape)
    print("Validation shape:", X_val.shape)

    model_results = []
    fitted_models = {}

    for C in C_values:

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=C,
                solver="lbfgs",
                max_iter=2000,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        predictions = model.predict(X_val)

        result = {
            "model": model_name,
            "C": C,
            "accuracy": accuracy_score(
                y_val, predictions
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "macro_f1": f1_score(
                y_val, predictions,
                average="macro"
            )
        }

        model_results.append(result)
        fitted_models[C] = model

        print(
            f"C={C:<7} "
            f"macro F1={result['macro_f1']:.4f}"
        )

    results_df = pd.DataFrame(model_results)

    best_row = results_df.sort_values(
        ["macro_f1", "C"],
        ascending=[False, True]
    ).iloc[0]

    best_C = float(best_row["C"])
    best_model = fitted_models[best_C]

    best_models[model_name] = best_model
    all_results.extend(model_results)

    joblib.dump(
        best_model,
        new_dir / f"{model_name}_best.joblib"
    )

    best_predictions = best_model.predict(X_val)

    predictions_df = labels_val.copy()
    predictions_df["prediction"] = best_predictions
    predictions_df["correct"] = (
        best_predictions == y_val
    )

    predictions_df.to_csv(
        new_dir / f"{model_name}_val_predictions.csv",
        index=False
    )

    print(
        f"\nBest C={best_C}, "
        f"validation macro F1="
        f"{best_row['macro_f1']:.4f}"
    )

# ---------------------------------------------
# 4. SAVE AND SUMMARIZE
# ---------------------------------------------

all_results_df = pd.DataFrame(all_results)

all_results_df.to_csv(
    new_dir / "representation_validation_grid.csv",
    index=False
)

best_summary = (
    all_results_df
    .sort_values(
        ["macro_f1", "C"],
        ascending=[False, True]
    )
    .drop_duplicates("model")
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

best_summary.to_csv(
    new_dir / "representation_validation_best.csv",
    index=False
)

print("\n" + "=" * 65)
print("BEST VALIDATION RESULTS")
print("=" * 65)

print(best_summary.to_string(index=False))

print("\nSTEP 5 COMPLETE")
print("No test predictions generated.")


In [ ]:

# EXPERIMENT 2 — STEP 6
# Compare all representation models on validation.
# No retraining and no test-set evaluation.

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

labels = pd.read_csv(
    new_dir / "labels_val.csv"
)

y_true = labels["genre"].to_numpy()

# Load validation features.
X_audio = np.load(saved_dir / "X_val_audio.npy")
X_visual = np.load(saved_dir / "X_val_visual.npy")
X_mm_audio = np.load(new_dir / "X_val_mm-audio.npy")
X_mm_visual = np.load(new_dir / "X_val_mm-visual.npy")

# Define models and feature combinations.
model_specs = {
    "CNN Audio": (
        saved_dir / "audio_logistic_best.joblib",
        X_audio
    ),
    "CNN Visual": (
        saved_dir / "visual_logistic_best.joblib",
        X_visual
    ),
    "CNN Early Fusion": (
        saved_dir / "fusion_logistic_best.joblib",
        np.concatenate(
            [X_audio, X_visual], axis=1
        )
    ),
    "MM Audio": (
        new_dir / "mm_audio_best.joblib",
        X_mm_audio
    ),
    "MM Visual": (
        new_dir / "mm_visual_best.joblib",
        X_mm_visual
    ),
    "MM Fusion": (
        new_dir / "mm_fusion_best.joblib",
        np.concatenate(
            [X_mm_audio, X_mm_visual], axis=1
        )
    ),
    "CNN Audio + MM Audio": (
        new_dir / "cnn_audio_mm_audio_model.joblib",
        np.concatenate(
            [X_audio, X_mm_audio], axis=1
        )
    ),
    "All Four": (
        new_dir / "all_four_best.joblib",
        np.concatenate(
            [
                X_audio, X_visual,
                X_mm_audio, X_mm_visual
            ],
            axis=1
        )
    )
}

rows = []

for name, (model_path, X) in model_specs.items():

    model = joblib.load(model_path)
    pred = model.predict(X)

    rows.append({
        "model": name,
        "dimensions": X.shape[1],
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, pred
        ),
        "macro_f1": f1_score(
            y_true, pred, average="macro"
        )
    })

comparison = pd.DataFrame(rows).sort_values(
    "macro_f1", ascending=False
)

print("=" * 75)
print("ALL REPRESENTATION MODELS — VALIDATION")
print("=" * 75)

print(comparison.to_string(index=False))

comparison.to_csv(
    new_dir / "all_representation_validation_comparison.csv",
    index=False
)

# Compare two fusion methods.
f1_by_model = comparison.set_index("model")["macro_f1"]

difference = (
    f1_by_model["MM Fusion"]
    - f1_by_model["CNN Early Fusion"]
)

print("\nMM Fusion minus CNN Early Fusion:")
print(f"{difference:+.4f}")

print("\nSTEP 6 COMPLETE")
print("Test set remains untouched.")


In [ ]:

# EXPERIMENT 2 — STEP 7
# Visual typicality analysis on VALIDATION only.

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_distances

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

# Load labels.
train_labels = pd.read_csv(
    new_dir / "labels_train.csv"
)

val_labels = pd.read_csv(
    new_dir / "labels_val.csv"
)

y_train = train_labels["genre"].to_numpy()
y_val = val_labels["genre"].to_numpy()

# Load CNN visual features for typicality.
X_train_visual = np.load(
    saved_dir / "X_train_visual.npy"
)

X_val_visual = np.load(
    saved_dir / "X_val_visual.npy"
)

# Standardize using TRAIN only.
visual_scaler = StandardScaler()
Z_train = visual_scaler.fit_transform(X_train_visual)
Z_val = visual_scaler.transform(X_val_visual)

# Genre centroids in standardized visual space.
centroids = {
    genre: Z_train[y_train == genre].mean(axis=0)
    for genre in np.unique(y_train)
}

# Cosine distance to TRUE-GENRE centroid.
distances = np.array([
    cosine_distances(
        Z_val[i:i+1],
        centroids[genre].reshape(1, -1)
    )[0, 0]
    for i, genre in enumerate(y_val)
])

details = val_labels.copy()
details["visual_typicality_distance"] = distances

# Define quintiles separately within each genre.
details["typicality_quintile"] = (
    details.groupby("genre")[
        "visual_typicality_distance"
    ]
    .transform(
        lambda s: pd.qcut(
            s.rank(method="first"),
            q=5,
            labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
        )
    )
    .astype(str)
)

# Load features.
X_audio = np.load(saved_dir / "X_val_audio.npy")
X_visual = X_val_visual
X_mm_audio = np.load(new_dir / "X_val_mm-audio.npy")
X_mm_visual = np.load(new_dir / "X_val_mm-visual.npy")

models = {
    "CNN Audio": (
        saved_dir / "audio_logistic_best.joblib",
        X_audio
    ),
    "CNN Early Fusion": (
        saved_dir / "fusion_logistic_best.joblib",
        np.concatenate(
            [X_audio, X_visual], axis=1
        )
    ),
    "MM Fusion": (
        new_dir / "mm_fusion_best.joblib",
        np.concatenate(
            [X_mm_audio, X_mm_visual], axis=1
        )
    ),
    "All Four": (
        new_dir / "all_four_best.joblib",
        np.concatenate(
            [
                X_audio, X_visual,
                X_mm_audio, X_mm_visual
            ],
            axis=1
        )
    )
}

for name, (path, X) in models.items():
    model = joblib.load(path)
    pred = model.predict(X)

    details[f"{name}_correct"] = (
        pred == y_val
    ).astype(int)

# Summarize accuracy by typicality quintile.
summary = (
    details.groupby("typicality_quintile")
    [[f"{name}_correct" for name in models]]
    .mean()
    .reindex(["Q1", "Q2", "Q3", "Q4", "Q5"])
)

summary.columns = list(models.keys())

summary["CNN Fusion Gain"] = (
    summary["CNN Early Fusion"]
    - summary["CNN Audio"]
)

summary["MM Fusion Gain"] = (
    summary["MM Fusion"]
    - summary["CNN Audio"]
)

summary["MM minus CNN Fusion"] = (
    summary["MM Fusion"]
    - summary["CNN Early Fusion"]
)

print("=" * 80)
print("VALIDATION ACCURACY BY VISUAL TYPICALITY")
print("=" * 80)

print((summary * 100).round(2).to_string())

summary.to_csv(
    new_dir / "representation_typicality_validation.csv"
)

details.to_csv(
    new_dir / "representation_typicality_validation_details.csv",
    index=False
)

print("\nSTEP 7 COMPLETE")
print("All values above are percentages.")
print("No test data used.")


In [ ]:

# EXPERIMENT 2 — STEP 8
# Final test evaluation of pretrained representation models.
# No retraining or hyperparameter selection.

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

labels = pd.read_csv(
    new_dir / "labels_test.csv"
)

y_true = labels["genre"].to_numpy()

X_audio = np.load(saved_dir / "X_test_audio.npy")
X_visual = np.load(saved_dir / "X_test_visual.npy")
X_mm_audio = np.load(new_dir / "X_test_mm-audio.npy")
X_mm_visual = np.load(new_dir / "X_test_mm-visual.npy")

model_specs = {
    "CNN Audio": (
        saved_dir / "audio_logistic_best.joblib",
        X_audio
    ),
    "CNN Early Fusion": (
        saved_dir / "fusion_logistic_best.joblib",
        np.concatenate(
            [X_audio, X_visual], axis=1
        )
    ),
    "MM Audio": (
        new_dir / "mm_audio_best.joblib",
        X_mm_audio
    ),
    "MM Fusion": (
        new_dir / "mm_fusion_best.joblib",
        np.concatenate(
            [X_mm_audio, X_mm_visual], axis=1
        )
    ),
    "All Four": (
        new_dir / "all_four_best.joblib",
        np.concatenate(
            [
                X_audio, X_visual,
                X_mm_audio, X_mm_visual
            ],
            axis=1
        )
    )
}

rows = []
predictions_df = labels.copy()

for name, (path, X) in model_specs.items():

    model = joblib.load(path)
    pred = model.predict(X)

    rows.append({
        "model": name,
        "dimensions": X.shape[1],
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, pred
        ),
        "macro_f1": f1_score(
            y_true, pred, average="macro"
        )
    })

    column = name.lower().replace(" ", "_")

    predictions_df[f"{column}_prediction"] = pred
    predictions_df[f"{column}_correct"] = (
        pred == y_true
    ).astype(int)

results = pd.DataFrame(rows).sort_values(
    "macro_f1", ascending=False
)

print("=" * 75)
print("FINAL TEST — REPRESENTATION COMPARISON")
print("=" * 75)

print(results.to_string(index=False))

results.to_csv(
    new_dir / "representation_final_test_metrics.csv",
    index=False
)

predictions_df.to_csv(
    new_dir / "representation_final_test_predictions.csv",
    index=False
)

print("\nSTEP 8 COMPLETE")
print("Models and hyperparameters were fixed before evaluation.")


In [ ]:

# EXPERIMENT 2 — STEP 9
# Test-set typicality analysis + artist-cluster bootstrap.
# No training, no hyperparameter tuning.

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

train_labels = pd.read_csv(
    new_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    new_dir / "labels_test.csv"
)

predictions = pd.read_csv(
    new_dir / "representation_final_test_predictions.csv"
)

assert predictions["msd_track_id"].equals(
    test_labels["msd_track_id"]
)

X_train_visual = np.load(
    saved_dir / "X_train_visual.npy"
)

X_test_visual = np.load(
    saved_dir / "X_test_visual.npy"
)

y_train = train_labels["genre"].to_numpy()
y_test = test_labels["genre"].to_numpy()

# 1. Fit visual scaler on training only.
scaler = StandardScaler()
Z_train = scaler.fit_transform(X_train_visual)
Z_test = scaler.transform(X_test_visual)

centroids = {
    genre: Z_train[y_train == genre].mean(axis=0)
    for genre in np.unique(y_train)
}

# 2. Compute cosine distance to true-genre centroid.
distances = np.empty(len(y_test))

for genre in np.unique(y_test):

    idx = np.flatnonzero(y_test == genre)

    Z = Z_test[idx]
    centroid = centroids[genre]

    numerator = Z @ centroid
    denominator = (
        np.linalg.norm(Z, axis=1)
        * np.linalg.norm(centroid)
    )

    distances[idx] = 1 - (
        numerator / np.maximum(denominator, 1e-12)
    )

details = test_labels.copy()
details["visual_typicality_distance"] = distances

# 3. Assign genre-specific test quintiles,
# as in our original retrospective analysis.
details["typicality_quintile"] = (
    details.groupby("genre")[
        "visual_typicality_distance"
    ]
    .transform(
        lambda s: pd.qcut(
            s.rank(method="first"),
            q=5,
            labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
        )
    )
    .astype(str)
)

# 4. Attach model correctness.
models = [
    "cnn_audio",
    "cnn_early_fusion",
    "mm_audio",
    "mm_fusion",
    "all_four"
]

for model in models:
    details[f"{model}_correct"] = (
        predictions[f"{model}_prediction"].to_numpy()
        == y_test
    ).astype(int)

# 5. Summarize accuracy by quintile.
correct_cols = [f"{m}_correct" for m in models]

summary = (
    details.groupby("typicality_quintile")[
        correct_cols
    ]
    .mean()
    .reindex(["Q1", "Q2", "Q3", "Q4", "Q5"])
)

summary.columns = models

summary["mm_vs_audio"] = (
    summary["mm_fusion"] - summary["cnn_audio"]
)

summary["mm_vs_cnn_fusion"] = (
    summary["mm_fusion"] - summary["cnn_early_fusion"]
)

print("=" * 85)
print("TEST ACCURACY BY VISUAL TYPICALITY (%)")
print("=" * 85)

print((summary * 100).round(2).to_string())

# 6. Artist-cluster bootstrap for Q5.
q5 = details[
    details["typicality_quintile"] == "Q5"
].copy()

artist_groups = {
    artist: group
    for artist, group in q5.groupby("msd_artist_id")
}

artists = list(artist_groups.keys())

rng = np.random.default_rng(42)
B = 2000

bootstrap_mm_vs_audio = []
bootstrap_mm_vs_cnn = []

for _ in range(B):

    sampled_artists = rng.choice(
        artists,
        size=len(artists),
        replace=True
    )

    sampled = pd.concat(
        [artist_groups[a] for a in sampled_artists],
        ignore_index=True
    )

    bootstrap_mm_vs_audio.append(
        sampled["mm_fusion_correct"].mean()
        - sampled["cnn_audio_correct"].mean()
    )

    bootstrap_mm_vs_cnn.append(
        sampled["mm_fusion_correct"].mean()
        - sampled["cnn_early_fusion_correct"].mean()
    )

ci_audio = np.quantile(
    bootstrap_mm_vs_audio, [0.025, 0.975]
)

ci_cnn = np.quantile(
    bootstrap_mm_vs_cnn, [0.025, 0.975]
)

print("\n" + "=" * 85)
print("Q5 — ARTIST-CLUSTER BOOTSTRAP")
print("=" * 85)

print("Q5 tracks:", len(q5))
print("Q5 artists:", len(artists))

print(
    "MM Fusion minus CNN Audio:",
    round(
        q5["mm_fusion_correct"].mean()
        - q5["cnn_audio_correct"].mean(),
        4
    ),
    "| 95% CI:",
    np.round(ci_audio, 4)
)

print(
    "MM Fusion minus CNN Early Fusion:",
    round(
        q5["mm_fusion_correct"].mean()
        - q5["cnn_early_fusion_correct"].mean(),
        4
    ),
    "| 95% CI:",
    np.round(ci_cnn, 4)
)

# 7. Save.
details.to_csv(
    new_dir / "representation_test_typicality_details.csv",
    index=False
)

summary.to_csv(
    new_dir / "representation_test_typicality_summary.csv"
)

print("\nSTEP 9 COMPLETE")


In [ ]:

# EXPERIMENT 2 — STEP 10
# Paired artist-cluster bootstrap for overall macro F1.

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

df = pd.read_csv(
    new_dir / "representation_final_test_predictions.csv"
)

y_true = df["genre"].to_numpy()
pred_mm = df["mm_fusion_prediction"].to_numpy()
pred_cnn = df["cnn_early_fusion_prediction"].to_numpy()
pred_audio = df["cnn_audio_prediction"].to_numpy()

classes = np.unique(y_true)

def macro_f1(y, pred):
    return f1_score(
        y, pred,
        labels=classes,
        average="macro",
        zero_division=0
    )

# Point estimates.
observed_mm_cnn = (
    macro_f1(y_true, pred_mm)
    - macro_f1(y_true, pred_cnn)
)

observed_mm_audio = (
    macro_f1(y_true, pred_mm)
    - macro_f1(y_true, pred_audio)
)

# Artist-cluster bootstrap:
# resample artists, retaining all their tracks.
artist_indices = [
    group.index.to_numpy()
    for _, group in df.groupby("msd_artist_id")
]

rng = np.random.default_rng(42)
B = 2000

diff_mm_cnn = np.empty(B)
diff_mm_audio = np.empty(B)

for b in range(B):

    sampled_groups = rng.integers(
        0, len(artist_indices),
        size=len(artist_indices)
    )

    idx = np.concatenate([
        artist_indices[i]
        for i in sampled_groups
    ])

    y = y_true[idx]

    mm_score = macro_f1(y, pred_mm[idx])

    diff_mm_cnn[b] = (
        mm_score - macro_f1(y, pred_cnn[idx])
    )

    diff_mm_audio[b] = (
        mm_score - macro_f1(y, pred_audio[idx])
    )

ci_mm_cnn = np.quantile(
    diff_mm_cnn, [0.025, 0.975]
)

ci_mm_audio = np.quantile(
    diff_mm_audio, [0.025, 0.975]
)

print("=" * 70)
print("OVERALL MACRO F1 — ARTIST-CLUSTER BOOTSTRAP")
print("=" * 70)

print(
    "MM Fusion - CNN Early Fusion:",
    f"{observed_mm_cnn:+.4f}",
    "| 95% CI:",
    np.round(ci_mm_cnn, 4)
)

print(
    "MM Fusion - CNN Audio:",
    f"{observed_mm_audio:+.4f}",
    "| 95% CI:",
    np.round(ci_mm_audio, 4)
)

results = pd.DataFrame([
    {
        "comparison": "MM Fusion - CNN Early Fusion",
        "difference": observed_mm_cnn,
        "ci_lower": ci_mm_cnn[0],
        "ci_upper": ci_mm_cnn[1]
    },
    {
        "comparison": "MM Fusion - CNN Audio",
        "difference": observed_mm_audio,
        "ci_lower": ci_mm_audio[0],
        "ci_upper": ci_mm_audio[1]
    }
])

results.to_csv(
    new_dir / "representation_macro_f1_bootstrap.csv",
    index=False
)

print("\nSTEP 10 COMPLETE")


neural network

In [ ]:

# ============================================================
# STAGE 6 — EXPERIMENT A
# STEP 1: LOAD AND VERIFY CNN EARLY-FUSION FEATURES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from sklearn.preprocessing import LabelEncoder

# Connect Google Drive.
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

nn_dir = (
    PROJECT / "finetuning_experiments"
    / "neural_network_experiments"
)

nn_dir.mkdir(parents=True, exist_ok=True)

# Load the existing artist-disjoint splits.
data = {}

for split in ["train", "val", "test"]:

    X_audio = np.load(
        saved_dir / f"X_{split}_audio.npy"
    )

    X_visual = np.load(
        saved_dir / f"X_{split}_visual.npy"
    )

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    # Concatenate audio and visual representations.
    X_fusion = np.concatenate(
        [X_audio, X_visual],
        axis=1
    ).astype(np.float32)

    assert len(X_fusion) == len(labels)
    assert np.isfinite(X_fusion).all()

    data[split] = {
        "X": X_fusion,
        "labels": labels
    }

# Encode genre names as integer classes.
encoder = LabelEncoder()

y_train = encoder.fit_transform(
    data["train"]["labels"]["genre"]
)

y_val = encoder.transform(
    data["val"]["labels"]["genre"]
)

y_test = encoder.transform(
    data["test"]["labels"]["genre"]
)

for split, y in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test)
]:
    data[split]["y"] = y

# Verify no artists appear in multiple splits.
artist_sets = {
    split: set(data[split]["labels"]["msd_artist_id"])
    for split in ["train", "val", "test"]
}

assert artist_sets["train"].isdisjoint(
    artist_sets["val"]
)

assert artist_sets["train"].isdisjoint(
    artist_sets["test"]
)

assert artist_sets["val"].isdisjoint(
    artist_sets["test"]
)

print("=" * 65)
print("NEURAL-NETWORK EXPERIMENT — DATA VERIFICATION")
print("=" * 65)

for split in ["train", "val", "test"]:

    X = data[split]["X"]
    y = data[split]["y"]

    print(
        f"{split.upper():5s} | "
        f"Tracks: {len(y):5d} | "
        f"Features: {X.shape[1]}"
    )

print("\nGenres:", list(encoder.classes_))

print("\nPyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

print("\nArtist-disjoint splits: VERIFIED")
print("STEP 1 COMPLETE")


In [ ]:

# FIX: Ensure genre labels are integer arrays

from sklearn.preprocessing import LabelEncoder
import numpy as np

encoder = LabelEncoder()

y_train = encoder.fit_transform(
    data["train"]["labels"]["genre"].astype(str)
).astype(np.int64)

y_val = encoder.transform(
    data["val"]["labels"]["genre"].astype(str)
).astype(np.int64)

y_test = encoder.transform(
    data["test"]["labels"]["genre"].astype(str)
).astype(np.int64)

print("Training labels:", y_train.dtype)
print("Validation labels:", y_val.dtype)
print("Test labels:", y_test.dtype)

print("Genres:", list(encoder.classes_))
print("STEP 1 FIX COMPLETE")


In [ ]:

import torch
import torch.nn as nn

class FusionMLP(nn.Module):

    def __init__(
        self,
        input_dim=4096,
        hidden1=256,
        hidden2=64,
        dropout1=0.30,
        dropout2=0.20,
        num_classes=6
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

print("FusionMLP is now defined!")


In [ ]:

# ============================================================
# STAGE 6 — EXPERIMENT A
# STEP 2: STANDARDIZATION + NEURAL NETWORK ARCHITECTURE
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# ------------------------------------------------------------
# 2. Standardize features
# ------------------------------------------------------------
# Fit the scaler ONLY on training data.
# Apply the same transformation to validation and test.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    data["train"]["X"]
).astype(np.float32)

X_val_scaled = scaler.transform(
    data["val"]["X"]
).astype(np.float32)

# We will transform the test set only when we reach
# the final exploratory evaluation.

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()

print("\nStandardization complete.")

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

# ------------------------------------------------------------
# 3. Convert to PyTorch datasets
# ------------------------------------------------------------

train_dataset = TensorDataset(
    torch.from_numpy(X_train_scaled),
    torch.tensor(y_train, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val_scaled),
    torch.tensor(y_val, dtype=torch.long)
)

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ------------------------------------------------------------
# 4. Define neural network
# ------------------------------------------------------------

class FusionMLP(nn.Module):

    def __init__(
        self,
        input_dim=4096,
        hidden1=256,
        hidden2=64,
        dropout1=0.30,
        dropout2=0.20,
        num_classes=6
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# ------------------------------------------------------------
# 5. Initialize model
# ------------------------------------------------------------

model = FusionMLP(
    input_dim=X_train_scaled.shape[1],
    num_classes=len(encoder.classes_)
).to(device)

print("\n" + "=" * 65)
print("NEURAL NETWORK ARCHITECTURE")
print("=" * 65)

print(model)

num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", f"{num_parameters:,}")

# ------------------------------------------------------------
# 6. Test forward pass
# ------------------------------------------------------------

sample_X, sample_y = next(iter(train_loader))

with torch.no_grad():

    sample_output = model(
        sample_X.to(device)
    )

print("\nInput batch shape:", tuple(sample_X.shape))
print("Output batch shape:", tuple(sample_output.shape))

assert sample_output.shape == (
    len(sample_X),
    len(encoder.classes_)
)

print("\nSTEP 2 COMPLETE")


In [ ]:

# ============================================================
# STAGE 6 — EXPERIMENT A
# STEP 3: TRAIN CNN EARLY-FUSION NEURAL NETWORK
# ============================================================

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 2. Initialize a fresh model
# ------------------------------------------------------------

model = FusionMLP(
    input_dim=X_train_scaled.shape[1],
    hidden1=256,
    hidden2=64,
    dropout1=0.30,
    dropout2=0.20,
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.01
)

# ------------------------------------------------------------
# 3. Training settings
# ------------------------------------------------------------

MAX_EPOCHS = 100
PATIENCE = 12
MIN_DELTA = 1e-4

best_val_f1 = -np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0

history = []

# ------------------------------------------------------------
# 4. Training loop
# ------------------------------------------------------------

for epoch in range(1, MAX_EPOCHS + 1):

    # TRAINING
    model.train()

    train_loss_sum = 0.0
    train_preds = []
    train_targets = []

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss_sum += (
            loss.item() * len(y_batch)
        )

        train_preds.extend(
            logits.argmax(dim=1).detach().cpu().numpy()
        )

        train_targets.extend(
            y_batch.cpu().numpy()
        )

    train_loss = (
        train_loss_sum / len(train_dataset)
    )

    train_f1 = f1_score(
        train_targets,
        train_preds,
        average="macro",
        zero_division=0
    )

    # VALIDATION
    model.eval()

    val_loss_sum = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            val_loss_sum += (
                loss.item() * len(y_batch)
            )

            val_preds.extend(
                logits.argmax(dim=1).cpu().numpy()
            )

            val_targets.extend(
                y_batch.cpu().numpy()
            )

    val_loss = (
        val_loss_sum / len(val_dataset)
    )

    val_f1 = f1_score(
        val_targets,
        val_preds,
        average="macro",
        zero_division=0
    )

    val_accuracy = accuracy_score(
        val_targets,
        val_preds
    )

    val_balanced_accuracy = balanced_accuracy_score(
        val_targets,
        val_preds
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_macro_f1": train_f1,
        "val_macro_f1": val_f1,
        "val_accuracy": val_accuracy,
        "val_balanced_accuracy": val_balanced_accuracy
    })

    # EARLY STOPPING
    if val_f1 > best_val_f1 + MIN_DELTA:

        best_val_f1 = val_f1
        best_epoch = epoch

        best_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break

# ------------------------------------------------------------
# 5. Restore best model
# ------------------------------------------------------------

model.load_state_dict(best_state)
model.eval()

# ------------------------------------------------------------
# 6. Save model, scaler, and training history
# ------------------------------------------------------------

nn_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "input_dim": X_train_scaled.shape[1],
        "num_classes": len(encoder.classes_),
        "hidden1": 256,
        "hidden2": 64,
        "dropout1": 0.30,
        "dropout2": 0.20,
        "seed": SEED
    },
    nn_dir / "cnn_fusion_mlp_best.pt"
)

joblib.dump(
    scaler,
    nn_dir / "cnn_fusion_mlp_scaler.joblib"
)

joblib.dump(
    encoder,
    nn_dir / "genre_label_encoder.joblib"
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    nn_dir / "cnn_fusion_mlp_training_history.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CNN FUSION NEURAL NETWORK — TRAINING COMPLETE")
print("=" * 70)

print("Best epoch:", best_epoch)

print(
    "Best validation macro F1:",
    round(best_val_f1, 6)
)

print(
    "Logistic regression baseline:",
    0.669152
)

print(
    "Difference:",
    round(best_val_f1 - 0.669152, 6)
)

print("\nSaved files in:", nn_dir)

print("\nSTEP 3 COMPLETE")


In [ ]:

# ============================================================
# STAGE 6 — EXPERIMENT B
# STEP 4: PREPARE MM AUDIO + MM VISUAL
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

mm_data = {}

for split in ["train", "val"]:

    X_audio = np.load(
        mm_dir / f"X_{split}_mm-audio.npy"
    )

    X_visual = np.load(
        mm_dir / f"X_{split}_mm-visual.npy"
    )

    mm_labels = pd.read_csv(
        mm_dir / f"labels_{split}.csv"
    )

    # Verify the same tracks and order as Experiment A.
    assert np.array_equal(
        mm_labels["msd_track_id"].to_numpy(),
        data[split]["labels"]["msd_track_id"].to_numpy()
    ), f"Track alignment mismatch in {split}"

    X = np.concatenate(
        [X_audio, X_visual],
        axis=1
    ).astype(np.float32)

    assert X.shape[1] == 400
    assert np.isfinite(X).all()

    y = encoder.transform(
        mm_labels["genre"].astype(str)
    ).astype(np.int64)

    mm_data[split] = {
        "X": X,
        "y": y
    }

# Standardize using training statistics only.
mm_scaler = StandardScaler()

X_mm_train_scaled = mm_scaler.fit_transform(
    mm_data["train"]["X"]
).astype(np.float32)

X_mm_val_scaled = mm_scaler.transform(
    mm_data["val"]["X"]
).astype(np.float32)

mm_train_dataset = TensorDataset(
    torch.from_numpy(X_mm_train_scaled),
    torch.from_numpy(mm_data["train"]["y"])
)

mm_val_dataset = TensorDataset(
    torch.from_numpy(X_mm_val_scaled),
    torch.from_numpy(mm_data["val"]["y"])
)

mm_train_loader = DataLoader(
    mm_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

mm_val_loader = DataLoader(
    mm_val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("=" * 65)
print("MM FUSION — NEURAL NETWORK DATA PREPARATION")
print("=" * 65)

print("Training:", X_mm_train_scaled.shape)
print("Validation:", X_mm_val_scaled.shape)
print("Classes:", list(encoder.classes_))
print("Device:", device)
print("Track alignment: VERIFIED")

print("\nSTEP 4 COMPLETE")


In [ ]:

# ============================================================
# STAGE 6 — EXPERIMENT B
# STEP 5: TRAIN MM FUSION NEURAL NETWORK
# ============================================================

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 2. Define a smaller neural network
# ------------------------------------------------------------

class MMFusionMLP(nn.Module):

    def __init__(
        self,
        input_dim=400,
        hidden1=128,
        hidden2=32,
        dropout1=0.40,
        dropout2=0.30,
        num_classes=6
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model_mm = MMFusionMLP(
    input_dim=X_mm_train_scaled.shape[1],
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_mm.parameters(),
    lr=0.0003,
    weight_decay=0.05
)

# ------------------------------------------------------------
# 3. Training settings
# ------------------------------------------------------------

MAX_EPOCHS = 100
PATIENCE = 12
MIN_DELTA = 1e-4

best_val_f1 = -np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0

history = []

# ------------------------------------------------------------
# 4. Training loop
# ------------------------------------------------------------

for epoch in range(1, MAX_EPOCHS + 1):

    model_mm.train()

    train_loss_sum = 0.0
    train_preds = []
    train_targets = []

    for X_batch, y_batch in mm_train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model_mm(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss_sum += (
            loss.item() * len(y_batch)
        )

        train_preds.extend(
            logits.argmax(dim=1).detach().cpu().numpy()
        )

        train_targets.extend(
            y_batch.cpu().numpy()
        )

    train_loss = (
        train_loss_sum / len(mm_train_dataset)
    )

    train_f1 = f1_score(
        train_targets,
        train_preds,
        average="macro",
        zero_division=0
    )

    # VALIDATION
    model_mm.eval()

    val_loss_sum = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in mm_val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model_mm(X_batch)
            loss = criterion(logits, y_batch)

            val_loss_sum += (
                loss.item() * len(y_batch)
            )

            val_preds.extend(
                logits.argmax(dim=1).cpu().numpy()
            )

            val_targets.extend(
                y_batch.cpu().numpy()
            )

    val_loss = (
        val_loss_sum / len(mm_val_dataset)
    )

    val_f1 = f1_score(
        val_targets,
        val_preds,
        average="macro",
        zero_division=0
    )

    val_accuracy = accuracy_score(
        val_targets,
        val_preds
    )

    val_balanced_accuracy = balanced_accuracy_score(
        val_targets,
        val_preds
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_macro_f1": train_f1,
        "val_macro_f1": val_f1,
        "val_accuracy": val_accuracy,
        "val_balanced_accuracy": val_balanced_accuracy
    })

    # EARLY STOPPING
    if val_f1 > best_val_f1 + MIN_DELTA:

        best_val_f1 = val_f1
        best_epoch = epoch

        best_state = copy.deepcopy(
            model_mm.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break

# ------------------------------------------------------------
# 5. Restore best checkpoint
# ------------------------------------------------------------

model_mm.load_state_dict(best_state)
model_mm.eval()

# ------------------------------------------------------------
# 6. Save model and history
# ------------------------------------------------------------

nn_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "input_dim": X_mm_train_scaled.shape[1],
        "num_classes": len(encoder.classes_),
        "hidden1": 128,
        "hidden2": 32,
        "dropout1": 0.40,
        "dropout2": 0.30,
        "seed": SEED
    },
    nn_dir / "mm_fusion_mlp_best.pt"
)

joblib.dump(
    mm_scaler,
    nn_dir / "mm_fusion_mlp_scaler.joblib"
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    nn_dir / "mm_fusion_mlp_training_history.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MM FUSION NEURAL NETWORK — TRAINING COMPLETE")
print("=" * 70)

print("Best epoch:", best_epoch)

print(
    "Best validation macro F1:",
    round(best_val_f1, 6)
)

print(
    "MM Fusion logistic regression baseline:",
    0.680940
)

print(
    "Difference:",
    round(best_val_f1 - 0.680940, 6)
)

print("\nSaved files in:", nn_dir)

print("\nSTEP 5 COMPLETE")


In [ ]:

# ============================================================
# STAGE 6 — STEP 6
# COMPARE NEURAL NETWORK TRAINING HISTORIES
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

cnn_history = pd.read_csv(
    nn_dir / "cnn_fusion_mlp_training_history.csv"
)

mm_history = pd.read_csv(
    nn_dir / "mm_fusion_mlp_training_history.csv"
)

experiments = [
    ("CNN Fusion Neural Network", cnn_history, "cnn"),
    ("MM Fusion Neural Network", mm_history, "mm")
]

for title, history, filename in experiments:

    best_idx = history["val_macro_f1"].idxmax()
    best_epoch = int(history.loc[best_idx, "epoch"])
    best_f1 = history.loc[best_idx, "val_macro_f1"]

    # Figure 1: Macro F1
    plt.figure(figsize=(9, 5))

    plt.plot(
        history["epoch"],
        history["train_macro_f1"],
        label="Training macro F1"
    )

    plt.plot(
        history["epoch"],
        history["val_macro_f1"],
        label="Validation macro F1"
    )

    plt.axvline(
        best_epoch,
        linestyle="--",
        label=f"Best epoch: {best_epoch}"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(title + " — Learning Curve")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        nn_dir / f"{filename}_f1_learning_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    # Figure 2: Loss
    plt.figure(figsize=(9, 5))

    plt.plot(
        history["epoch"],
        history["train_loss"],
        label="Training loss"
    )

    plt.plot(
        history["epoch"],
        history["val_loss"],
        label="Validation loss"
    )

    plt.axvline(
        best_epoch,
        linestyle="--",
        label=f"Best epoch: {best_epoch}"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title + " — Loss Curve")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        nn_dir / f"{filename}_loss_learning_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"{title}: best epoch = {best_epoch}, "
        f"validation macro F1 = {best_f1:.6f}"
    )

print("\nSTEP 6 COMPLETE — FOUR FIGURES SAVED")


In [ ]:

# ============================================================
# STAGE 6 — STEP 7
# VALIDATE SAVED NEURAL-NETWORK CHECKPOINTS
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. Load saved checkpoints
# ------------------------------------------------------------

cnn_checkpoint = torch.load(
    nn_dir / "cnn_fusion_mlp_best.pt",
    map_location=device,
    weights_only=True
)

mm_checkpoint = torch.load(
    nn_dir / "mm_fusion_mlp_best.pt",
    map_location=device,
    weights_only=True
)

# ------------------------------------------------------------
# 2. Reconstruct the selected architectures
# ------------------------------------------------------------

cnn_model = FusionMLP(
    input_dim=cnn_checkpoint["input_dim"],
    hidden1=cnn_checkpoint["hidden1"],
    hidden2=cnn_checkpoint["hidden2"],
    dropout1=cnn_checkpoint["dropout1"],
    dropout2=cnn_checkpoint["dropout2"],
    num_classes=cnn_checkpoint["num_classes"]
).to(device)

cnn_model.load_state_dict(
    cnn_checkpoint["model_state_dict"]
)

mm_model = MMFusionMLP(
    input_dim=mm_checkpoint["input_dim"],
    hidden1=mm_checkpoint["hidden1"],
    hidden2=mm_checkpoint["hidden2"],
    dropout1=mm_checkpoint["dropout1"],
    dropout2=mm_checkpoint["dropout2"],
    num_classes=mm_checkpoint["num_classes"]
).to(device)

mm_model.load_state_dict(
    mm_checkpoint["model_state_dict"]
)

cnn_model.eval()
mm_model.eval()

# ------------------------------------------------------------
# 3. Generate validation predictions
# ------------------------------------------------------------

def predict_nn(model, X):
    model.eval()

    with torch.no_grad():
        tensor = torch.from_numpy(
            X.astype(np.float32)
        ).to(device)

        logits = model(tensor)

        predictions = (
            logits.argmax(dim=1).cpu().numpy()
        )

    return predictions

cnn_pred = predict_nn(
    cnn_model, X_val_scaled
)

mm_pred = predict_nn(
    mm_model, X_mm_val_scaled
)

# Verify that both datasets use the same labels.
assert np.array_equal(
    y_val, mm_data["val"]["y"]
)

# ------------------------------------------------------------
# 4. Compare metrics
# ------------------------------------------------------------

rows = []

for name, pred, X in [
    ("CNN Fusion NN", cnn_pred, X_val_scaled),
    ("MM Fusion NN", mm_pred, X_mm_val_scaled)
]:

    rows.append({
        "model": name,
        "dimensions": X.shape[1],
        "accuracy": accuracy_score(y_val, pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, pred
        ),
        "macro_f1": f1_score(
            y_val, pred,
            average="macro"
        )
    })

results = pd.DataFrame(rows)

print("=" * 75)
print("NEURAL NETWORK — VALIDATION COMPARISON")
print("=" * 75)

print(results.to_string(index=False))

# ------------------------------------------------------------
# 5. Compare with logistic regression
# ------------------------------------------------------------

print("\nLOGISTIC REGRESSION BASELINES")

print("CNN Fusion LR:", 0.669152)
print("MM Fusion LR:", 0.680940)

print("\nNEURAL NETWORK IMPROVEMENTS")

print(
    "CNN NN minus CNN LR:",
    round(
        results.loc[
            results["model"] == "CNN Fusion NN",
            "macro_f1"
        ].iloc[0] - 0.669152,
        6
    )
)

print(
    "MM NN minus MM LR:",
    round(
        results.loc[
            results["model"] == "MM Fusion NN",
            "macro_f1"
        ].iloc[0] - 0.680940,
        6
    )
)

# ------------------------------------------------------------
# 6. Save predictions and metrics
# ------------------------------------------------------------

predictions_df = data["val"]["labels"].copy()

predictions_df["cnn_nn_prediction"] = (
    encoder.inverse_transform(cnn_pred)
)

predictions_df["mm_nn_prediction"] = (
    encoder.inverse_transform(mm_pred)
)

predictions_df["cnn_nn_correct"] = (
    cnn_pred == y_val
).astype(int)

predictions_df["mm_nn_correct"] = (
    mm_pred == y_val
).astype(int)

results.to_csv(
    nn_dir / "neural_network_validation_metrics.csv",
    index=False
)

predictions_df.to_csv(
    nn_dir / "neural_network_validation_predictions.csv",
    index=False
)

print("\nSTEP 7 COMPLETE")
print("Validation metrics and predictions saved.")


In [ ]:

# ============================================================
# STAGE 6 — STEP 8
# FINAL EXPLORATORY TEST EVALUATION: NEURAL NETWORKS
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Load test labels and features
# ------------------------------------------------------------

test_labels = data["test"]["labels"].copy()

X_cnn_test = data["test"]["X"].astype(np.float32)

X_mm_audio_test = np.load(
    mm_dir / "X_test_mm-audio.npy"
)

X_mm_visual_test = np.load(
    mm_dir / "X_test_mm-visual.npy"
)

mm_test_labels = pd.read_csv(
    mm_dir / "labels_test.csv"
)

# Verify exact track alignment.
assert np.array_equal(
    test_labels["msd_track_id"].to_numpy(),
    mm_test_labels["msd_track_id"].to_numpy()
)

assert np.array_equal(
    test_labels["genre"].to_numpy(),
    mm_test_labels["genre"].to_numpy()
)

X_mm_test = np.concatenate(
    [X_mm_audio_test, X_mm_visual_test],
    axis=1
).astype(np.float32)

# ------------------------------------------------------------
# 2. Load the original training-fitted scalers
# ------------------------------------------------------------

cnn_scaler = joblib.load(
    nn_dir / "cnn_fusion_mlp_scaler.joblib"
)

mm_scaler_saved = joblib.load(
    nn_dir / "mm_fusion_mlp_scaler.joblib"
)

X_cnn_test_scaled = cnn_scaler.transform(
    X_cnn_test
).astype(np.float32)

X_mm_test_scaled = mm_scaler_saved.transform(
    X_mm_test
).astype(np.float32)

assert np.isfinite(X_cnn_test_scaled).all()
assert np.isfinite(X_mm_test_scaled).all()

# ------------------------------------------------------------
# 3. Generate predictions using saved best models
# ------------------------------------------------------------

def predict_test(model, X, batch_size=256):

    model.eval()
    predictions = []

    with torch.no_grad():

        for start in range(0, len(X), batch_size):

            batch = torch.from_numpy(
                X[start:start + batch_size]
            ).to(device)

            logits = model(batch)

            predictions.append(
                logits.argmax(dim=1).cpu().numpy()
            )

    return np.concatenate(predictions)

cnn_test_pred = predict_test(
    cnn_model,
    X_cnn_test_scaled
)

mm_test_pred = predict_test(
    mm_model,
    X_mm_test_scaled
)

y_test_encoded = encoder.transform(
    test_labels["genre"].astype(str)
)

# ------------------------------------------------------------
# 4. Calculate test metrics
# ------------------------------------------------------------

rows = []

for name, pred, dimensions in [
    ("CNN Fusion NN", cnn_test_pred, 4096),
    ("MM Fusion NN", mm_test_pred, 400)
]:

    rows.append({
        "model": name,
        "dimensions": dimensions,
        "accuracy": accuracy_score(
            y_test_encoded, pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test_encoded, pred
        ),
        "macro_f1": f1_score(
            y_test_encoded, pred,
            average="macro"
        )
    })

nn_results = pd.DataFrame(rows)

print("=" * 75)
print("NEURAL NETWORKS — EXPLORATORY TEST RESULTS")
print("=" * 75)

print(nn_results.to_string(index=False))

# ------------------------------------------------------------
# 5. Compare against previous logistic regression results
# ------------------------------------------------------------

lr_results = pd.read_csv(
    mm_dir / "representation_final_test_metrics.csv"
)

comparison = pd.concat(
    [
        lr_results[
            lr_results["model"].isin([
                "CNN Audio",
                "CNN Early Fusion",
                "MM Audio",
                "MM Fusion",
                "All Four"
            ])
        ],
        nn_results
    ],
    ignore_index=True
)

comparison = comparison.sort_values(
    "macro_f1",
    ascending=False
)

print("\n" + "=" * 75)
print("ALL MODELS — EXPLORATORY TEST COMPARISON")
print("=" * 75)

print(comparison.to_string(index=False))

# ------------------------------------------------------------
# 6. Save test predictions
# ------------------------------------------------------------

predictions_df = test_labels.copy()

predictions_df["cnn_nn_prediction"] = (
    encoder.inverse_transform(cnn_test_pred)
)

predictions_df["mm_nn_prediction"] = (
    encoder.inverse_transform(mm_test_pred)
)

predictions_df["cnn_nn_correct"] = (
    cnn_test_pred == y_test_encoded
).astype(int)

predictions_df["mm_nn_correct"] = (
    mm_test_pred == y_test_encoded
).astype(int)

nn_results.to_csv(
    nn_dir / "neural_network_test_metrics.csv",
    index=False
)

comparison.to_csv(
    nn_dir / "all_models_test_comparison.csv",
    index=False
)

predictions_df.to_csv(
    nn_dir / "neural_network_test_predictions.csv",
    index=False
)

print("\nSTEP 8 COMPLETE")
print("Test metrics and predictions saved.")


In [ ]:

# ============================================================
# STAGE 6 — STEP 9B
# CORRECTED NEURAL NETWORK VISUAL-TYPICALITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Load original visual-typicality assignments
# ------------------------------------------------------------

typ = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

assert typ["msd_track_id"].is_unique

# Original quintiles are integers 1, 2, 3, 4, 5.
assert set(typ["typicality_quintile"]) == {
    1, 2, 3, 4, 5
}

typ = typ.copy()

typ["typicality_quintile"] = (
    "Q" + typ["typicality_quintile"].astype(str)
)

# ------------------------------------------------------------
# 2. Load saved neural-network test predictions
# ------------------------------------------------------------

nn_predictions = pd.read_csv(
    nn_dir / "neural_network_test_predictions.csv"
)

assert nn_predictions["msd_track_id"].is_unique

analysis = nn_predictions.merge(
    typ[["msd_track_id", "typicality_quintile"]],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert len(analysis) == 1792
assert analysis["typicality_quintile"].notna().all()

# ------------------------------------------------------------
# 3. Load logistic-regression predictions
# ------------------------------------------------------------

lr_predictions = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

lr_columns = {
    "cnn_audio_correct": "CNN Audio LR",
    "cnn_early_fusion_correct": "CNN Fusion LR",
    "mm_fusion_correct": "MM Fusion LR"
}

assert lr_predictions["msd_track_id"].is_unique

assert all(
    col in lr_predictions.columns
    for col in lr_columns
)

analysis = analysis.merge(
    lr_predictions[
        ["msd_track_id"] + list(lr_columns)
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert analysis[list(lr_columns)].notna().all().all()

# ------------------------------------------------------------
# 4. Calculate accuracy by quintile
# ------------------------------------------------------------

correct_columns = {
    **lr_columns,
    "cnn_nn_correct": "CNN Fusion NN",
    "mm_nn_correct": "MM Fusion NN"
}

summary = (
    analysis.groupby("typicality_quintile")[
        list(correct_columns)
    ]
    .mean()
    .mul(100)
    .rename(columns=correct_columns)
    .reindex(["Q1", "Q2", "Q3", "Q4", "Q5"])
)

print("=" * 85)
print("TEST ACCURACY BY VISUAL TYPICALITY (%)")
print("=" * 85)

print(summary.round(2).to_string())

# ------------------------------------------------------------
# 5. Compare Q5 performance
# ------------------------------------------------------------

q5 = summary.loc["Q5"]

print("\n" + "=" * 85)
print("Q5 — MOST VISUALLY ATYPICAL COVERS")
print("=" * 85)

comparisons = {
    "CNN Fusion NN minus CNN Audio LR":
        q5["CNN Fusion NN"] - q5["CNN Audio LR"],

    "MM Fusion NN minus CNN Audio LR":
        q5["MM Fusion NN"] - q5["CNN Audio LR"],

    "CNN Fusion NN minus CNN Fusion LR":
        q5["CNN Fusion NN"] - q5["CNN Fusion LR"],

    "MM Fusion NN minus MM Fusion LR":
        q5["MM Fusion NN"] - q5["MM Fusion LR"]
}

for name, difference in comparisons.items():
    print(f"{name}: {difference:+.2f} percentage points")

# ------------------------------------------------------------
# 6. Create research figure
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

for column in summary.columns:

    plt.plot(
        summary.index,
        summary[column],
        marker="o",
        linewidth=2,
        label=column
    )

plt.xlabel(
    "Visual Typicality Quintile "
    "(Q1 = Most Typical, Q5 = Most Atypical)"
)

plt.ylabel("Test Accuracy (%)")

plt.title(
    "Music Genre Classification Accuracy "
    "by Visual Typicality"
)

plt.legend(
    loc="best",
    fontsize=9
)

plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    nn_dir / "neural_network_typicality_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

summary.to_csv(
    nn_dir / "neural_network_typicality_summary.csv"
)

analysis.to_csv(
    nn_dir / "neural_network_typicality_details.csv",
    index=False
)

print("\n" + "=" * 85)
print("STEP 9 COMPLETE")
print("=" * 85)

print("Tracks analyzed:", len(analysis))

print("\nTracks per quintile:")
print(
    analysis["typicality_quintile"]
    .value_counts()
    .sort_index()
)

print("\nFigure and CSV files saved to:", nn_dir)


In [ ]:

# ============================================================
# STAGE 6 — STEP 10
# ARTIST-CLUSTER BOOTSTRAP: NN VS LOGISTIC REGRESSION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# Reload saved Step 9 results.
analysis = pd.read_csv(
    nn_dir / "neural_network_typicality_details.csv"
)

assert analysis["msd_artist_id"].notna().all()
assert analysis["msd_track_id"].is_unique

# Retrieve the corresponding model predictions.
lr_preds = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

nn_preds = pd.read_csv(
    nn_dir / "neural_network_test_predictions.csv"
)

prediction_cols = [
    "cnn_early_fusion_prediction",
    "mm_fusion_prediction"
]

# Existing LR predictions use these column names.
# Check before merging to avoid accidental mismatches.
print("LR columns:", lr_preds.columns.tolist())

# We will use the CNN and MM predictions already
# stored in the previous experiment.
lr_source = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

cnn_lr = lr_source[
    ["msd_track_id", "early_fusion_prediction"]
].rename(columns={
    "early_fusion_prediction": "cnn_lr_pred"
})

mm_lr = lr_preds[
    ["msd_track_id", "mm_fusion_prediction"]
].rename(columns={
    "mm_fusion_prediction": "mm_lr_pred"
})

df = analysis[
    [
        "msd_track_id",
        "msd_artist_id",
        "genre",
        "typicality_quintile",
        "cnn_nn_prediction",
        "mm_nn_prediction"
    ]
].copy()

df = df.merge(
    cnn_lr,
    on="msd_track_id",
    validate="one_to_one"
)

df = df.merge(
    mm_lr,
    on="msd_track_id",
    validate="one_to_one"
)

assert len(df) == 1792
assert df.notna().all().all()

# ------------------------------------------------------------
# Bootstrap setup
# ------------------------------------------------------------

SEED = 42
N_BOOT = 2000
rng = np.random.default_rng(SEED)

artists = df["msd_artist_id"].unique()

artist_indices = {
    artist: np.flatnonzero(
        df["msd_artist_id"].to_numpy() == artist
    )
    for artist in artists
}

y = df["genre"].to_numpy()

preds = {
    "CNN LR": df["cnn_lr_pred"].to_numpy(),
    "CNN NN": df["cnn_nn_prediction"].to_numpy(),
    "MM LR": df["mm_lr_pred"].to_numpy(),
    "MM NN": df["mm_nn_prediction"].to_numpy()
}

q5_mask = (
    df["typicality_quintile"].to_numpy() == "Q5"
)

comparisons = [
    ("CNN NN - CNN LR", "CNN NN", "CNN LR"),
    ("MM NN - MM LR", "MM NN", "MM LR")
]

def macro_f1_diff(indices, a, b):
    return (
        f1_score(
            y[indices],
            preds[a][indices],
            average="macro",
            labels=np.unique(y),
            zero_division=0
        )
        -
        f1_score(
            y[indices],
            preds[b][indices],
            average="macro",
            labels=np.unique(y),
            zero_division=0
        )
    )

def accuracy_diff(indices, a, b):
    return (
        np.mean(preds[a][indices] == y[indices])
        -
        np.mean(preds[b][indices] == y[indices])
    )

results = []
bootstrap_samples = {}

for label, a, b in comparisons:

    overall_diffs = []
    q5_diffs = []

    observed_overall = macro_f1_diff(
        np.arange(len(df)), a, b
    )

    observed_q5 = accuracy_diff(
        np.flatnonzero(q5_mask), a, b
    )

    for _ in range(N_BOOT):

        sampled_artists = rng.choice(
            artists,
            size=len(artists),
            replace=True
        )

        indices = np.concatenate([
            artist_indices[artist]
            for artist in sampled_artists
        ])

        overall_diffs.append(
            macro_f1_diff(indices, a, b)
        )

        q5_indices = indices[q5_mask[indices]]

        if len(q5_indices) > 0:
            q5_diffs.append(
                accuracy_diff(q5_indices, a, b)
            )

    for metric, observed, samples in [
        ("Overall macro F1", observed_overall, overall_diffs),
        ("Q5 accuracy", observed_q5, q5_diffs)
    ]:

        low, high = np.percentile(
            samples, [2.5, 97.5]
        )

        results.append({
            "comparison": label,
            "metric": metric,
            "difference": observed,
            "ci_lower": low,
            "ci_upper": high
        })

        bootstrap_samples[(label, metric)] = samples

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

bootstrap_results = pd.DataFrame(results)

print("\n" + "=" * 85)
print("NEURAL NETWORK VS LOGISTIC REGRESSION")
print("ARTIST-CLUSTER BOOTSTRAP — 95% CONFIDENCE INTERVALS")
print("=" * 85)

print(
    bootstrap_results.to_string(
        index=False,
        float_format=lambda x: f"{x:+.4f}"
    )
)

bootstrap_results.to_csv(
    nn_dir / "neural_network_bootstrap_results.csv",
    index=False
)

print("\nSTEP 10 COMPLETE")
print("Bootstrap results saved.")


In [ ]:

# ============================================================
# STAGE 6 — STEP 11
# CONFUSION MATRICES AND PER-GENRE PERFORMANCE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Load saved test predictions
# ------------------------------------------------------------

lr = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

nn = pd.read_csv(
    nn_dir / "neural_network_test_predictions.csv"
)

assert lr["msd_track_id"].is_unique
assert nn["msd_track_id"].is_unique

df = lr[
    [
        "msd_track_id",
        "genre",
        "cnn_audio_prediction",
        "cnn_early_fusion_prediction",
        "mm_fusion_prediction"
    ]
].merge(
    nn[["msd_track_id", "genre", "mm_nn_prediction"]],
    on="msd_track_id",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_nn")
)

assert len(df) == 1792
assert (df["genre"] == df["genre_nn"]).all()

genres = list(encoder.classes_)
y_true = df["genre"].to_numpy()

models = {
    "CNN Audio LR": "cnn_audio_prediction",
    "CNN Fusion LR": "cnn_early_fusion_prediction",
    "MM Fusion LR": "mm_fusion_prediction",
    "MM Fusion NN": "mm_nn_prediction"
}

# ------------------------------------------------------------
# 2. Confusion matrices
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 11),
    constrained_layout=True
)

for ax, (name, col) in zip(axes.flat, models.items()):

    cm = confusion_matrix(
        y_true,
        df[col],
        labels=genres,
        normalize="true"
    ) * 100

    im = ax.imshow(
        cm,
        cmap="Blues",
        vmin=0,
        vmax=100
    )

    ax.set_title(name, fontsize=12)
    ax.set_xticks(range(len(genres)))
    ax.set_yticks(range(len(genres)))
    ax.set_xticklabels(genres, rotation=45, ha="right")
    ax.set_yticklabels(genres)

    ax.set_xlabel("Predicted genre")
    ax.set_ylabel("True genre")

    for i in range(len(genres)):
        for j in range(len(genres)):
            ax.text(
                j, i,
                f"{cm[i, j]:.0f}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if cm[i, j] > 50 else "black"
            )

fig.colorbar(
    im,
    ax=axes,
    shrink=0.65,
    label="Percentage of true genre (%)"
)

fig.suptitle(
    "Genre Classification — Normalized Confusion Matrices",
    fontsize=15
)

fig.savefig(
    nn_dir / "final_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Per-genre precision, recall and F1
# ------------------------------------------------------------

rows = []

for name, col in models.items():

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            df[col],
            labels=genres,
            zero_division=0
        )
    )

    for i, genre in enumerate(genres):
        rows.append({
            "model": name,
            "genre": genre,
            "precision": precision[i],
            "recall": recall[i],
            "f1": f1[i],
            "support": support[i]
        })

genre_results = pd.DataFrame(rows)

f1_table = genre_results.pivot(
    index="genre",
    columns="model",
    values="f1"
).reindex(
    index=genres,
    columns=list(models)
)

print("=" * 85)
print("PER-GENRE TEST F1")
print("=" * 85)
print(f1_table.round(4).to_string())

# ------------------------------------------------------------
# 4. Per-genre F1 comparison figure
# ------------------------------------------------------------

ax = f1_table.plot(
    kind="bar",
    figsize=(11, 6),
    width=0.8
)

ax.set_title("Per-Genre F1 by Classification Model")
ax.set_xlabel("Music genre")
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1)
ax.legend(title="Model", fontsize=9)

plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    nn_dir / "final_per_genre_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Identify where multimodal fusion helps
# ------------------------------------------------------------

genre_gains = pd.DataFrame({
    "genre": genres,
    "CNN_fusion_gain": (
        f1_table["CNN Fusion LR"]
        - f1_table["CNN Audio LR"]
    ).to_numpy(),
    "MM_fusion_gain": (
        f1_table["MM Fusion LR"]
        - f1_table["CNN Audio LR"]
    ).to_numpy(),
    "MM_NN_gain": (
        f1_table["MM Fusion NN"]
        - f1_table["CNN Audio LR"]
    ).to_numpy()
})

print("\n" + "=" * 85)
print("PER-GENRE F1 GAINS RELATIVE TO CNN AUDIO LR")
print("=" * 85)
print(genre_gains.round(4).to_string(index=False))

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

genre_results.to_csv(
    nn_dir / "final_per_genre_metrics.csv",
    index=False
)

genre_gains.to_csv(
    nn_dir / "final_per_genre_f1_gains.csv",
    index=False
)

print("\nSTEP 11 COMPLETE")
print("Two figures and per-genre tables saved.")


In [ ]:

# ============================================================
# STAGE 6 — STEP 12
# MAIN RESEARCH FIGURE
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load saved experimental results
gains = pd.read_csv(
    nn_dir / "final_per_genre_f1_gains.csv"
)

typicality = pd.read_csv(
    nn_dir / "neural_network_typicality_summary.csv",
    index_col=0
)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 5.5),
    constrained_layout=True
)

# PANEL A — Per-genre F1 improvements
ax = axes[0]

x = np.arange(len(gains))
width = 0.35

ax.bar(
    x - width / 2,
    gains["CNN_fusion_gain"],
    width,
    label="CNN Fusion LR"
)

ax.bar(
    x + width / 2,
    gains["MM_fusion_gain"],
    width,
    label="MM Fusion LR"
)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(gains["genre"])
ax.set_ylabel("F1 gain over CNN Audio LR")
ax.set_title("A. Benefit of multimodal fusion by genre")
ax.legend()
ax.grid(axis="y", alpha=0.2)

# PANEL B — Visual typicality
ax = axes[1]

for name in [
    "CNN Audio LR",
    "CNN Fusion LR",
    "MM Fusion LR",
    "MM Fusion NN"
]:
    ax.plot(
        typicality.index,
        typicality[name],
        marker="o",
        linewidth=2,
        label=name
    )

ax.set_xlabel("Visual typicality (Q1 = typical, Q5 = atypical)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("B. Accuracy across visual-typicality groups")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

fig.suptitle(
    "When Does Album-Cover Information Help Genre Classification?",
    fontsize=15,
    fontweight="bold"
)

fig.savefig(
    nn_dir / "main_research_findings.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    nn_dir / "main_research_findings.pdf",
    bbox_inches="tight"
)

plt.show()

print("STEP 12 COMPLETE")
print("Main research figure saved as PNG and PDF.")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

new_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

# 1. Load our existing split metadata
frames = []

for split in ["train", "val", "test"]:
    path = saved_dir / f"labels_{split}.csv"
    df = pd.read_csv(path)
    df["our_split"] = split
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

print("=" * 75)
print("PHASE 1 — PRETRAINED FEATURE AUDIT")
print("=" * 75)

print("\nTotal tracks:", len(data))
print("Unique tracks:", data["msd_track_id"].nunique())
print("Unique artists:", data["msd_artist_id"].nunique())

assert len(data) == 11870
assert data["msd_track_id"].is_unique

# 2. Check our artist-disjoint splits
artist_sets = {
    split: set(
        data.loc[
            data["our_split"] == split,
            "msd_artist_id"
        ]
    )
    for split in ["train", "val", "test"]
}

for a, b in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]:
    overlap = artist_sets[a] & artist_sets[b]
    print(f"\nArtist overlap {a} / {b}: {len(overlap)}")
    assert len(overlap) == 0

# 3. Inspect original split information
print("\nAvailable columns:")
print(data.columns.tolist())

if "original_split" not in data.columns:
    raise ValueError(
        "Original split information is missing. "
        "We need to recover it from the MSD-I archive."
    )

print("\nOriginal split values:")
print(data["original_split"].value_counts(dropna=False))

# 4. Compare original vs our new splits
comparison = pd.crosstab(
    data["original_split"],
    data["our_split"]
)

print("\n" + "=" * 75)
print("ORIGINAL SPLIT VS OUR SPLIT")
print("=" * 75)
print(comparison.to_string())

# 5. Determine which tracks changed partitions
def normalize_split(value):
    value = str(value).strip().lower()
    aliases = {
        "training": "train",
        "validation": "val",
        "valid": "val",
        "dev": "val",
        "testing": "test"
    }
    return aliases.get(value, value)

data["original_normalized"] = (
    data["original_split"].map(normalize_split)
)

data["changed_split"] = (
    data["original_normalized"] != data["our_split"]
)

print("\nTracks assigned to different partitions:",
      int(data["changed_split"].sum()))

print("\nOriginal training tracks in our test set:",
      int((
          (data["original_normalized"] == "train")
          & (data["our_split"] == "test")
      ).sum()))

print("\nOriginal validation tracks in our test set:",
      int((
          (data["original_normalized"] == "val")
          & (data["our_split"] == "test")
      ).sum()))

# 6. Save audit
audit_dir = PROJECT / "finetuning_experiments" / "methodology_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

data.to_csv(
    audit_dir / "original_vs_new_split_audit.csv",
    index=False
)

comparison.to_csv(
    audit_dir / "split_overlap_matrix.csv"
)

print("\nSTEP 1A COMPLETE")
print("Audit saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1B
# INSPECT ORIGINAL ORAMAS ARCHIVE SPLIT FILES
# ============================================================

from pathlib import Path
import tarfile
import pandas as pd
import io

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

archive_path = (
    PROJECT / "audio_features" / "msdi-data.tar.gz"
)

assert archive_path.exists(), (
    f"Archive not found: {archive_path}"
)

print("=" * 75)
print("PHASE 1 — ORIGINAL ARCHIVE VERIFICATION")
print("=" * 75)

print("\nArchive:", archive_path.name)

original_tables = {}

with tarfile.open(archive_path, "r:gz") as tar:

    members = {
        m.name: m
        for m in tar.getmembers()
        if m.isfile()
    }

    for split in ["train", "val", "test"]:

        suffix = f"items_index_{split}_MSD-I.tsv"

        matches = [
            name for name in members
            if name.endswith(suffix)
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected one archive file for {split}, "
                f"found {matches}"
            )

        member = members[matches[0]]

        raw = tar.extractfile(member).read()

        # Read without assuming whether the file has a header.
        table = pd.read_csv(
            io.BytesIO(raw),
            sep="\t",
            header=None,
            dtype=str
        )

        original_tables[split] = table

        print(f"\n{split.upper()} ORIGINAL INDEX")
        print("Archive path:", matches[0])
        print("Shape:", table.shape)
        print("First five rows:")
        print(table.head().to_string(index=False, header=False))

print("\n" + "=" * 75)
print("OUR SAVED METADATA")
print("=" * 75)

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    print(f"\n{split.upper()}")
    print("Tracks:", len(labels))
    print("First five IDs:")
    print(
        labels["msd_track_id"]
        .head()
        .to_list()
    )

print("\nSTEP 1B INSPECTION COMPLETE")
print("Original index tables loaded for comparison.")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1C
# EXACT VERIFICATION AGAINST ORIGINAL ORAMAS SPLITS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Build original archive track-to-split mapping
# ------------------------------------------------------------

original_rows = []

for split in ["train", "val", "test"]:

    # original_tables was loaded in Step 1B.
    ids = (
        original_tables[split]
        .iloc[:, 0]
        .astype(str)
        .str.strip()
    )

    for position, track_id in enumerate(ids):

        original_rows.append({
            "msd_track_id": track_id,
            "archive_split": split,
            "archive_position": position
        })

original_df = pd.DataFrame(original_rows)

print("=" * 75)
print("ORIGINAL ARCHIVE — TRACK VERIFICATION")
print("=" * 75)

print("Total archive rows:", len(original_df))
print(
    "Unique archive track IDs:",
    original_df["msd_track_id"].nunique()
)

duplicate_ids = original_df[
    original_df.duplicated(
        "msd_track_id",
        keep=False
    )
]

print(
    "Duplicate archive entries:",
    len(duplicate_ids)
)

assert duplicate_ids.empty, (
    "Some track IDs appear more than once "
    "in the original archive."
)

# ------------------------------------------------------------
# 2. Load our saved labels
# ------------------------------------------------------------

frames = []

for split in ["train", "val", "test"]:

    df = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    df["our_split"] = split

    frames.append(df)

our_df = pd.concat(
    frames,
    ignore_index=True
)

assert our_df["msd_track_id"].is_unique

# ------------------------------------------------------------
# 3. Match against original archive
# ------------------------------------------------------------

verified = our_df.merge(
    original_df,
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

missing = verified[
    verified["archive_split"].isna()
]

print("\nTracks missing from original archive:",
      len(missing))

assert missing.empty, (
    "Some of our tracks were not found "
    "in the original archive."
)

# ------------------------------------------------------------
# 4. Compare split assignments
# ------------------------------------------------------------

verified["split_match"] = (
    verified["our_split"]
    == verified["archive_split"]
)

mismatches = verified[
    ~verified["split_match"]
]

print(
    "Tracks assigned to a different split:",
    len(mismatches)
)

print("\nArchive split versus our split:")

comparison = pd.crosstab(
    verified["archive_split"],
    verified["our_split"]
)

print(comparison.to_string())

assert mismatches.empty, (
    "Some tracks have different original "
    "and current split assignments."
)

# ------------------------------------------------------------
# 5. Verify relative track ordering
# ------------------------------------------------------------

print("\nTRACK ORDER VERIFICATION")

order_results = []

for split in ["train", "val", "test"]:

    subset = verified[
        verified["our_split"] == split
    ].copy()

    # The saved CSV's order should follow
    # the original archive's relative order.
    positions = subset["archive_position"].to_numpy()

    order_matches = bool(
        np.all(np.diff(positions) > 0)
    )

    print(
        f"{split.upper()}: "
        f"{len(subset)} tracks | "
        f"Archive order preserved: {order_matches}"
    )

    order_results.append({
        "split": split,
        "tracks": len(subset),
        "archive_order_preserved": order_matches
    })

# ------------------------------------------------------------
# 6. Save audit files
# ------------------------------------------------------------

verified.to_csv(
    audit_dir / "exact_archive_split_verification.csv",
    index=False
)

comparison.to_csv(
    audit_dir / "exact_archive_split_matrix.csv"
)

pd.DataFrame(order_results).to_csv(
    audit_dir / "archive_order_verification.csv",
    index=False
)

print("\n" + "=" * 75)
print("STEP 1C COMPLETE")
print("=" * 75)

print("Tracks checked:", len(verified))
print("Missing tracks:", len(missing))
print("Split mismatches:", len(mismatches))
print("Results saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1D
# VERIFY FEATURE ROW ALIGNMENT AND ARRAY SHAPES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

print("=" * 75)
print("PHASE 1 — FEATURE ALIGNMENT AUDIT")
print("=" * 75)

results = []

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    n = len(labels)

    print(f"\n{'=' * 60}")
    print(f"{split.upper()} — {n} tracks")
    print(f"{'=' * 60}")

    feature_rows = labels["feature_row"].to_numpy()

    print("First 10 feature_row values:")
    print(feature_rows[:10])

    print("Unique feature_row values:",
          len(np.unique(feature_rows)))

    print("Min feature_row:", feature_rows.min())
    print("Max feature_row:", feature_rows.max())

    # This checks whether feature_row is a complete
    # local row index. It does not assume the rows
    # are in increasing order.
    is_local_permutation = np.array_equal(
        np.sort(feature_rows),
        np.arange(n)
    )

    print(
        "feature_row is permutation of 0..n-1:",
        is_local_permutation
    )

    arrays = {
        "CNN Audio": (
            saved_dir / f"X_{split}_audio.npy"
        ),
        "CNN Visual": (
            saved_dir / f"X_{split}_visual.npy"
        ),
        "MM Audio": (
            mm_dir / f"X_{split}_mm-audio.npy"
        ),
        "MM Visual": (
            mm_dir / f"X_{split}_mm-visual.npy"
        )
    }

    for name, path in arrays.items():

        if not path.exists():
            print(f"{name}: FILE MISSING")
            continue

        X = np.load(
            path,
            mmap_mode="r"
        )

        rows_match = X.shape[0] == n

        print(
            f"{name}: shape={X.shape}, "
            f"rows_match={rows_match}"
        )

        results.append({
            "split": split,
            "representation": name,
            "n_labels": n,
            "n_feature_rows": X.shape[0],
            "feature_dimensions": X.shape[1],
            "rows_match": rows_match,
            "local_row_permutation": is_local_permutation
        })

summary = pd.DataFrame(results)

print("\n" + "=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

print(summary.to_string(index=False))

print("\nAll array sizes match:",
      bool(summary["rows_match"].all()))

print("\nAll feature_row indices are local permutations:",
      bool(summary["local_row_permutation"].all()))

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(
    audit_dir / "feature_alignment_audit.csv",
    index=False
)

print("\nSTEP 1D COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1E
# IDENTIFY ORIGINAL FEATURE FILES
# ============================================================

import tarfile
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")
archive_path = PROJECT / "audio_features" / "msdi-data.tar.gz"

with tarfile.open(archive_path, "r:gz") as tar:
    feature_files = [
        (m.name, m.size)
        for m in tar.getmembers()
        if m.isfile()
        and any(
            keyword in m.name.lower()
            for keyword in [
                "feature",
                "audio",
                "visual",
                "representation",
                ".npy",
                ".npz"
            ]
        )
    ]

print("=" * 75)
print("ORIGINAL ARCHIVE — FEATURE FILES")
print("=" * 75)

print("Matching files:", len(feature_files))

for name, size in feature_files[:100]:
    print(f"{size / (1024**2):8.2f} MB | {name}")

if len(feature_files) > 100:
    print(f"\n... {len(feature_files) - 100} additional files")

print("\nSTEP 1E INSPECTION COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1F
# DEFINITIVE FEATURE ALIGNMENT VERIFICATION
# ============================================================

from pathlib import Path
import tarfile
import io
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

archive_path = PROJECT / "audio_features" / "msdi-data.tar.gz"

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

representations = {
    "audio": {
        "folder": saved_dir,
        "dimension": 2048
    },
    "visual": {
        "folder": saved_dir,
        "dimension": 2048
    },
    "mm-audio": {
        "folder": mm_dir,
        "dimension": 200
    },
    "mm-visual": {
        "folder": mm_dir,
        "dimension": 200
    }
}

results = []

print("=" * 75)
print("PHASE 1 — DEFINITIVE FEATURE ALIGNMENT AUDIT")
print("=" * 75)

with tarfile.open(archive_path, "r:gz") as tar:

    for split in ["train", "val", "test"]:

        labels = pd.read_csv(
            saved_dir / f"labels_{split}.csv"
        )

        indices = labels["feature_row"].to_numpy(
            dtype=int
        )

        print(f"\n{'=' * 65}")
        print(f"{split.upper()} — {len(labels)} tracks")
        print(f"{'=' * 65}")

        for representation, config in representations.items():

            # Original archive feature file
            archive_name = (
                f"msdi-data/train_data/"
                f"X_{split}_{representation}_MSD-I.npy"
            )

            member = tar.getmember(archive_name)

            with tar.extractfile(member) as f:
                original = np.load(
                    io.BytesIO(f.read()),
                    allow_pickle=False
                )

            # Our filtered feature file
            filtered_path = (
                config["folder"]
                / f"X_{split}_{representation}.npy"
            )

            filtered = np.load(
                filtered_path,
                mmap_mode="r"
            )

            assert filtered.shape == (
                len(labels),
                config["dimension"]
            ), (
                f"Unexpected filtered shape: "
                f"{filtered.shape}"
            )

            assert indices.min() >= 0
            assert indices.max() < original.shape[0]

            # Select original archive rows corresponding
            # to our filtered dataset.
            expected = original[indices]

            assert expected.shape == filtered.shape

            # Compare every feature value.
            matches = np.isclose(
                expected,
                filtered,
                rtol=1e-5,
                atol=1e-7,
                equal_nan=True
            )

            row_matches = matches.all(axis=1)

            matched = int(row_matches.sum())
            mismatched = int((~row_matches).sum())

            max_difference = float(
                np.max(
                    np.abs(
                        expected.astype(np.float64)
                        - filtered.astype(np.float64)
                    )
                )
            )

            print(
                f"{representation:12s} | "
                f"Matched: {matched:5d}/{len(labels)} | "
                f"Mismatched: {mismatched:4d} | "
                f"Max diff: {max_difference:.3e}"
            )

            results.append({
                "split": split,
                "representation": representation,
                "tracks": len(labels),
                "matched_rows": matched,
                "mismatched_rows": mismatched,
                "max_absolute_difference": max_difference
            })

summary = pd.DataFrame(results)

summary.to_csv(
    audit_dir / "definitive_feature_alignment.csv",
    index=False
)

print("\n" + "=" * 75)
print("FINAL RESULTS")
print("=" * 75)

print(summary.to_string(index=False))

all_match = bool(
    (summary["mismatched_rows"] == 0).all()
)

print("\nALL FEATURE VECTORS MATCH:", all_match)

if all_match:
    print(
        "\nSUCCESS: All four representations match "
        "the corresponding original archive rows "
        "for every selected track."
    )
else:
    print(
        "\nWARNING: Some feature vectors do not match. "
        "We need to investigate their indexing."
    )

print("\nAudit saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1G
# INSPECT ORIGINAL ARCHIVE DOCUMENTATION
# ============================================================

from pathlib import Path
import tarfile

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")
archive_path = PROJECT / "audio_features" / "msdi-data.tar.gz"

print("=" * 75)
print("PHASE 1 — FEATURE PROVENANCE INVESTIGATION")
print("=" * 75)

with tarfile.open(archive_path, "r:gz") as tar:

    files = [
        m for m in tar.getmembers()
        if m.isfile()
    ]

    print("\nTOTAL FILES:", len(files))

    print("\nCOMPLETE ARCHIVE FILE LIST:")
    for member in files:
        print(member.name)

    keywords = [
        "readme",
        "license",
        "metadata",
        "description",
        "config",
        "info",
        "citation"
    ]

    documentation = [
        m for m in files
        if any(
            keyword in m.name.lower()
            for keyword in keywords
        )
    ]

    print("\n" + "=" * 75)
    print("POTENTIAL DOCUMENTATION FILES")
    print("=" * 75)

    if not documentation:
        print("No obvious documentation files found.")

    for member in documentation:

        print("\nFILE:", member.name)

        if member.size > 200_000:
            print("File too large to display.")
            continue

        try:
            content = tar.extractfile(member).read()
            print(
                content.decode(
                    "utf-8",
                    errors="replace"
                )[:12000]
            )
        except Exception as exc:
            print("Could not read:", exc)

print("\nSTEP 1G INSPECTION COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1H
# SEARCH FOR ALBUM-LEVEL METADATA
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

print("=" * 75)
print("PHASE 1 — ALBUM METADATA INVESTIGATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Search filenames that might contain album information
# ------------------------------------------------------------

keywords = [
    "album",
    "metadata",
    "track_info",
    "track_metadata",
    "musicbrainz",
    "discogs",
    "msd"
]

matches = []

for path in PROJECT.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()

    if any(keyword in name for keyword in keywords):
        matches.append(path)

print("\nPOTENTIAL METADATA FILES:")
print("Total matches:", len(matches))

for path in matches[:100]:
    print(
        path.relative_to(PROJECT),
        "|",
        round(path.stat().st_size / (1024**2), 2),
        "MB"
    )

if len(matches) > 100:
    print(
        f"\n... {len(matches) - 100} additional files"
    )

# ------------------------------------------------------------
# 2. Inspect columns in small tabular files
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("POTENTIAL ALBUM COLUMNS")
print("=" * 75)

tabular_extensions = {
    ".csv",
    ".tsv",
    ".parquet"
}

candidates = []

for path in PROJECT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in tabular_extensions:
        continue

    try:
        if path.suffix.lower() == ".csv":
            columns = pd.read_csv(
                path,
                nrows=0
            ).columns.tolist()

        elif path.suffix.lower() == ".tsv":
            columns = pd.read_csv(
                path,
                sep="\t",
                nrows=0
            ).columns.tolist()

        else:
            columns = pd.read_parquet(
                path
            ).columns.tolist()

        album_columns = [
            col for col in columns
            if "album" in col.lower()
            or "release" in col.lower()
        ]

        if album_columns:
            candidates.append({
                "file": str(path.relative_to(PROJECT)),
                "album_columns": album_columns
            })

    except Exception:
        continue

print("Files containing album/release columns:",
      len(candidates))

for candidate in candidates:
    print("\nFile:", candidate["file"])
    print("Columns:", candidate["album_columns"])

print("\nSTEP 1H INSPECTION COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1I
# VERIFY ALBUM-LEVEL SEPARATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load original MSD-I metadata
# ------------------------------------------------------------

metadata_path = (
    PROJECT / "audio_features" / "MSD-I_dataset.tsv"
)

metadata = pd.read_csv(
    metadata_path,
    sep="\t",
    low_memory=False
)

print("=" * 75)
print("PHASE 1 — ALBUM-LEVEL SEPARATION AUDIT")
print("=" * 75)

print("\nOriginal metadata shape:", metadata.shape)
print("\nAvailable columns:")
print(metadata.columns.tolist())

required = ["msd_track_id", "album_index"]

missing_columns = [
    c for c in required
    if c not in metadata.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 2. Load current experimental splits
# ------------------------------------------------------------

frames = []

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    labels["our_split"] = split

    frames.append(labels)

our_data = pd.concat(
    frames,
    ignore_index=True
)

assert len(our_data) == 11870
assert our_data["msd_track_id"].is_unique

# ------------------------------------------------------------
# 3. Check metadata uniqueness
# ------------------------------------------------------------

metadata_subset = metadata[
    ["msd_track_id", "album_index"]
].copy()

duplicate_tracks = (
    metadata_subset["msd_track_id"]
    .duplicated()
    .sum()
)

print("\nDuplicate track IDs in metadata:",
      duplicate_tracks)

if duplicate_tracks:
    # A repeated track ID is acceptable only if
    # its album assignment is consistent.
    inconsistent = (
        metadata_subset
        .groupby("msd_track_id")["album_index"]
        .nunique(dropna=False)
    )

    inconsistent = inconsistent[
        inconsistent > 1
    ]

    if len(inconsistent):
        raise ValueError(
            "Some tracks have conflicting album IDs."
        )

metadata_subset = (
    metadata_subset
    .drop_duplicates("msd_track_id")
)

# ------------------------------------------------------------
# 4. Match album IDs to current dataset
# ------------------------------------------------------------

merged = our_data.merge(
    metadata_subset,
    on="msd_track_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nTracks in our dataset:", len(merged))

print("Tracks found in metadata:",
      int((merged["_merge"] == "both").sum()))

print("Tracks missing from metadata:",
      int((merged["_merge"] != "both").sum()))

print("Tracks with missing album IDs:",
      int(merged["album_index"].isna().sum()))

# ------------------------------------------------------------
# 5. Check album overlap between splits
# ------------------------------------------------------------

album_sets = {}

for split in ["train", "val", "test"]:

    subset = merged[
        merged["our_split"] == split
    ]

    album_sets[split] = set(
        subset["album_index"].dropna()
    )

    print(f"\n{split.upper()}")
    print("Tracks:", len(subset))
    print("Unique albums:", len(album_sets[split]))

print("\n" + "=" * 75)
print("ALBUM OVERLAP RESULTS")
print("=" * 75)

overlap_results = []

for a, b in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]:

    overlap = album_sets[a] & album_sets[b]

    print(
        f"{a.upper()} / {b.upper()}: "
        f"{len(overlap)} shared albums"
    )

    overlap_results.append({
        "split_a": a,
        "split_b": b,
        "shared_albums": len(overlap)
    })

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

merged.drop(
    columns=["_merge"]
).to_csv(
    audit_dir / "album_level_metadata_audit.csv",
    index=False
)

pd.DataFrame(overlap_results).to_csv(
    audit_dir / "album_overlap_audit.csv",
    index=False
)

print("\n" + "=" * 75)
print("FINAL ASSESSMENT")
print("=" * 75)

complete_metadata = (
    (merged["_merge"] == "both").all()
    and merged["album_index"].notna().all()
)

no_overlap = all(
    result["shared_albums"] == 0
    for result in overlap_results
)

print("Complete album metadata:", complete_metadata)
print("No album overlap:", no_overlap)

if complete_metadata and no_overlap:
    print(
        "\nSUCCESS: The current dataset is "
        "album-disjoint across all three splits."
    )
elif not complete_metadata:
    print(
        "\nINCOMPLETE: Some album assignments "
        "could not be verified."
    )
else:
    print(
        "\nWARNING: Albums appear in more than "
        "one split. Inspect the overlap report."
    )

print("\nAudit saved to:", audit_dir)
print("STEP 1I COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1J
# VALIDATE ALBUM IDENTIFIERS AND IMAGE URL OVERLAP
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

metadata = pd.read_csv(
    PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    sep="\t",
    low_memory=False
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("PHASE 1 — ALBUM IDENTIFIER VALIDATION")
print("=" * 75)

# 1. Album index frequency across the full dataset
album_counts = metadata["album_index"].value_counts(
    dropna=False
)

print("\nFULL ORIGINAL DATASET")
print("Tracks:", len(metadata))
print("Unique album indices:",
      metadata["album_index"].nunique())
print("Album indices used more than once:",
      int((album_counts > 1).sum()))
print("Maximum tracks per album index:",
      int(album_counts.max()))

print("\nAlbum index frequency distribution:")
print(
    album_counts.value_counts()
    .sort_index()
    .head(15)
    .to_string()
)

# 2. Image URL frequency
valid_urls = metadata.dropna(
    subset=["image_url"]
).copy()

valid_urls["image_url"] = (
    valid_urls["image_url"]
    .astype(str)
    .str.strip()
)

valid_urls = valid_urls[
    valid_urls["image_url"] != ""
]

url_counts = valid_urls["image_url"].value_counts()

print("\nIMAGE URL INFORMATION")
print("Tracks with image URLs:", len(valid_urls))
print("Unique image URLs:", len(url_counts))
print("Image URLs used more than once:",
      int((url_counts > 1).sum()))
print("Maximum tracks sharing one URL:",
      int(url_counts.max()) if len(url_counts) else 0)

# 3. Cross-split image URL overlap
# Use original metadata's split assignments.
print("\nORIGINAL SPLIT VALUES:")
print(metadata["set"].value_counts(dropna=False))

# Normalize split names
aliases = {
    "training": "train",
    "validation": "val",
    "valid": "val",
    "dev": "val",
    "testing": "test"
}

valid_urls["normalized_split"] = (
    valid_urls["set"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(aliases)
)

url_sets = {
    split: set(
        valid_urls.loc[
            valid_urls["normalized_split"] == split,
            "image_url"
        ]
    )
    for split in ["train", "val", "test"]
}

print("\nIMAGE URL OVERLAP ACROSS ORIGINAL SPLITS")

for a, b in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]:
    shared = url_sets[a] & url_sets[b]
    print(f"{a} / {b}: {len(shared)} shared URLs")

# 4. Inspect a few repeated album indices
repeated = album_counts[
    album_counts > 1
].head(5)

print("\nEXAMPLES OF REPEATED ALBUM INDICES")

if repeated.empty:
    print("No repeated album indices found.")
else:
    for album_id in repeated.index:
        rows = metadata.loc[
            metadata["album_index"] == album_id,
            ["msd_track_id", "album_index", "set", "msd_artist_id"]
        ]
        print("\nAlbum index:", album_id)
        print(rows.head(10).to_string(index=False))

print("\nSTEP 1J COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1K
# ALBUM AND COVER-URL OVERLAP IN OUR EXPERIMENTAL SUBSET
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(
    PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    sep="\t",
    low_memory=False
)

frames = []

for split in ["train", "val", "test"]:
    df = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )
    df["our_split"] = split
    frames.append(df)

our_data = pd.concat(
    frames,
    ignore_index=True
)

# Match original album and image metadata
subset = our_data.merge(
    metadata[
        ["msd_track_id", "album_index", "image_url"]
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert len(subset) == 11870
assert subset["album_index"].notna().all()

# Normalize URLs for consistent comparisons
subset["image_url_clean"] = (
    subset["image_url"]
    .astype("string")
    .str.strip()
)

subset.loc[
    subset["image_url_clean"] == "",
    "image_url_clean"
] = pd.NA

pairs = [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]

print("=" * 75)
print("PHASE 1 — SUBSET IMAGE OVERLAP AUDIT")
print("=" * 75)

print("\nEXPERIMENTAL DATASET:", len(subset))

for split in ["train", "val", "test"]:
    part = subset[subset["our_split"] == split]

    print(f"\n{split.upper()}")
    print("Tracks:", len(part))
    print("Unique album indices:",
          part["album_index"].nunique())
    print("Unique image URLs:",
          part["image_url_clean"].nunique())

# ------------------------------------------------------------
# Compare album IDs and cover URLs
# ------------------------------------------------------------

results = []

for a, b in pairs:

    A = subset[subset["our_split"] == a]
    B = subset[subset["our_split"] == b]

    album_overlap = (
        set(A["album_index"].dropna())
        & set(B["album_index"].dropna())
    )

    url_overlap = (
        set(A["image_url_clean"].dropna())
        & set(B["image_url_clean"].dropna())
    )

    affected_A = A[
        A["image_url_clean"].isin(url_overlap)
    ]

    affected_B = B[
        B["image_url_clean"].isin(url_overlap)
    ]

    print("\n" + "-" * 60)
    print(f"{a.upper()} vs {b.upper()}")
    print("-" * 60)

    print("Shared album indices:",
          len(album_overlap))

    print("Shared cover URLs:",
          len(url_overlap))

    print(f"Affected tracks in {a}:",
          len(affected_A))

    print(f"Affected tracks in {b}:",
          len(affected_B))

    results.append({
        "split_a": a,
        "split_b": b,
        "shared_album_indices": len(album_overlap),
        "shared_image_urls": len(url_overlap),
        "affected_tracks_a": len(affected_A),
        "affected_tracks_b": len(affected_B)
    })

# ------------------------------------------------------------
# Full original dataset album-index check
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FULL ORIGINAL DATASET — ALBUM INDEX OVERLAP")
print("=" * 75)

for a, b in pairs:

    A = set(
        metadata.loc[
            metadata["set"] == a,
            "album_index"
        ].dropna()
    )

    B = set(
        metadata.loc[
            metadata["set"] == b,
            "album_index"
        ].dropna()
    )

    print(
        f"{a} / {b}: {len(A & B)} shared album indices"
    )

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv(
    audit_dir / "subset_album_cover_overlap.csv",
    index=False
)

subset.to_csv(
    audit_dir / "subset_album_cover_metadata.csv",
    index=False
)

print("\n" + "=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

print(results_df.to_string(index=False))

print("\nSTEP 1K COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1L
# INSPECT PERCEPTUAL HASH DATA AND COVERAGE
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

files = {
    "Perceptual hashes":
        PROJECT / "album_perceptual_hashes.csv",

    "Near-duplicate candidates":
        PROJECT / "near_duplicate_candidates.csv",

    "Clean album metadata":
        PROJECT / "clean_album_metadata.csv",

    "Final album splits":
        PROJECT / "final_album_splits.csv"
}

print("=" * 75)
print("PHASE 1 — PERCEPTUAL DUPLICATE DATA INSPECTION")
print("=" * 75)

for name, path in files.items():

    print(f"\n{'-' * 65}")
    print(name)
    print(f"{'-' * 65}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    print("\nFirst three rows:")
    print(df.head(3).to_string(index=False))

    if "album_index" in df.columns:
        print(
            "\nUnique album indices:",
            df["album_index"].nunique()
        )

# ------------------------------------------------------------
# Current experimental dataset
# ------------------------------------------------------------

frames = []

for split in ["train", "val", "test"]:

    labels = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )

    labels["our_split"] = split
    frames.append(labels)

our_data = pd.concat(
    frames,
    ignore_index=True
)

print("\n" + "=" * 75)
print("CURRENT EXPERIMENTAL DATASET")
print("=" * 75)

print("Tracks:", len(our_data))
print("Unique tracks:",
      our_data["msd_track_id"].nunique())

# Match original album IDs
metadata = pd.read_csv(
    PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    sep="\t"
)

our_albums = our_data.merge(
    metadata[["msd_track_id", "album_index"]],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

print("Unique album indices:",
      our_albums["album_index"].nunique())

# Check hash coverage
hash_path = PROJECT / "album_perceptual_hashes.csv"

if hash_path.exists():

    hashes = pd.read_csv(hash_path)

    if "album_index" in hashes.columns:

        covered = set(
            hashes["album_index"].dropna()
        )

        selected = set(
            our_albums["album_index"].dropna()
        )

        print("\nPERCEPTUAL HASH COVERAGE")
        print("Albums in our experiment:",
              len(selected))

        print("Albums with hash records:",
              len(selected & covered))

        print("Albums without hash records:",
              len(selected - covered))

        print(
            "Coverage percentage:",
            round(
                100 * len(selected & covered)
                / len(selected),
                2
            ),
            "%"
        )

print("\nSTEP 1L INSPECTION COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1M
# PERCEPTUAL HASH CROSS-SPLIT AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
from itertools import combinations

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

hashes = pd.read_csv(
    PROJECT / "album_perceptual_hashes.csv"
)

candidates = pd.read_csv(
    PROJECT / "near_duplicate_candidates.csv"
)

print("=" * 75)
print("PHASE 1 — PERCEPTUAL HASH DUPLICATE AUDIT")
print("=" * 75)

print("\nEXISTING CANDIDATE PAIRS")
print(candidates.to_string(index=False))

print("\nCandidate distance distribution:")
print(
    candidates["distance"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 1. Normalize perceptual hashes
# ------------------------------------------------------------

hashes["phash_clean"] = (
    hashes["phash"]
    .astype("string")
    .str.strip()
    .str.lower()
)

hashes["split_clean"] = (
    hashes["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

hashes.loc[
    hashes["phash_clean"].isin(["", "nan", "none"]),
    "phash_clean"
] = pd.NA

print("\nHASH DATA INTEGRITY")
print("Rows:", len(hashes))
print("Unique albums:",
      hashes["album_index"].nunique())
print("Missing hashes:",
      hashes["phash_clean"].isna().sum())

assert hashes["album_index"].is_unique
assert hashes["phash_clean"].notna().all()

# ------------------------------------------------------------
# 2. Independently find all identical perceptual hashes
# ------------------------------------------------------------

exact_pairs = []

for phash, group in hashes.groupby("phash_clean"):

    if len(group) < 2:
        continue

    records = group[
        ["album_index", "genre", "split_clean"]
    ].to_dict("records")

    for a, b in combinations(records, 2):

        if a["split_clean"] == b["split_clean"]:
            continue

        exact_pairs.append({
            "album_a": a["album_index"],
            "album_b": b["album_index"],
            "genre_a": a["genre"],
            "genre_b": b["genre"],
            "split_a": a["split_clean"],
            "split_b": b["split_clean"],
            "distance": 0,
            "phash": phash
        })

exact_df = pd.DataFrame(
    exact_pairs,
    columns=[
        "album_a", "album_b",
        "genre_a", "genre_b",
        "split_a", "split_b",
        "distance", "phash"
    ]
)

print("\n" + "=" * 75)
print("INDEPENDENT EXACT-HASH MATCHES")
print("=" * 75)

print("Cross-split pairs:", len(exact_df))

if not exact_df.empty:

    print("\nAll cross-split exact-hash pairs:")
    print(exact_df.to_string(index=False))

    affected = set(exact_df["album_a"]) | set(
        exact_df["album_b"]
    )

    print("\nUnique affected albums:", len(affected))

    print("\nPairs by split combination:")

    exact_df["split_pair"] = exact_df.apply(
        lambda row: " / ".join(
            sorted([row["split_a"], row["split_b"]])
        ),
        axis=1
    )

    print(
        exact_df["split_pair"]
        .value_counts()
        .to_string()
    )

else:
    print("No cross-split exact-hash matches found.")

# ------------------------------------------------------------
# 3. Compare with previous candidate file
# ------------------------------------------------------------

def canonical_pair(a, b):
    return tuple(sorted([int(a), int(b)]))

exact_keys = {
    canonical_pair(a, b)
    for a, b in zip(
        exact_df["album_a"],
        exact_df["album_b"]
    )
}

candidate_zero = candidates[
    candidates["distance"] == 0
]

candidate_keys = {
    canonical_pair(a, b)
    for a, b in zip(
        candidate_zero["album_a"],
        candidate_zero["album_b"]
    )
}

print("\n" + "=" * 75)
print("COMPARISON WITH EXISTING CANDIDATES")
print("=" * 75)

print("Exact cross-split pairs found:",
      len(exact_keys))

print("Distance-zero candidate pairs:",
      len(candidate_keys))

print(
    "Exact cross-split pairs missing from candidates:",
    len(exact_keys - candidate_keys)
)

print(
    "Candidate distance-zero pairs not in "
    "cross-split exact matches:",
    len(candidate_keys - exact_keys)
)

# ------------------------------------------------------------
# 4. Save results
# ------------------------------------------------------------

exact_df.to_csv(
    audit_dir / "exact_phash_cross_split_pairs.csv",
    index=False
)

print("\nSTEP 1M COMPLETE")
print("Audit saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1N
# IDENTIFY TEST TRACKS AFFECTED BY NEAR-DUPLICATE COVERS
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load candidate pairs and original metadata
# ------------------------------------------------------------

candidates = pd.read_csv(
    PROJECT / "near_duplicate_candidates.csv"
)

metadata = pd.read_csv(
    PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    sep="\t"
)

test_labels = pd.read_csv(
    saved_dir / "labels_test.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

print("=" * 75)
print("PHASE 1 — TEST COVER DUPLICATE SENSITIVITY")
print("=" * 75)

# ------------------------------------------------------------
# 2. Identify all test albums in candidate pairs
# ------------------------------------------------------------

test_album_ids = set()

for _, row in candidates.iterrows():

    if row["split_a"] == "test":
        test_album_ids.add(row["album_a"])

    if row["split_b"] == "test":
        test_album_ids.add(row["album_b"])

print("\nUnique test albums flagged:",
      len(test_album_ids))

# ------------------------------------------------------------
# 3. Match albums to track IDs
# ------------------------------------------------------------

flagged_metadata = metadata[
    metadata["album_index"].isin(test_album_ids)
][
    ["msd_track_id", "album_index", "genre"]
].copy()

flagged_tracks = test_labels.merge(
    flagged_metadata[
        ["msd_track_id", "album_index"]
    ],
    on="msd_track_id",
    how="inner",
    validate="one_to_one"
)

print("Flagged test tracks:",
      len(flagged_tracks))

# ------------------------------------------------------------
# 4. Join typicality and predictions
# ------------------------------------------------------------

prediction_columns = [
    "msd_track_id",
    "typicality_quintile",
    "visual_distance",
    "audio_correct",
    "early_fusion_correct",
    "visual_correct"
]

available = [
    col for col in prediction_columns
    if col in typicality.columns
]

details = flagged_tracks.merge(
    typicality[available],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 5. Summarize affected examples
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FLAGGED TEST EXAMPLES")
print("=" * 75)

display_columns = [
    "msd_track_id",
    "album_index",
    "genre",
    "typicality_quintile",
    "visual_distance",
    "audio_correct",
    "early_fusion_correct",
    "visual_correct"
]

display_columns = [
    col for col in display_columns
    if col in details.columns
]

print(
    details[display_columns]
    .to_string(index=False)
)

print("\nFlagged examples by typicality quintile:")

if "typicality_quintile" in details.columns:
    print(
        details["typicality_quintile"]
        .value_counts()
        .sort_index()
        .to_string()
    )

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

details.to_csv(
    audit_dir / "flagged_test_cover_examples.csv",
    index=False
)

print("\nSTEP 1N COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1O
# CORRECT PERCEPTUAL-HASH ALBUM IDENTIFIER MAPPING
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

candidates = pd.read_csv(
    PROJECT / "near_duplicate_candidates.csv"
)

clean_metadata = pd.read_csv(
    PROJECT / "clean_album_metadata.csv"
)

original_metadata = pd.read_csv(
    PROJECT / "audio_features" / "MSD-I_dataset.tsv",
    sep="\t"
)

frames = []

for split in ["train", "val", "test"]:
    df = pd.read_csv(
        saved_dir / f"labels_{split}.csv"
    )
    df["our_split"] = split
    frames.append(df)

our_data = pd.concat(frames, ignore_index=True)

print("=" * 75)
print("PHASE 1 — CORRECTED NEAR-DUPLICATE MAPPING")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify clean metadata's album ID uniqueness
# ------------------------------------------------------------

print("\nCLEAN METADATA")
print("Rows:", len(clean_metadata))
print(
    "Unique album IDs:",
    clean_metadata["album_index"].nunique()
)
print(
    "Unique track IDs:",
    clean_metadata["msd_track_id"].nunique()
)

assert clean_metadata["album_index"].is_unique
assert clean_metadata["msd_track_id"].is_unique

# ------------------------------------------------------------
# 2. Check whether album IDs mean the same thing
#    in cleaned and original metadata
# ------------------------------------------------------------

comparison = clean_metadata[
    ["msd_track_id", "album_index"]
].merge(
    original_metadata[
        ["msd_track_id", "album_index"]
    ],
    on="msd_track_id",
    how="inner",
    suffixes=("_clean", "_original"),
    validate="one_to_one"
)

agreement = (
    comparison["album_index_clean"]
    == comparison["album_index_original"]
)

print("\nALBUM IDENTIFIER CONSISTENCY")
print("Tracks compared:", len(comparison))
print("Matching album indices:", int(agreement.sum()))
print("Different album indices:", int((~agreement).sum()))

# ------------------------------------------------------------
# 3. Expand candidate pairs into track-level records
# ------------------------------------------------------------

album_to_track = clean_metadata.set_index(
    "album_index"
)["msd_track_id"]

expanded = candidates.copy()

expanded["track_a"] = expanded[
    "album_a"
].map(album_to_track)

expanded["track_b"] = expanded[
    "album_b"
].map(album_to_track)

split_lookup = our_data.set_index(
    "msd_track_id"
)["our_split"]

expanded["current_split_a"] = (
    expanded["track_a"].map(split_lookup)
)

expanded["current_split_b"] = (
    expanded["track_b"].map(split_lookup)
)

print("\nCANDIDATE PAIR COVERAGE")
print("Total pairs:", len(expanded))

print(
    "Pairs with both track IDs found:",
    int(
        expanded[["track_a", "track_b"]]
        .notna().all(axis=1).sum()
    )
)

print(
    "Pairs with both tracks in current experiment:",
    int(
        expanded[
            ["current_split_a", "current_split_b"]
        ].notna().all(axis=1).sum()
    )
)

print("\nEXPANDED CANDIDATE PAIRS")

print(
    expanded[
        [
            "album_a", "album_b",
            "distance",
            "split_a", "split_b",
            "current_split_a", "current_split_b",
            "track_a", "track_b"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 4. Identify genuinely affected current test tracks
# ------------------------------------------------------------

test_tracks = set()

for _, row in expanded.iterrows():

    if row["current_split_a"] == "test":
        test_tracks.add(row["track_a"])

    if row["current_split_b"] == "test":
        test_tracks.add(row["track_b"])

print("\n" + "=" * 75)
print("CORRECTED TEST COVER RESULTS")
print("=" * 75)

print(
    "Unique flagged test tracks:",
    len(test_tracks)
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

flagged = typicality[
    typicality["msd_track_id"].isin(test_tracks)
].copy()

print(
    "Flagged tracks found in typicality data:",
    len(flagged)
)

columns = [
    "msd_track_id",
    "genre",
    "typicality_quintile",
    "audio_correct",
    "visual_correct",
    "early_fusion_correct"
]

print(
    "\nCORRECTED FLAGGED TEST EXAMPLES"
)

print(
    flagged[columns].to_string(index=False)
)

print("\nTYPICALITY DISTRIBUTION")

print(
    flagged["typicality_quintile"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 5. Save corrected results
# ------------------------------------------------------------

expanded.to_csv(
    audit_dir / "corrected_near_duplicate_pairs.csv",
    index=False
)

flagged.to_csv(
    audit_dir / "corrected_flagged_test_covers.csv",
    index=False
)

print("\nSTEP 1O COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1P
# TRAIN-TEST COVER SIMILARITY SENSITIVITY ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

pairs = pd.read_csv(
    audit_dir / "corrected_near_duplicate_pairs.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

print("=" * 75)
print("PHASE 1 — COVER SIMILARITY SENSITIVITY ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Identify flagged train-test pairs using CURRENT splits
# ------------------------------------------------------------

train_test_pairs = pairs[
    (
        (pairs["current_split_a"] == "train")
        & (pairs["current_split_b"] == "test")
    )
    |
    (
        (pairs["current_split_a"] == "test")
        & (pairs["current_split_b"] == "train")
    )
].copy()

print("\nFlagged train-test candidate pairs:",
      len(train_test_pairs))

test_tracks = set()

for _, row in train_test_pairs.iterrows():

    if row["current_split_a"] == "test":
        test_tracks.add(row["track_a"])

    if row["current_split_b"] == "test":
        test_tracks.add(row["track_b"])

print("Unique affected test tracks:", len(test_tracks))

print("\nTrain-test candidate distances:")
print(
    train_test_pairs["distance"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 2. Identify affected test examples
# ------------------------------------------------------------

df = typicality.copy()

df["flagged_train_test_cover"] = (
    df["msd_track_id"].isin(test_tracks)
)

print("\nTotal test examples:", len(df))
print(
    "Flagged test examples:",
    int(df["flagged_train_test_cover"].sum())
)
print(
    "Unflagged test examples:",
    int((~df["flagged_train_test_cover"]).sum())
)

# ------------------------------------------------------------
# 3. Compare model accuracy
# ------------------------------------------------------------

def summarize(group, name):

    return {
        "group": name,
        "n": len(group),
        "audio_accuracy": group[
            "audio_correct"
        ].astype(float).mean(),
        "visual_accuracy": group[
            "visual_correct"
        ].astype(float).mean(),
        "fusion_accuracy": group[
            "early_fusion_correct"
        ].astype(float).mean(),
    }

summary = pd.DataFrame([
    summarize(df, "Full test set"),
    summarize(
        df[df["flagged_train_test_cover"]],
        "Flagged train-test covers"
    ),
    summarize(
        df[~df["flagged_train_test_cover"]],
        "Excluding flagged covers"
    )
])

summary["fusion_minus_audio"] = (
    summary["fusion_accuracy"]
    - summary["audio_accuracy"]
)

print("\n" + "=" * 75)
print("OVERALL SENSITIVITY RESULTS")
print("=" * 75)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 4. Typicality quintile comparison
# ------------------------------------------------------------

quintile_rows = []

for q in sorted(
    df["typicality_quintile"].dropna().unique()
):

    full = df[
        df["typicality_quintile"] == q
    ]

    clean = full[
        ~full["flagged_train_test_cover"]
    ]

    quintile_rows.append({
        "quintile": q,
        "n_full": len(full),
        "n_excluding_flagged": len(clean),
        "audio_full": full["audio_correct"].mean(),
        "fusion_full": full["early_fusion_correct"].mean(),
        "audio_excluding": clean["audio_correct"].mean(),
        "fusion_excluding": clean[
            "early_fusion_correct"
        ].mean()
    })

quintile_summary = pd.DataFrame(quintile_rows)

quintile_summary["fusion_gain_full"] = (
    quintile_summary["fusion_full"]
    - quintile_summary["audio_full"]
)

quintile_summary["fusion_gain_excluding"] = (
    quintile_summary["fusion_excluding"]
    - quintile_summary["audio_excluding"]
)

print("\n" + "=" * 75)
print("TYPICALITY SENSITIVITY")
print("=" * 75)

print(
    quintile_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------

train_test_pairs.to_csv(
    audit_dir / "current_train_test_cover_candidates.csv",
    index=False
)

summary.to_csv(
    audit_dir / "cover_similarity_sensitivity_overall.csv",
    index=False
)

quintile_summary.to_csv(
    audit_dir / "cover_similarity_sensitivity_quintiles.csv",
    index=False
)

print("\nSTEP 1P COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1Q
# RECONSTRUCT VISUAL-TYPICALITY DISTANCES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, normalize

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load visual features and labels
# ------------------------------------------------------------

X_train = np.load(saved_dir / "X_train_visual.npy")
X_test = np.load(saved_dir / "X_test_visual.npy")

train_labels = pd.read_csv(saved_dir / "labels_train.csv")
test_labels = pd.read_csv(saved_dir / "labels_test.csv")

saved_typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

assert len(X_train) == len(train_labels)
assert len(X_test) == len(test_labels)
assert test_labels["msd_track_id"].is_unique
assert saved_typicality["msd_track_id"].is_unique

# Align saved distances to feature row order
aligned = test_labels[["msd_track_id"]].merge(
    saved_typicality[
        ["msd_track_id", "visual_distance",
         "typicality_quintile"]
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert aligned["visual_distance"].notna().all()

saved_distances = aligned[
    "visual_distance"
].to_numpy(dtype=float)

train_genres = train_labels["genre"].to_numpy()
test_genres = test_labels["genre"].to_numpy()

print("=" * 75)
print("PHASE 1 — VISUAL-TYPICALITY RECONSTRUCTION")
print("=" * 75)

print("\nTrain feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
print("Saved distances:", len(saved_distances))

# ------------------------------------------------------------
# 2. Candidate distance calculation
# ------------------------------------------------------------

def centroid_distances(Xtr, Xte, ytr, yte):

    centroids = {
        genre: Xtr[ytr == genre].mean(axis=0)
        for genre in np.unique(ytr)
    }

    return np.array([
        np.linalg.norm(
            Xte[i] - centroids[genre]
        )
        for i, genre in enumerate(yte)
    ])

# Raw embeddings
raw_distances = centroid_distances(
    X_train, X_test,
    train_genres, test_genres
)

# Standardized embeddings:
# fit scaler ONLY on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_distances = centroid_distances(
    X_train_scaled, X_test_scaled,
    train_genres, test_genres
)

# Unit-normalized embeddings
X_train_unit = normalize(X_train)
X_test_unit = normalize(X_test)

unit_distances = centroid_distances(
    X_train_unit, X_test_unit,
    train_genres, test_genres
)

# ------------------------------------------------------------
# 3. Compare against saved distances
# ------------------------------------------------------------

methods = {
    "Raw Euclidean": raw_distances,
    "Standardized Euclidean": scaled_distances,
    "Unit-normalized Euclidean": unit_distances
}

rows = []

for name, calculated in methods.items():

    absolute_error = np.abs(
        calculated - saved_distances
    )

    rows.append({
        "method": name,
        "mean_absolute_error": absolute_error.mean(),
        "max_absolute_error": absolute_error.max(),
        "correlation": np.corrcoef(
            calculated,
            saved_distances
        )[0, 1],
        "exact_close_fraction": np.isclose(
            calculated,
            saved_distances,
            rtol=1e-5,
            atol=1e-7
        ).mean()
    })

comparison = pd.DataFrame(rows)

print("\n" + "=" * 75)
print("DISTANCE RECONSTRUCTION RESULTS")
print("=" * 75)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# ------------------------------------------------------------
# 4. Inspect quintile construction
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SAVED TYPICALITY QUINTILES")
print("=" * 75)

print(
    aligned.groupby("typicality_quintile")[
        "visual_distance"
    ].agg(["count", "min", "max", "mean"])
    .to_string()
)

# Check whether quintiles follow increasing distance
ranges = aligned.groupby(
    "typicality_quintile"
)["visual_distance"].agg(["min", "max"])

ordered = all(
    ranges.iloc[i]["max"]
    <= ranges.iloc[i + 1]["min"]
    for i in range(len(ranges) - 1)
)

print("\nQuintiles ordered by distance:", ordered)

# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

comparison.to_csv(
    audit_dir / "visual_typicality_reconstruction.csv",
    index=False
)

print("\nSTEP 1Q COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1R
# INVESTIGATE VISUAL TYPICALITY CONSTRUCTION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_distances

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

audit_dir.mkdir(parents=True, exist_ok=True)

train_labels = pd.read_csv(
    saved_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    saved_dir / "labels_test.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

aligned = test_labels[["msd_track_id", "genre"]].merge(
    typicality[
        ["msd_track_id", "visual_distance",
         "typicality_quintile"]
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert aligned["visual_distance"].notna().all()

y_train = train_labels["genre"].to_numpy()
y_test = test_labels["genre"].to_numpy()
saved = aligned["visual_distance"].to_numpy()

print("=" * 75)
print("PHASE 1 — TYPICALITY FORMULA INVESTIGATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Check within-genre quintile ordering
# ------------------------------------------------------------

print("\nWITHIN-GENRE QUINTILE RANGES")

genre_rows = []

for genre, group in aligned.groupby("genre"):

    ranges = group.groupby(
        "typicality_quintile"
    )["visual_distance"].agg(
        ["count", "min", "max"]
    )

    ordered = all(
        ranges.iloc[i]["max"]
        <= ranges.iloc[i + 1]["min"]
        for i in range(len(ranges) - 1)
    )

    genre_rows.append({
        "genre": genre,
        "quintiles_ordered": ordered,
        "number_of_quintiles": len(ranges)
    })

    print(f"\n{genre}")
    print(ranges.to_string())
    print("Ordered:", ordered)

# ------------------------------------------------------------
# 2. Reconstruct cosine distance to true-genre centroid
# ------------------------------------------------------------

def cosine_to_centroid(Xtr, Xte):

    centroids = {
        genre: Xtr[y_train == genre].mean(axis=0)
        for genre in np.unique(y_train)
    }

    result = []

    for i, genre in enumerate(y_test):

        x = Xte[i].reshape(1, -1)
        c = centroids[genre].reshape(1, -1)

        result.append(
            cosine_distances(x, c)[0, 0]
        )

    return np.array(result)

def euclidean_unit_to_centroid(Xtr, Xte):

    Xtr = normalize(Xtr)
    Xte = normalize(Xte)

    centroids = {
        genre: Xtr[y_train == genre].mean(axis=0)
        for genre in np.unique(y_train)
    }

    return np.array([
        np.linalg.norm(
            Xte[i] - centroids[genre]
        )
        for i, genre in enumerate(y_test)
    ])

def compare(name, calculated):

    error = np.abs(calculated - saved)

    return {
        "method": name,
        "MAE": error.mean(),
        "max_error": error.max(),
        "correlation": np.corrcoef(
            calculated, saved
        )[0, 1],
        "exact_match_fraction": np.isclose(
            calculated, saved,
            rtol=1e-5,
            atol=1e-7
        ).mean()
    }

results = []

# CNN visual features
Xtr_cnn = np.load(
    saved_dir / "X_train_visual.npy"
)

Xte_cnn = np.load(
    saved_dir / "X_test_visual.npy"
)

results.append(
    compare(
        "CNN cosine to centroid",
        cosine_to_centroid(Xtr_cnn, Xte_cnn)
    )
)

results.append(
    compare(
        "CNN unit Euclidean",
        euclidean_unit_to_centroid(
            Xtr_cnn, Xte_cnn
        )
    )
)

# MM visual features
mm_train_path = (
    mm_dir / "X_train_mm-visual.npy"
)

mm_test_path = (
    mm_dir / "X_test_mm-visual.npy"
)

if mm_train_path.exists() and mm_test_path.exists():

    Xtr_mm = np.load(mm_train_path)
    Xte_mm = np.load(mm_test_path)

    assert len(Xtr_mm) == len(y_train)
    assert len(Xte_mm) == len(y_test)

    results.append(
        compare(
            "MM cosine to centroid",
            cosine_to_centroid(Xtr_mm, Xte_mm)
        )
    )

    results.append(
        compare(
            "MM unit Euclidean",
            euclidean_unit_to_centroid(
                Xtr_mm, Xte_mm
            )
        )
    )

comparison = pd.DataFrame(results)

print("\n" + "=" * 75)
print("DISTANCE FORMULA COMPARISON")
print("=" * 75)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n" + "=" * 75)
print("GENRE QUINTILE ORDERING SUMMARY")
print("=" * 75)

print(
    pd.DataFrame(genre_rows).to_string(index=False)
)

comparison.to_csv(
    audit_dir / "typicality_formula_investigation.csv",
    index=False
)

pd.DataFrame(genre_rows).to_csv(
    audit_dir / "typicality_genre_quintile_audit.csv",
    index=False
)

print("\nSTEP 1R COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1S
# LOCATE THE ORIGINAL VISUAL-TYPICALITY IMPLEMENTATION
# ============================================================

from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

keywords = [
    "visual_distance",
    "typicality_quintile",
    "final_test_typicality_details",
    "centroid_distances",
    "genre_centroid",
    "visual_centroid",
    "pd.qcut",
    "qcut(",
]

extensions = {".py", ".ipynb"}

print("=" * 75)
print("PHASE 1 — ORIGINAL TYPICALITY CODE SEARCH")
print("=" * 75)

matches = []
files_checked = 0

for path in PROJECT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    files_checked += 1

    try:
        if path.suffix.lower() == ".ipynb":

            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )

            sections = []

            for cell_number, cell in enumerate(
                notebook.get("cells", [])
            ):

                if cell.get("cell_type") != "code":
                    continue

                source = cell.get("source", [])

                if isinstance(source, list):
                    source = "".join(source)

                sections.append(
                    (f"Cell {cell_number}", source)
                )

        else:

            sections = [
                (
                    "Python script",
                    path.read_text(
                        encoding="utf-8",
                        errors="replace"
                    )
                )
            ]

        for section_name, source in sections:

            lines = source.splitlines()

            hit_indices = [
                i for i, line in enumerate(lines)
                if any(
                    word.lower() in line.lower()
                    for word in keywords
                )
            ]

            if not hit_indices:
                continue

            matches.append(
                (str(path.relative_to(PROJECT)),
                 section_name,
                 len(hit_indices))
            )

            print("\n" + "-" * 75)
            print(
                "FILE:",
                path.relative_to(PROJECT)
            )
            print("SECTION:", section_name)
            print("Keyword matches:", len(hit_indices))

            # Show context around the first few matches
            shown_lines = set()

            for index in hit_indices[:8]:

                start = max(0, index - 5)
                end = min(len(lines), index + 7)

                for j in range(start, end):
                    if j not in shown_lines:
                        print(
                            f"{j + 1:04d}: {lines[j][:180]}"
                        )
                        shown_lines.add(j)

                print("...")

    except Exception as error:

        print(
            "Could not inspect:",
            path.name,
            str(error)[:120]
        )

print("\n" + "=" * 75)
print("SEARCH SUMMARY")
print("=" * 75)

print("Code files checked:", files_checked)
print("Sections with matches:", len(matches))

if matches:

    print("\nMatching files:")

    for filename, section, count in matches:
        print(
            f"- {filename} | {section} | "
            f"{count} matches"
        )

else:
    print(
        "\nNo matching saved code found. "
        "The original calculation may exist "
        "only in a Colab notebook that was "
        "not saved to this project folder."
    )

print("\nSTEP 1S COMPLETE")


In [ ]:

# ============================================================
# PHASE 1 — STEP 1T
# TEST CENTROID NORMALIZATION VARIANTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize, StandardScaler

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

train_labels = pd.read_csv(
    saved_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    saved_dir / "labels_test.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

aligned = test_labels[["msd_track_id"]].merge(
    typicality[
        ["msd_track_id", "visual_distance"]
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert aligned["visual_distance"].notna().all()

saved = aligned["visual_distance"].to_numpy()

y_train = train_labels["genre"].to_numpy()
y_test = test_labels["genre"].to_numpy()

print("=" * 75)
print("PHASE 1 — CENTROID NORMALIZATION INVESTIGATION")
print("=" * 75)

def reconstruct(Xtr, Xte, variant):

    if variant.startswith("standardized"):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

    if variant == "normalize_vectors_then_centroid":
        Xtr = normalize(Xtr)
        Xte = normalize(Xte)

    centroids = {
        g: Xtr[y_train == g].mean(axis=0)
        for g in np.unique(y_train)
    }

    if variant in [
        "normalize_centroid_and_test",
        "standardized_normalize_centroid_and_test",
        "normalize_vectors_then_centroid_normalized"
    ]:

        centroids = {
            g: normalize(
                centroids[g].reshape(1, -1)
            )[0]
            for g in centroids
        }

        Xte = normalize(Xte)

    elif variant == "normalize_vectors_then_centroid":
        pass

    elif variant == "standardized_raw":
        pass

    elif variant == "raw":
        pass

    else:
        raise ValueError(
            f"Unknown variant: {variant}"
        )

    return np.array([
        np.linalg.norm(
            Xte[i] - centroids[g]
        )
        for i, g in enumerate(y_test)
    ])

def evaluate(name, calculated):

    error = np.abs(calculated - saved)

    return {
        "method": name,
        "MAE": error.mean(),
        "max_error": error.max(),
        "correlation": np.corrcoef(
            calculated, saved
        )[0, 1],
        "exact_fraction": np.isclose(
            calculated,
            saved,
            rtol=1e-5,
            atol=1e-7
        ).mean()
    }

variants = [
    "raw",
    "normalize_centroid_and_test",
    "normalize_vectors_then_centroid",
    "normalize_vectors_then_centroid_normalized",
    "standardized_raw",
    "standardized_normalize_centroid_and_test"
]

feature_sets = {
    "CNN Visual": (
        saved_dir / "X_train_visual.npy",
        saved_dir / "X_test_visual.npy"
    ),
    "MM Visual": (
        mm_dir / "X_train_mm-visual.npy",
        mm_dir / "X_test_mm-visual.npy"
    )
}

results = []

for feature_name, (train_path, test_path) in feature_sets.items():

    if not train_path.exists() or not test_path.exists():
        print("Missing:", feature_name)
        continue

    Xtr = np.load(train_path)
    Xte = np.load(test_path)

    assert len(Xtr) == len(y_train)
    assert len(Xte) == len(y_test)

    for variant in variants:

        calculated = reconstruct(
            Xtr, Xte, variant
        )

        results.append(
            evaluate(
                f"{feature_name} — {variant}",
                calculated
            )
        )

comparison = pd.DataFrame(results).sort_values(
    "MAE"
)

print("\nRECONSTRUCTION RESULTS")
print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

comparison.to_csv(
    audit_dir / "centroid_normalization_variants.csv",
    index=False
)

print("\nSTEP 1T COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1U
# RECONSTRUCT VISUAL TYPICALITY AND QUINTILE ASSIGNMENTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, normalize
from scipy.stats import spearmanr
from sklearn.metrics import confusion_matrix

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load data and align by track ID
# ------------------------------------------------------------

X_train = np.load(saved_dir / "X_train_visual.npy")
X_test = np.load(saved_dir / "X_test_visual.npy")

train_labels = pd.read_csv(
    saved_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    saved_dir / "labels_test.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

aligned = test_labels[
    ["msd_track_id", "genre"]
].merge(
    typicality[
        [
            "msd_track_id",
            "visual_distance",
            "typicality_quintile"
        ]
    ],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert aligned["visual_distance"].notna().all()
assert aligned["typicality_quintile"].notna().all()

y_train = train_labels["genre"].to_numpy()
y_test = test_labels["genre"].to_numpy()

saved = aligned["visual_distance"].to_numpy(
    dtype=float
)

# ------------------------------------------------------------
# 2. Reconstruct standardized, normalized CNN distances
# ------------------------------------------------------------

scaler = StandardScaler()

Xtr = scaler.fit_transform(X_train)
Xte = scaler.transform(X_test)

centroids = {
    genre: Xtr[y_train == genre].mean(axis=0)
    for genre in np.unique(y_train)
}

centroids = {
    genre: normalize(
        centroid.reshape(1, -1)
    )[0]
    for genre, centroid in centroids.items()
}

Xte = normalize(Xte)

reconstructed = np.array([
    np.linalg.norm(
        Xte[i] - centroids[genre]
    )
    for i, genre in enumerate(y_test)
])

# ------------------------------------------------------------
# 3. Examine linear relationship
# ------------------------------------------------------------

slope, intercept = np.polyfit(
    reconstructed,
    saved,
    deg=1
)

fitted = slope * reconstructed + intercept

print("=" * 75)
print("PHASE 1 — TYPICALITY RECONSTRUCTION")
print("=" * 75)

print("\nLINEAR RELATIONSHIP")

print("Slope:", slope)
print("Intercept:", intercept)

print(
    "Correlation:",
    np.corrcoef(reconstructed, saved)[0, 1]
)

print(
    "Spearman rank correlation:",
    spearmanr(reconstructed, saved).statistic
)

print(
    "MAE after linear transformation:",
    np.mean(np.abs(fitted - saved))
)

print(
    "Maximum error after transformation:",
    np.max(np.abs(fitted - saved))
)

# ------------------------------------------------------------
# 4. Reconstruct quintiles within each genre
# ------------------------------------------------------------

aligned["reconstructed_distance"] = reconstructed

def assign_quintiles(group):

    group = group.copy()

    # Rank first to avoid duplicate bin-edge issues
    ranks = group[
        "reconstructed_distance"
    ].rank(method="first")

    group["reconstructed_quintile"] = pd.qcut(
        ranks,
        q=5,
        labels=[1, 2, 3, 4, 5]
    ).astype(int)

    return group

reconstructed_df = pd.concat(
    [
        assign_quintiles(group)
        for _, group in aligned.groupby("genre")
    ],
    ignore_index=True
)

reconstructed_df["quintile_match"] = (
    reconstructed_df["reconstructed_quintile"]
    == reconstructed_df["typicality_quintile"]
)

agreement = reconstructed_df[
    "quintile_match"
].mean()

print("\n" + "=" * 75)
print("QUINTILE RECONSTRUCTION")
print("=" * 75)

print(
    "Overall quintile agreement:",
    f"{agreement:.2%}"
)

print("\nAgreement by genre:")

print(
    reconstructed_df.groupby("genre")[
        "quintile_match"
    ].agg(["count", "mean"])
    .to_string()
)

print("\nOriginal vs reconstructed quintiles:")

print(
    pd.crosstab(
        reconstructed_df["typicality_quintile"],
        reconstructed_df["reconstructed_quintile"],
        rownames=["Saved"],
        colnames=["Reconstructed"]
    ).to_string()
)

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

reconstructed_df.to_csv(
    audit_dir / "visual_typicality_reconstructed_quintiles.csv",
    index=False
)

print("\nSTEP 1U COMPLETE")
print("Saved to:", audit_dir)


In [ ]:

# ============================================================
# PHASE 1 — STEP 1V
# VERIFY EXACT DISTANCE TRANSFORMATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

audit_dir = (
    PROJECT / "finetuning_experiments"
    / "methodology_audit"
)

df = pd.read_csv(
    audit_dir / "visual_typicality_reconstructed_quintiles.csv"
)

saved = df["visual_distance"].to_numpy(dtype=float)

euclidean = df[
    "reconstructed_distance"
].to_numpy(dtype=float)

# Candidate transformations
candidates = {
    "Euclidean distance": euclidean,
    "Squared Euclidean distance": euclidean ** 2,
    "Cosine distance": euclidean ** 2 / 2,
    "Half Euclidean distance": euclidean / 2,
    "Euclidean / sqrt(2)": euclidean / np.sqrt(2),
}

rows = []

for name, values in candidates.items():

    errors = np.abs(values - saved)

    rows.append({
        "transformation": name,
        "MAE": errors.mean(),
        "max_error": errors.max(),
        "exact_fraction": np.isclose(
            values,
            saved,
            rtol=1e-5,
            atol=1e-7
        ).mean()
    })

results = pd.DataFrame(rows).sort_values("MAE")

print("=" * 75)
print("PHASE 1 — EXACT DISTANCE TRANSFORMATION")
print("=" * 75)

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.10f}"
    )
)

print("\nFIRST 10 SAVED VS RECONSTRUCTED VALUES")

comparison = pd.DataFrame({
    "saved_distance": saved[:10],
    "euclidean": euclidean[:10],
    "squared_euclidean": euclidean[:10] ** 2,
    "cosine_distance": euclidean[:10] ** 2 / 2
})

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

results.to_csv(
    audit_dir / "exact_typicality_transformation_audit.csv",
    index=False
)

print("\nSTEP 1V COMPLETE")


In [ ]:

# ============================================================
# PHASE 2 — STEP 2A
# AUDIO-TO-FUSION PREDICTION TRANSITIONS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)

phase2_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

required = [
    "msd_track_id",
    "genre",
    "typicality_quintile",
    "audio_correct",
    "early_fusion_correct"
]

assert all(c in df.columns for c in required)
assert df["msd_track_id"].is_unique
assert len(df) == 1792

# Convert saved correctness indicators safely
def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.to_numpy()

    values = series.astype(str).str.strip().str.lower()

    assert values.isin(
        ["true", "false", "1", "0"]
    ).all(), "Unexpected correctness values"

    return values.isin(["true", "1"]).to_numpy()

audio = to_bool(df["audio_correct"])
fusion = to_bool(df["early_fusion_correct"])

conditions = [
    audio & fusion,
    (~audio) & fusion,
    audio & (~fusion),
    (~audio) & (~fusion)
]

labels = [
    "Both correct",
    "Rescued",
    "Harmed",
    "Both wrong"
]

df["transition"] = np.select(
    conditions,
    labels,
    default="Unknown"
)

assert (df["transition"] != "Unknown").all()

print("=" * 75)
print("PHASE 2 — AUDIO VS EARLY-FUSION TRANSITIONS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Overall transition counts
# ------------------------------------------------------------

overall = df["transition"].value_counts().reindex(
    labels, fill_value=0
)

print("\nOVERALL TRANSITIONS")
print(overall.to_string())

print("\nOverall transition percentages:")
print((100 * overall / len(df)).round(2).to_string())

# ------------------------------------------------------------
# 2. Transitions by visual-typicality quintile
# ------------------------------------------------------------

counts = pd.crosstab(
    df["typicality_quintile"],
    df["transition"]
).reindex(columns=labels, fill_value=0)

percentages = counts.div(
    counts.sum(axis=1), axis=0
) * 100

summary = counts.copy()

summary["N"] = counts.sum(axis=1)
summary["Net rescued"] = (
    counts["Rescued"] - counts["Harmed"]
)
summary["Fusion gain (pp)"] = (
    100 * summary["Net rescued"] / summary["N"]
)

print("\n" + "=" * 75)
print("TRANSITION COUNTS BY QUINTILE")
print("=" * 75)
print(summary.to_string())

print("\n" + "=" * 75)
print("TRANSITION PERCENTAGES BY QUINTILE")
print("=" * 75)
print(percentages.round(2).to_string())

# ------------------------------------------------------------
# 3. By genre and quintile
# ------------------------------------------------------------

genre_counts = pd.crosstab(
    [
        df["genre"],
        df["typicality_quintile"]
    ],
    df["transition"]
).reindex(columns=labels, fill_value=0)

genre_counts["N"] = genre_counts[labels].sum(axis=1)

genre_counts["Fusion gain (pp)"] = (
    100 * (
        genre_counts["Rescued"]
        - genre_counts["Harmed"]
    ) / genre_counts["N"]
)

print("\n" + "=" * 75)
print("FUSION GAINS BY GENRE AND QUINTILE")
print("=" * 75)

print(
    genre_counts[
        ["N", "Rescued", "Harmed", "Fusion gain (pp)"]
    ].round(2).to_string()
)

# ------------------------------------------------------------
# 4. Verify consistency with original accuracy results
# ------------------------------------------------------------

direct_gain = (
    fusion.mean() - audio.mean()
)

transition_gain = (
    (overall["Rescued"] - overall["Harmed"])
    / len(df)
)

assert np.isclose(
    direct_gain,
    transition_gain
)

print("\n" + "=" * 75)
print("CONSISTENCY CHECK")
print("=" * 75)

print(
    "Direct fusion gain (pp):",
    round(100 * direct_gain, 4)
)

print(
    "Transition-based gain (pp):",
    round(100 * transition_gain, 4)
)

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

df.to_csv(
    phase2_dir / "test_prediction_transitions.csv",
    index=False
)

summary.to_csv(
    phase2_dir / "transitions_by_quintile.csv"
)

percentages.to_csv(
    phase2_dir / "transition_percentages_by_quintile.csv"
)

genre_counts.to_csv(
    phase2_dir / "transitions_by_genre_quintile.csv"
)

print("\nSTEP 2A COMPLETE")
print("Saved to:", phase2_dir)


In [ ]:

# ============================================================
# PHASE 2 — STEP 2B
# CNN VS MM FUSION ERROR TRANSITIONS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)
phase2_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load existing prediction files
# ------------------------------------------------------------

cnn = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

mm = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

print("=" * 75)
print("PHASE 2 — CNN VS MM ERROR TRANSITIONS")
print("=" * 75)

print("\nCNN columns:")
print(cnn.columns.tolist())

print("\nMM prediction columns:")
print(mm.columns.tolist())

# ------------------------------------------------------------
# 2. Identify MM prediction columns
# ------------------------------------------------------------

# We first inspect the available columns instead of
# assuming their names, because saved prediction formats
# may differ between experiment stages.

possible_audio = [
    c for c in mm.columns
    if "audio" in c.lower()
    and ("pred" in c.lower() or "correct" in c.lower())
]

possible_fusion = [
    c for c in mm.columns
    if "fusion" in c.lower()
    and ("pred" in c.lower() or "correct" in c.lower())
]

print("\nPossible MM audio columns:")
print(possible_audio)

print("\nPossible MM fusion columns:")
print(possible_fusion)

print("\nFirst three MM prediction rows:")
print(mm.head(3).to_string(index=False))

print("\nSTEP 2B — COLUMN INSPECTION COMPLETE")


In [ ]:

# ============================================================
# PHASE 2 — STEP 2C
# CNN VS MM FUSION: PAIRED ERROR TRANSITION ANALYSIS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)
phase2_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

predictions = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

assert typicality["msd_track_id"].is_unique
assert predictions["msd_track_id"].is_unique

# Align by track ID, not by row position
df = typicality[
    [
        "msd_track_id",
        "genre",
        "msd_artist_id",
        "typicality_quintile",
        "audio_correct",
        "early_fusion_correct"
    ]
].merge(
    predictions[
        [
            "msd_track_id",
            "genre",
            "cnn_audio_correct",
            "cnn_early_fusion_correct",
            "mm_audio_correct",
            "mm_fusion_correct"
        ]
    ],
    on="msd_track_id",
    how="inner",
    suffixes=("_typicality", "_predictions"),
    validate="one_to_one"
)

assert len(df) == 1792
assert (
    df["genre_typicality"]
    == df["genre_predictions"]
).all(), "Genre mismatch between files"

# ------------------------------------------------------------
# 2. Convert correctness columns to booleans
# ------------------------------------------------------------

correctness_columns = [
    "audio_correct",
    "early_fusion_correct",
    "cnn_audio_correct",
    "cnn_early_fusion_correct",
    "mm_audio_correct",
    "mm_fusion_correct"
]

def to_bool(series):

    values = series.astype(str).str.strip().str.lower()

    assert values.isin(
        ["true", "false", "1", "0"]
    ).all(), "Unexpected correctness values"

    return values.isin(["true", "1"]).to_numpy()

for column in correctness_columns:
    df[column] = to_bool(df[column])

# ------------------------------------------------------------
# 3. Verify CNN predictions agree between files
# ------------------------------------------------------------

cnn_audio_agreement = (
    df["audio_correct"]
    == df["cnn_audio_correct"]
).mean()

cnn_fusion_agreement = (
    df["early_fusion_correct"]
    == df["cnn_early_fusion_correct"]
).mean()

print("=" * 75)
print("PHASE 2 — CNN VS MM FUSION TRANSITIONS")
print("=" * 75)

print("\nCROSS-FILE CONSISTENCY CHECK")

print(
    "CNN audio correctness agreement:",
    f"{cnn_audio_agreement:.2%}"
)

print(
    "CNN fusion correctness agreement:",
    f"{cnn_fusion_agreement:.2%}"
)

assert cnn_audio_agreement == 1.0
assert cnn_fusion_agreement == 1.0

# ------------------------------------------------------------
# 4. Define transition categories
# ------------------------------------------------------------

def classify_transitions(audio, fusion):

    return np.select(
        [
            audio & fusion,
            (~audio) & fusion,
            audio & (~fusion),
            (~audio) & (~fusion)
        ],
        [
            "Both correct",
            "Rescued",
            "Harmed",
            "Both wrong"
        ],
        default="Unknown"
    )

df["cnn_transition"] = classify_transitions(
    df["cnn_audio_correct"].to_numpy(),
    df["cnn_early_fusion_correct"].to_numpy()
)

df["mm_transition"] = classify_transitions(
    df["mm_audio_correct"].to_numpy(),
    df["mm_fusion_correct"].to_numpy()
)

transition_order = [
    "Both correct",
    "Rescued",
    "Harmed",
    "Both wrong"
]

# ------------------------------------------------------------
# 5. Overall transition counts
# ------------------------------------------------------------

overall_rows = []

for representation, column in [
    ("CNN", "cnn_transition"),
    ("MM", "mm_transition")
]:

    counts = df[column].value_counts().reindex(
        transition_order,
        fill_value=0
    )

    n = len(df)

    overall_rows.append({
        "representation": representation,
        "N": n,
        "both_correct": counts["Both correct"],
        "rescued": counts["Rescued"],
        "harmed": counts["Harmed"],
        "both_wrong": counts["Both wrong"],
        "rescue_rate_pct": 100 * counts["Rescued"] / n,
        "harm_rate_pct": 100 * counts["Harmed"] / n,
        "net_gain_pp": (
            100 * (
                counts["Rescued"]
                - counts["Harmed"]
            ) / n
        )
    })

overall = pd.DataFrame(overall_rows)

print("\n" + "=" * 75)
print("OVERALL TRANSITION COMPARISON")
print("=" * 75)

print(
    overall.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# ------------------------------------------------------------
# 6. Quintile-level transitions
# ------------------------------------------------------------

quintile_rows = []

for representation, column in [
    ("CNN", "cnn_transition"),
    ("MM", "mm_transition")
]:

    for quintile, group in df.groupby(
        "typicality_quintile"
    ):

        counts = group[column].value_counts().reindex(
            transition_order,
            fill_value=0
        )

        n = len(group)

        quintile_rows.append({
            "representation": representation,
            "quintile": int(quintile),
            "N": n,
            "both_correct": counts["Both correct"],
            "rescued": counts["Rescued"],
            "harmed": counts["Harmed"],
            "both_wrong": counts["Both wrong"],
            "rescue_pct": 100 * counts["Rescued"] / n,
            "harm_pct": 100 * counts["Harmed"] / n,
            "net_gain_pp": (
                100 * (
                    counts["Rescued"]
                    - counts["Harmed"]
                ) / n
            )
        })

quintile_results = pd.DataFrame(quintile_rows)

print("\n" + "=" * 75)
print("QUINTILE TRANSITION COMPARISON")
print("=" * 75)

print(
    quintile_results[
        [
            "representation",
            "quintile",
            "N",
            "rescued",
            "harmed",
            "rescue_pct",
            "harm_pct",
            "net_gain_pp"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# ------------------------------------------------------------
# 7. Side-by-side fusion gain comparison
# ------------------------------------------------------------

comparison = quintile_results.pivot(
    index="quintile",
    columns="representation",
    values="net_gain_pp"
)

comparison["MM_minus_CNN_gain_pp"] = (
    comparison["MM"] - comparison["CNN"]
)

print("\n" + "=" * 75)
print("NET FUSION GAIN BY QUINTILE")
print("=" * 75)

print(
    comparison.round(2).to_string()
)

# ------------------------------------------------------------
# 8. Verify transition identity for each representation
# ------------------------------------------------------------

for representation, audio_col, fusion_col in [
    (
        "CNN",
        "cnn_audio_correct",
        "cnn_early_fusion_correct"
    ),
    (
        "MM",
        "mm_audio_correct",
        "mm_fusion_correct"
    )
]:

    direct_gain = 100 * (
        df[fusion_col].mean()
        - df[audio_col].mean()
    )

    transition_gain = overall.loc[
        overall["representation"] == representation,
        "net_gain_pp"
    ].iloc[0]

    assert np.isclose(
        direct_gain,
        transition_gain
    )

print("\nAll transition consistency checks passed.")

# ------------------------------------------------------------
# 9. Save results
# ------------------------------------------------------------

df.to_csv(
    phase2_dir / "cnn_mm_paired_transitions.csv",
    index=False
)

overall.to_csv(
    phase2_dir / "cnn_mm_overall_transitions.csv",
    index=False
)

quintile_results.to_csv(
    phase2_dir / "cnn_mm_quintile_transitions.csv",
    index=False
)

comparison.to_csv(
    phase2_dir / "cnn_mm_net_gain_comparison.csv"
)

print("\nSTEP 2C COMPLETE")
print("Saved to:", phase2_dir)


In [ ]:

# ============================================================
# PHASE 3 — STEP 3A
# ARTIST-CLUSTER BOOTSTRAP FOR FUSION GAINS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)

phase3_dir = (
    PROJECT / "finetuning_experiments"
    / "phase3_uncertainty"
)
phase3_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    phase2_dir / "cnn_mm_paired_transitions.csv"
)

required = [
    "msd_artist_id",
    "typicality_quintile",
    "cnn_audio_correct",
    "cnn_early_fusion_correct",
    "mm_audio_correct",
    "mm_fusion_correct"
]

assert all(c in df.columns for c in required)
assert len(df) == 1792
assert df["msd_artist_id"].notna().all()

def as_bool(series):
    s = series.astype(str).str.strip().str.lower()
    assert s.isin(["true", "false", "1", "0"]).all()
    return s.isin(["true", "1"]).to_numpy()

# Track-level gain: +1 rescued, -1 harmed, 0 unchanged
cnn_gain = (
    as_bool(df["cnn_early_fusion_correct"]).astype(int)
    - as_bool(df["cnn_audio_correct"]).astype(int)
)

mm_gain = (
    as_bool(df["mm_fusion_correct"]).astype(int)
    - as_bool(df["mm_audio_correct"]).astype(int)
)

gain = np.column_stack([
    cnn_gain,
    mm_gain,
    mm_gain - cnn_gain
])

model_names = [
    "CNN fusion vs CNN audio",
    "MM fusion vs MM audio",
    "MM gain minus CNN gain"
]

quintiles = df["typicality_quintile"].to_numpy(
    dtype=int
)

assert set(np.unique(quintiles)) == {1, 2, 3, 4, 5}

artists = df["msd_artist_id"].astype(str).to_numpy()
unique_artists, artist_codes = np.unique(
    artists, return_inverse=True
)

n_artists = len(unique_artists)
n_boot = 3000
rng = np.random.default_rng(2026)

print("=" * 75)
print("PHASE 3 — ARTIST-CLUSTER BOOTSTRAP")
print("=" * 75)

print("Test tracks:", len(df))
print("Unique test artists:", n_artists)
print("Bootstrap repetitions:", n_boot)

# ------------------------------------------------------------
# 1. Preaggregate counts and gain sums by artist/quintile
# ------------------------------------------------------------

# Shape: artists x quintiles
counts = np.zeros((n_artists, 5), dtype=np.int32)

# Shape: artists x quintiles x comparisons
sums = np.zeros((n_artists, 5, 3), dtype=float)

for i in range(len(df)):
    a = artist_codes[i]
    q = quintiles[i] - 1
    counts[a, q] += 1
    sums[a, q, :] += gain[i, :]

# Original estimates
original_n = counts.sum(axis=0)
original_sum = sums.sum(axis=0)

original = (
    100 * original_sum / original_n[:, None]
)

# ------------------------------------------------------------
# 2. Cluster bootstrap
# ------------------------------------------------------------

boot = np.full(
    (n_boot, 5, 3),
    np.nan,
    dtype=float
)

for b in range(n_boot):

    sampled = rng.integers(
        0, n_artists, size=n_artists
    )

    # Number of times each artist was selected
    weights = np.bincount(
        sampled, minlength=n_artists
    )

    sample_counts = weights @ counts

    sample_sums = np.einsum(
        "a,aqm->qm",
        weights,
        sums
    )

    valid = sample_counts > 0

    boot[b, valid, :] = (
        100
        * sample_sums[valid, :]
        / sample_counts[valid, None]
    )

# ------------------------------------------------------------
# 3. Summarize percentile confidence intervals
# ------------------------------------------------------------

rows = []

for q in range(5):
    for m, name in enumerate(model_names):

        values = boot[:, q, m]
        values = values[np.isfinite(values)]

        lower, upper = np.percentile(
            values, [2.5, 97.5]
        )

        rows.append({
            "quintile": q + 1,
            "comparison": name,
            "N": int(original_n[q]),
            "estimate_pp": original[q, m],
            "CI_lower_pp": lower,
            "CI_upper_pp": upper,
            "valid_bootstraps": len(values)
        })

results = pd.DataFrame(rows)

print("\n" + "=" * 75)
print("FUSION GAIN AND 95% ARTIST-CLUSTER CIs")
print("=" * 75)

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# ------------------------------------------------------------
# 4. Overall bootstrap gains
# ------------------------------------------------------------

overall_original = 100 * gain.mean(axis=0)

overall_boot = np.zeros((n_boot, 3))

for b in range(n_boot):

    sampled = rng.integers(
        0, n_artists, size=n_artists
    )

    weights = np.bincount(
        sampled, minlength=n_artists
    )

    n = weights @ counts.sum(axis=1)
    total = np.einsum(
        "a,am->m",
        weights,
        sums.sum(axis=1)
    )

    overall_boot[b, :] = 100 * total / n

overall_rows = []

for m, name in enumerate(model_names):

    lo, hi = np.percentile(
        overall_boot[:, m],
        [2.5, 97.5]
    )

    overall_rows.append({
        "comparison": name,
        "estimate_pp": overall_original[m],
        "CI_lower_pp": lo,
        "CI_upper_pp": hi
    })

overall_results = pd.DataFrame(overall_rows)

print("\n" + "=" * 75)
print("OVERALL FUSION GAIN AND 95% CIs")
print("=" * 75)

print(
    overall_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

results.to_csv(
    phase3_dir / "quintile_fusion_gain_cluster_bootstrap.csv",
    index=False
)

overall_results.to_csv(
    phase3_dir / "overall_fusion_gain_cluster_bootstrap.csv",
    index=False
)

np.save(
    phase3_dir / "quintile_bootstrap_samples.npy",
    boot
)

print("\nSTEP 3A COMPLETE")
print("Saved to:", phase3_dir)


In [ ]:

# ============================================================
# PHASE 3 — STEP 3B
# Q1–Q5 FUSION GAIN CONTRASTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

phase3_dir = (
    PROJECT / "finetuning_experiments"
    / "phase3_uncertainty"
)

boot = np.load(
    phase3_dir / "quintile_bootstrap_samples.npy"
)

estimates = pd.read_csv(
    phase3_dir / "quintile_fusion_gain_cluster_bootstrap.csv"
)

assert boot.ndim == 3
assert boot.shape[1:] == (5, 3)

names = [
    "CNN fusion vs CNN audio",
    "MM fusion vs MM audio",
    "MM gain minus CNN gain"
]

print("=" * 75)
print("PHASE 3 — Q1 TO Q5 CONTRAST ANALYSIS")
print("=" * 75)

rows = []

for m, name in enumerate(names):

    q1 = estimates.loc[
        (estimates["quintile"] == 1)
        & (estimates["comparison"] == name),
        "estimate_pp"
    ].iloc[0]

    q5 = estimates.loc[
        (estimates["quintile"] == 5)
        & (estimates["comparison"] == name),
        "estimate_pp"
    ].iloc[0]

    contrast = q1 - q5

    # Paired bootstrap: subtract Q5 from Q1
    # within the SAME bootstrap repetition.
    boot_contrast = boot[:, 0, m] - boot[:, 4, m]

    boot_contrast = boot_contrast[
        np.isfinite(boot_contrast)
    ]

    lo, hi = np.percentile(
        boot_contrast, [2.5, 97.5]
    )

    rows.append({
        "comparison": name,
        "Q1_gain_pp": q1,
        "Q5_gain_pp": q5,
        "Q1_minus_Q5_pp": contrast,
        "CI_lower_pp": lo,
        "CI_upper_pp": hi,
        "valid_bootstraps": len(boot_contrast)
    })

results = pd.DataFrame(rows)

print("\nQ1–Q5 FUSION GAIN CONTRASTS")

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

results.to_csv(
    phase3_dir / "q1_q5_fusion_gain_contrasts.csv",
    index=False
)

print("\nSTEP 3B COMPLETE")
print("Saved to:", phase3_dir)


In [ ]:

# ============================================================
# PHASE 4 — STEP 4A (CORRECTED)
# IDENTIFY REAL ALBUM-COVER EXAMPLES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)

phase4_dir = (
    PROJECT / "finetuning_experiments"
    / "phase4_album_examples"
)

phase4_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

df = pd.read_csv(
    phase2_dir / "cnn_mm_paired_transitions.csv"
)

typicality = pd.read_csv(
    saved_dir / "final_test_typicality_details.csv"
)

metadata = pd.read_csv(
    PROJECT / "clean_album_metadata.csv"
)

# ------------------------------------------------------------
# 2. Add missing visual distances
# ------------------------------------------------------------

distances = typicality[
    ["msd_track_id", "visual_distance"]
].copy()

assert distances["msd_track_id"].is_unique

df = df.merge(
    distances,
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert len(df) == 1792
assert df["visual_distance"].notna().all()

# ------------------------------------------------------------
# 3. Add album-cover metadata
# ------------------------------------------------------------

cover_metadata = metadata[
    ["msd_track_id", "album_index", "image_url"]
].copy()

assert cover_metadata["msd_track_id"].is_unique

df = df.merge(
    cover_metadata,
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert len(df) == 1792

print("=" * 75)
print("PHASE 4 — ALBUM-COVER EXAMPLE DISCOVERY")
print("=" * 75)

print("\nDATA INTEGRITY")

print("Test tracks:", len(df))

print(
    "Tracks with visual distances:",
    df["visual_distance"].notna().sum()
)

print(
    "Tracks with image URLs:",
    df["image_url"].notna().sum()
)

# ------------------------------------------------------------
# 4. Define interesting example categories
# ------------------------------------------------------------

groups = {
    "Q1_CNN_rescued": (
        (df["typicality_quintile"] == 1)
        & (df["cnn_transition"] == "Rescued")
    ),

    "Q5_CNN_harmed": (
        (df["typicality_quintile"] == 5)
        & (df["cnn_transition"] == "Harmed")
    ),

    "Q1_both_fusions_rescued": (
        (df["typicality_quintile"] == 1)
        & (df["cnn_transition"] == "Rescued")
        & (df["mm_transition"] == "Rescued")
    ),

    "Q5_both_fusions_harmed": (
        (df["typicality_quintile"] == 5)
        & (df["cnn_transition"] == "Harmed")
        & (df["mm_transition"] == "Harmed")
    )
}

# ------------------------------------------------------------
# 5. Select representative tracks
# ------------------------------------------------------------

examples = []

for name, mask in groups.items():

    subset = df.loc[mask].copy()

    # Q1: smallest distance first
    # Q5: largest distance first

    ascending = name.startswith("Q1")

    subset = subset.sort_values(
        "visual_distance",
        ascending=ascending
    )

    selected = subset.head(5).copy()

    selected["example_group"] = name

    examples.append(selected)

    print("\n" + "-" * 75)
    print(name)
    print("Available tracks:", len(subset))

    print(
        selected[
            [
                "msd_track_id",
                "genre_typicality",
                "visual_distance",
                "cnn_transition",
                "mm_transition",
                "image_url"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 6. Save selected examples
# ------------------------------------------------------------

selected_examples = pd.concat(
    examples,
    ignore_index=True
)

selected_examples.to_csv(
    phase4_dir / "candidate_album_examples.csv",
    index=False
)

df.to_csv(
    phase4_dir / "test_predictions_with_cover_metadata.csv",
    index=False
)

print("\n" + "=" * 75)
print("STEP 4A COMPLETE")
print("=" * 75)

print("Selected example rows:", len(selected_examples))
print("Saved to:", phase4_dir)


In [ ]:

# ============================================================
# PHASE 4 — STEP 4B
# VISUALIZE REPRESENTATIVE ALBUM COVERS
# ============================================================

from pathlib import Path
from io import BytesIO
import pandas as pd
import requests
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

mm_dir = (
    PROJECT / "finetuning_experiments"
    / "oramas_representation_experiment"
)

phase4_dir = (
    PROJECT / "finetuning_experiments"
    / "phase4_album_examples"
)
phase4_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load verified test data and predictions
# ------------------------------------------------------------

df = pd.read_csv(
    phase4_dir / "test_predictions_with_cover_metadata.csv"
)

pred = pd.read_csv(
    mm_dir / "representation_final_test_predictions.csv"
)

prediction_cols = [
    "msd_track_id",
    "cnn_audio_prediction",
    "cnn_early_fusion_prediction",
    "mm_audio_prediction",
    "mm_fusion_prediction"
]

assert df["msd_track_id"].is_unique
assert pred["msd_track_id"].is_unique

df = df.merge(
    pred[prediction_cols],
    on="msd_track_id",
    how="left",
    validate="one_to_one"
)

assert len(df) == 1792
assert df["mm_fusion_prediction"].notna().all()

# ------------------------------------------------------------
# 2. Select examples across different genres
# ------------------------------------------------------------

def select_diverse(group, n=6, ascending=True):

    group = group.sort_values(
        "visual_distance",
        ascending=ascending
    )

    # First pass: one example per genre
    selected = group.drop_duplicates(
        subset=["genre_typicality"]
    ).head(n)

    # Second pass: fill any remaining slots
    if len(selected) < n:
        remaining = group[
            ~group["msd_track_id"].isin(
                selected["msd_track_id"]
            )
        ]

        selected = pd.concat(
            [
                selected,
                remaining.head(n - len(selected))
            ]
        )

    return selected.copy()

q1 = df[
    (df["typicality_quintile"] == 1)
    & (df["cnn_transition"] == "Rescued")
    & (df["mm_transition"] == "Rescued")
]

q5 = df[
    (df["typicality_quintile"] == 5)
    & (df["cnn_transition"] == "Harmed")
    & (df["mm_transition"] == "Harmed")
]

q1_examples = select_diverse(
    q1, n=6, ascending=True
)

q5_examples = select_diverse(
    q5, n=6, ascending=False
)

q1_examples["example_group"] = "Q1_both_rescued"
q5_examples["example_group"] = "Q5_both_harmed"

selected = pd.concat(
    [q1_examples, q5_examples],
    ignore_index=True
)

assert selected["msd_track_id"].is_unique

print("=" * 75)
print("PHASE 4 — REAL ALBUM-COVER EXAMPLES")
print("=" * 75)

print("\nSelected examples:", len(selected))

# ------------------------------------------------------------
# 3. Download images for display only
# ------------------------------------------------------------

def load_cover(url, size=(220, 220)):

    try:
        response = requests.get(
            str(url).replace(
                "http://", "https://", 1
            ),
            timeout=10,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        img = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        img = ImageOps.fit(img, size)

        return img, True

    except Exception:
        # Try the original URL if HTTPS failed
        try:
            response = requests.get(
                str(url),
                timeout=10,
                headers={
                    "User-Agent": "Mozilla/5.0"
                }
            )
            response.raise_for_status()

            img = Image.open(
                BytesIO(response.content)
            ).convert("RGB")

            return ImageOps.fit(img, size), True

        except Exception:
            placeholder = Image.new(
                "RGB", size, "#e5e5e5"
            )
            draw = ImageDraw.Draw(placeholder)
            draw.text(
                (30, 95),
                "Image unavailable",
                fill="black"
            )
            return placeholder, False

# ------------------------------------------------------------
# 4. Display each group
# ------------------------------------------------------------

availability = {}

for group_name, group in [
    ("Q1 — BOTH FUSIONS RESCUED", q1_examples),
    ("Q5 — BOTH FUSIONS HARMED", q5_examples)
]:

    print("\n" + "=" * 75)
    print(group_name)
    print("=" * 75)

    fig, axes = plt.subplots(
        2, 3, figsize=(15, 11)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(
        axes, group.iterrows()
    ):

        img, success = load_cover(
            row["image_url"]
        )

        availability[row["msd_track_id"]] = success

        ax.imshow(img)
        ax.axis("off")

        title = (
            f"True: {row['genre_typicality']}\n"
            f"Distance: {row['visual_distance']:.3f}\n"
            f"CNN audio: {row['cnn_audio_prediction']}\n"
            f"CNN fusion: {row['cnn_early_fusion_prediction']}\n"
            f"MM audio: {row['mm_audio_prediction']}\n"
            f"MM fusion: {row['mm_fusion_prediction']}"
        )

        ax.set_title(
            title,
            fontsize=10,
            pad=10
        )

    for ax in axes[len(group):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# 5. Save selection and image availability
# ------------------------------------------------------------

selected["image_available"] = (
    selected["msd_track_id"].map(availability)
)

selected.to_csv(
    phase4_dir / "selected_visual_examples.csv",
    index=False
)

print("\n" + "=" * 75)
print("IMAGE AVAILABILITY")
print("=" * 75)

print(
    selected.groupby("example_group")[
        "image_available"
    ].agg(["count", "sum"]).to_string()
)

print("\nSTEP 4B COMPLETE")
print("Saved to:", phase4_dir)


In [ ]:

# ============================================================
# PHASE 4 — STEP 4C
# SAVE SIX REPORT EXAMPLES
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")
phase4_dir = (
    PROJECT / "finetuning_experiments"
    / "phase4_album_examples"
)

df = pd.read_csv(
    phase4_dir / "selected_visual_examples.csv"
)

assert df["msd_track_id"].is_unique
assert df["image_available"].astype(str).str.lower().eq("true").all()

# Three different genres from each group
preferred_genres = {
    "Q1_both_rescued": ["Pop", "Rap", "Jazz"],
    "Q5_both_harmed": ["Rock", "Rap", "Jazz"]
}

selected_rows = []

for group, genres in preferred_genres.items():

    subset = df[df["example_group"] == group].copy()

    for genre in genres:

        matches = subset[
            subset["genre_typicality"] == genre
        ]

        if matches.empty:
            print(f"Unavailable: {group}, {genre}")
            continue

        ascending = group.startswith("Q1")

        row = matches.sort_values(
            "visual_distance",
            ascending=ascending
        ).iloc[0]

        selected_rows.append(row)

report_examples = pd.DataFrame(selected_rows)

assert report_examples["msd_track_id"].is_unique

columns = [
    "example_group",
    "msd_track_id",
    "genre_typicality",
    "visual_distance",
    "cnn_audio_prediction",
    "cnn_early_fusion_prediction",
    "mm_audio_prediction",
    "mm_fusion_prediction",
    "image_url"
]

report_examples = report_examples[columns]

report_examples.to_csv(
    phase4_dir / "final_report_album_examples.csv",
    index=False
)

print("=" * 70)
print("PHASE 4 — FINAL REPORT EXAMPLES")
print("=" * 70)

print(
    report_examples.drop(columns="image_url")
    .to_string(index=False)
)

print("\nNumber selected:", len(report_examples))
print("Saved to:", phase4_dir)


In [ ]:

# ============================================================
# PHASE 5 — STEP 5A
# PCA VISUALIZATION OF AUDIO AND VISUAL EMBEDDINGS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

saved_dir = (
    PROJECT / "finetuning_experiments"
    / "multimodal_artist_disjoint"
)

phase5_dir = (
    PROJECT / "finetuning_experiments"
    / "phase5_embedding_analysis"
)
phase5_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load labels and embeddings
# ------------------------------------------------------------

train_labels = pd.read_csv(
    saved_dir / "labels_train.csv"
)

test_labels = pd.read_csv(
    saved_dir / "labels_test.csv"
)

X_audio_train = np.load(
    saved_dir / "X_train_audio.npy"
)

X_audio_test = np.load(
    saved_dir / "X_test_audio.npy"
)

X_visual_train = np.load(
    saved_dir / "X_train_visual.npy"
)

X_visual_test = np.load(
    saved_dir / "X_test_visual.npy"
)

assert len(train_labels) == len(X_audio_train)
assert len(test_labels) == len(X_audio_test)
assert len(train_labels) == len(X_visual_train)
assert len(test_labels) == len(X_visual_test)

assert len(test_labels) == 1792

# ------------------------------------------------------------
# 2. Fit PCA on training data only
# ------------------------------------------------------------

def compute_pca(X_train, X_test):

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(X_train)
    test_scaled = scaler.transform(X_test)

    pca = PCA(
        n_components=2,
        svd_solver="randomized",
        random_state=42
    )

    pca.fit(train_scaled)

    test_projected = pca.transform(test_scaled)

    return test_projected, pca.explained_variance_ratio_

audio_pca, audio_variance = compute_pca(
    X_audio_train,
    X_audio_test
)

visual_pca, visual_variance = compute_pca(
    X_visual_train,
    X_visual_test
)

# ------------------------------------------------------------
# 3. Plot the test embeddings
# ------------------------------------------------------------

genres = sorted(test_labels["genre"].unique())

colors = plt.get_cmap("tab10")(
    np.linspace(0, 1, len(genres))
)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6)
)

for ax, points, variance, title in [
    (
        axes[0],
        audio_pca,
        audio_variance,
        "CNN Audio Embeddings"
    ),
    (
        axes[1],
        visual_pca,
        visual_variance,
        "CNN Visual Embeddings"
    )
]:

    for genre, color in zip(genres, colors):

        mask = (
            test_labels["genre"].to_numpy() == genre
        )

        ax.scatter(
            points[mask, 0],
            points[mask, 1],
            s=12,
            alpha=0.45,
            color=color,
            label=genre,
            edgecolors="none"
        )

    ax.set_title(title, fontsize=13)

    ax.set_xlabel(
        f"PC1 ({variance[0] * 100:.2f}% variance)"
    )

    ax.set_ylabel(
        f"PC2 ({variance[1] * 100:.2f}% variance)"
    )

    ax.grid(alpha=0.15)

    ax.legend(
        fontsize=9,
        markerscale=1.5
    )

plt.suptitle(
    "PCA of CNN Audio and Visual Embeddings",
    fontsize=15
)

plt.tight_layout()

figure_path = (
    phase5_dir / "audio_visual_pca_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 4. Report explained variance
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 5 — PCA EMBEDDING ANALYSIS")
print("=" * 75)

print("\nAUDIO PCA")
print(
    "PC1 explained variance:",
    f"{audio_variance[0] * 100:.2f}%"
)
print(
    "PC2 explained variance:",
    f"{audio_variance[1] * 100:.2f}%"
)
print(
    "Combined explained variance:",
    f"{audio_variance.sum() * 100:.2f}%"
)

print("\nVISUAL PCA")
print(
    "PC1 explained variance:",
    f"{visual_variance[0] * 100:.2f}%"
)
print(
    "PC2 explained variance:",
    f"{visual_variance[1] * 100:.2f}%"
)
print(
    "Combined explained variance:",
    f"{visual_variance.sum() * 100:.2f}%"
)

# ------------------------------------------------------------
# 5. Save coordinates for reproducibility
# ------------------------------------------------------------

coordinates = test_labels[
    ["msd_track_id", "genre", "msd_artist_id"]
].copy()

coordinates["audio_PC1"] = audio_pca[:, 0]
coordinates["audio_PC2"] = audio_pca[:, 1]

coordinates["visual_PC1"] = visual_pca[:, 0]
coordinates["visual_PC2"] = visual_pca[:, 1]

coordinates.to_csv(
    phase5_dir / "test_pca_coordinates.csv",
    index=False
)

print("\nSTEP 5A COMPLETE")
print("Figure saved to:", figure_path)
print("Coordinates saved to:", phase5_dir)


In [ ]:

# ============================================================
# PHASE 6 — STEP 6A
# MAIN SCIENTIFIC FIGURE: FUSION GAIN BY TYPICALITY
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

phase3_dir = (
    PROJECT / "finetuning_experiments"
    / "phase3_uncertainty"
)

figures_dir = (
    PROJECT / "finetuning_experiments"
    / "final_report_figures"
)
figures_dir.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(
    phase3_dir / "quintile_fusion_gain_cluster_bootstrap.csv"
)

models = {
    "CNN fusion vs CNN audio": {
        "label": "CNN fusion − CNN audio",
        "color": "#2563A6",
        "offset": -0.07
    },
    "MM fusion vs MM audio": {
        "label": "MM fusion − MM audio",
        "color": "#D17A32",
        "offset": 0.07
    }
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, ax = plt.subplots(figsize=(9, 5.5))

for model, style in models.items():

    subset = results[
        results["comparison"] == model
    ].sort_values("quintile")

    assert len(subset) == 5

    x = subset["quintile"].to_numpy(dtype=float)
    y = subset["estimate_pp"].to_numpy(dtype=float)

    lower = subset["CI_lower_pp"].to_numpy(dtype=float)
    upper = subset["CI_upper_pp"].to_numpy(dtype=float)

    error = np.vstack([
        y - lower,
        upper - y
    ])

    ax.errorbar(
        x + style["offset"],
        y,
        yerr=error,
        fmt="o-",
        color=style["color"],
        linewidth=2,
        markersize=7,
        capsize=4,
        capthick=1.5,
        elinewidth=1.5,
        label=style["label"]
    )

ax.axhline(
    0,
    color="#555555",
    linewidth=1.2,
    linestyle="--"
)

ax.set_xticks([1, 2, 3, 4, 5])

ax.set_xticklabels([
    "Q1\nMost typical",
    "Q2",
    "Q3",
    "Q4",
    "Q5\nMost atypical"
])

ax.set_ylabel(
    "Fusion − audio-only accuracy (percentage points)"
)

ax.set_xlabel(
    "Album-cover visual typicality (within true genre)"
)

ax.set_title(
    "The benefit of multimodal fusion depends on\n"
    "album-cover visual typicality",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlim(0.7, 5.3)
ax.set_ylim(-30, 28)

ax.grid(
    axis="y",
    alpha=0.18
)

ax.legend(
    loc="upper right",
    frameon=False,
    fontsize=10
)

fig.text(
    0.12, 0.015,
    "Error bars: 95% artist-cluster bootstrap CIs "
    "(3,000 resamples). N = 1,792 test tracks.\n"
    "Typicality uses true genre; analysis is retrospective.",
    fontsize=8.5,
    color="#555555"
)

fig.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.83,
    bottom=0.22
)

png_path = (
    figures_dir / "figure1_fusion_gain_by_typicality.png"
)

pdf_path = (
    figures_dir / "figure1_fusion_gain_by_typicality.pdf"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("=" * 70)
print("PHASE 6 — MAIN FIGURE COMPLETE")
print("=" * 70)

print("PNG:", png_path)
print("PDF:", pdf_path)


In [ ]:

# ============================================================
# PHASE 6 — STEP 6B
# FIGURE 2: RESCUED VS HARMED PREDICTIONS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

phase2_dir = (
    PROJECT / "finetuning_experiments"
    / "phase2_error_analysis"
)

figures_dir = (
    PROJECT / "finetuning_experiments"
    / "final_report_figures"
)
figures_dir.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(
    phase2_dir / "cnn_mm_quintile_transitions.csv"
)

# ------------------------------------------------------------
# 1. Validate data
# ------------------------------------------------------------

assert set(results["representation"]) == {"CNN", "MM"}

for representation in ["CNN", "MM"]:
    subset = results[
        results["representation"] == representation
    ]
    assert set(subset["quintile"]) == {1, 2, 3, 4, 5}

assert np.allclose(
    results["rescue_pct"] - results["harm_pct"],
    results["net_gain_pp"]
)

# ------------------------------------------------------------
# 2. Figure styling
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

rescue_color = "#247A62"
harm_color = "#C65D4A"

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    sharey=True
)

# ------------------------------------------------------------
# 3. Plot each representation
# ------------------------------------------------------------

for ax, representation in zip(
    axes, ["CNN", "MM"]
):

    subset = results[
        results["representation"] == representation
    ].sort_values("quintile")

    x = subset["quintile"].to_numpy()

    rescue = subset["rescue_pct"].to_numpy()
    harm = subset["harm_pct"].to_numpy()

    ax.plot(
        x,
        rescue,
        "o-",
        color=rescue_color,
        linewidth=2.3,
        markersize=7,
        label="Rescued"
    )

    ax.plot(
        x,
        harm,
        "s-",
        color=harm_color,
        linewidth=2.3,
        markersize=7,
        label="Harmed"
    )

    ax.fill_between(
        x,
        rescue,
        harm,
        alpha=0.08,
        color="#777777"
    )

    ax.set_title(
        f"{representation} representation",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(["Q1", "Q2", "Q3", "Q4", "Q5"])

    ax.set_xlabel(
        "Visual-typicality quintile"
    )

    ax.set_ylim(0, 27)
    ax.grid(axis="y", alpha=0.18)

    ax.legend(
        loc="upper center",
        frameon=False,
        ncol=2
    )

axes[0].set_ylabel(
    "Percentage of test tracks in quintile (%)"
)

fig.suptitle(
    "Fusion rescues fewer predictions and harms more\n"
    "as album covers become visually atypical",
    fontsize=15,
    fontweight="bold",
    y=1.04
)

fig.text(
    0.5,
    -0.025,
    "Q1 = most visually typical; Q5 = most visually atypical. "
    "Both models use the same true-genre-defined quintiles.",
    ha="center",
    fontsize=9,
    color="#555555"
)

fig.tight_layout()

# ------------------------------------------------------------
# 4. Save publication-quality files
# ------------------------------------------------------------

png_path = (
    figures_dir / "figure2_rescued_vs_harmed.png"
)

pdf_path = (
    figures_dir / "figure2_rescued_vs_harmed.pdf"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("=" * 70)
print("PHASE 6 — FIGURE 2 COMPLETE")
print("=" * 70)

print("PNG:", png_path)
print("PDF:", pdf_path)

# ------------------------------------------------------------
# 5. Print underlying counts for verification
# ------------------------------------------------------------

print("\nUnderlying prediction-transition counts:")

print(
    results[
        [
            "representation",
            "quintile",
            "N",
            "rescued",
            "harmed",
            "net_gain_pp"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


In [ ]:

# ============================================================
# PHASE 6 — STEP 6C
# FIGURE 3: REAL ALBUM-COVER EXAMPLES
# ============================================================

from pathlib import Path
from io import BytesIO
import pandas as pd
import requests
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/MSD-I_project")

phase4_dir = (
    PROJECT / "finetuning_experiments"
    / "phase4_album_examples"
)

figures_dir = (
    PROJECT / "finetuning_experiments"
    / "final_report_figures"
)
figures_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    phase4_dir / "final_report_album_examples.csv"
)

assert len(df) == 6
assert df["msd_track_id"].is_unique

# ------------------------------------------------------------
# 1. Load each cover
# ------------------------------------------------------------

def load_cover(url):

    for candidate in [
        str(url).replace("http://", "https://", 1),
        str(url)
    ]:
        try:
            response = requests.get(
                candidate,
                timeout=12,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()

            image = Image.open(
                BytesIO(response.content)
            ).convert("RGB")

            return ImageOps.fit(image, (400, 400)), True

        except Exception:
            continue

    placeholder = Image.new(
        "RGB", (400, 400), "#e5e5e5"
    )

    draw = ImageDraw.Draw(placeholder)
    draw.text(
        (105, 190),
        "Image unavailable",
        fill="black"
    )

    return placeholder, False

# ------------------------------------------------------------
# 2. Create figure
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "pdf.fonttype": 42
})

fig, axes = plt.subplots(
    2, 3,
    figsize=(12, 10)
)

availability = []

for ax, (_, row) in zip(
    axes.flatten(),
    df.iterrows()
):

    image, success = load_cover(row["image_url"])

    availability.append(success)

    ax.imshow(image)
    ax.axis("off")

    true_genre = row["genre_typicality"]

    cnn_audio = row["cnn_audio_prediction"]
    cnn_fusion = row["cnn_early_fusion_prediction"]

    mm_audio = row["mm_audio_prediction"]
    mm_fusion = row["mm_fusion_prediction"]

    ax.set_title(
        f"True genre: {true_genre}\n"
        f"Visual distance: {row['visual_distance']:.3f}",
        fontsize=11,
        fontweight="bold",
        pad=9
    )

    caption = (
        f"CNN: {cnn_audio} → {cnn_fusion}\n"
        f"MM: {mm_audio} → {mm_fusion}"
    )

    ax.text(
        0.5,
        -0.08,
        caption,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=10
    )


# Clearer row headings and more vertical spacing

fig.text(
    0.5, 0.985,
    "A. Visually typical covers — fusion rescues audio predictions",
    ha="center",
    va="top",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.5, 0.495,
    "B. Visually atypical covers — fusion harms audio predictions",
    ha="center",
    va="top",
    fontsize=13,
    fontweight="bold"
)

fig.subplots_adjust(
    left=0.04,
    right=0.96,
    top=0.90,
    bottom=0.06,
    hspace=0.85,
    wspace=0.12
)


# ------------------------------------------------------------
# 3. Save PNG and PDF
# ------------------------------------------------------------

png_path = (
    figures_dir / "figure3_album_cover_examples.png"
)

pdf_path = (
    figures_dir / "figure3_album_cover_examples.pdf"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 4. Verify image availability
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 6 — FIGURE 3 COMPLETE")
print("=" * 70)

print("Covers successfully loaded:", sum(availability), "/ 6")

if not all(availability):
    print(
        "WARNING: Some images were unavailable. "
        "Replace placeholders before using the figure."
    )

print("PNG:", png_path)
print("PDF:", pdf_path)
